# Árbol de decisión C4.5 contra Random Forest

**Inteligencia Artificial · 5.º nivel ISI · Ciclo lectivo 2026**

En este cuaderno se compara un **árbol de decisión C4.5**, programado íntegramente por el equipo sin usar librerías
de árboles, con el **Random Forest** de la librería scikit-learn. Los dos modelos se entrenan y se evalúan con los
mismos datos, para que la comparación sea justa.

**Este cuaderno es autocontenido:** incluye todo el código del programa y los datos que usa, así que no descarga ningún
repositorio. Lo único que instala es la librería [anywidget](https://anywidget.dev) (con `pip`), que permite mostrar la
interfaz dentro del cuaderno.

| Objetivo de la consigna | Dónde se cumple |
|---|---|
| Implementar el algoritmo de árbol de decisión | El código está en la sección **1.1** y se usa en las secciones 2 y 3. |
| Comparar el árbol con el Random Forest | En la interfaz, en la pestaña **Comparar**. |
| Analizar cómo cambian las métricas al modificar los parámetros | En la interfaz, en las pestañas **Árbol** y **Experimentos**. |

### Contenido

0. **Preparación:** instala anywidget y crea la carpeta de trabajo.
1. **Código del proyecto:** el programa completo y los datos. Cada celda escribe un archivo en la carpeta de trabajo.
2. **Interfaz interactiva:** permite usar todo el trabajo desde una sola pantalla, sin escribir código.
3. **El árbol desde código:** muestra cómo usar el mismo modelo directamente desde Python.
4. **Verificación:** ejecuta las pruebas automáticas del proyecto. Es opcional.

> **Cómo usarlo:** ejecutá las celdas en orden. En Colab, podés usar el menú *Entorno de ejecución → Ejecutar todas*.

## 0. Preparación

Esta celda instala anywidget, crea la carpeta de trabajo (`tp_ia_2026`) y deja a Python listo para encontrar el código
que escriben las celdas de la sección 1. Si el cuaderno se vuelve a ejecutar en la misma sesión, también descarta
cualquier versión anterior del código que Python tuviera cargada.

In [ ]:
# Prepara el entorno: instala anywidget y crea la carpeta de trabajo donde se escriben los archivos del proyecto.
import os, subprocess, sys

def run_quiet(command, **kwargs):
    """Ejecuta un comando sin mostrar su salida; si falla, la muestra para poder ver qué pasó."""
    result = subprocess.run(command, capture_output=True, text=True, **kwargs)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:], sep="\n")
        raise RuntimeError("Falló el comando: " + " ".join(command))
    return result

ROOT = os.path.join("/content" if os.path.isdir("/content") else os.getcwd(), "tp_ia_2026")
os.makedirs(ROOT, exist_ok=True)
os.chdir(ROOT)
for folder in [
    "data/iris",
    "data/uci_heart",
    "src/tp_ia",
    "src/tp_ia/c45",
    "src/tp_ia/data",
    "src/tp_ia/eval",
    "src/tp_ia/web",
    "src/tp_ia/web/static",
    "tests",
]:
    os.makedirs(folder, exist_ok=True)

run_quiet([sys.executable, "-m", "pip", "install", "-q", "anywidget"])
sys.path.insert(0, os.path.join(ROOT, "src"))
for name in [m for m in sys.modules if m == "tp_ia" or m.startswith("tp_ia.")]:
    del sys.modules[name]                  # por si esta sesión ya había cargado una versión anterior del código
print(f"✔ Listo: carpeta de trabajo {ROOT}")

## 1. Código del proyecto

Cada celda de esta sección escribe un archivo del programa en la carpeta de trabajo. La primera línea de la celda
(`%%writefile`) dice qué archivo es, y el resto es su código completo. No hace falta leerlas para usar el cuaderno, pero
son el programa entero: no hay nada afuera.

El programa se organiza en cuatro bloques (secciones 1.1 a 1.4) y usa dos archivos de datos (sección 1.5). Las pruebas
automáticas están en la sección 4.

### 1.1 El árbol C4.5 (`tp_ia.c45`)

Es la parte central del trabajo y no usa ninguna librería de árboles de decisión.

- `__init__.py`: el paquete `tp_ia`, que reúne todo el programa.
- `__init__.py`: lo que se puede importar del árbol.
- `criteria.py`: entropía, ganancia de información y razón de ganancia.
- `splits.py`: busca el mejor corte de cada nodo: por valor en los atributos nominales y por umbral en los numéricos.
- `tree.py`: el árbol completo (`C45DecisionTree`): se entrena con `fit`, predice, da probabilidades y explica cada predicción.
- `pruning.py`: la poda por error pesimista, como la propone Quinlan.
- `export.py`: exporta el árbol como estructura de datos, como código para Graphviz y como recorrido de un caso.

In [ ]:
%%writefile src/tp_ia/__init__.py
"""TP IA 2026: árbol de decisión C4.5 propio vs Random Forest (scikit-learn)."""


In [ ]:
%%writefile src/tp_ia/c45/__init__.py
"""Árbol de decisión C4.5 (implementación propia)."""
from .export import trace_case, tree_to_dict, tree_to_dot
from .pruning import added_errors, confidence_coefficient, estimated_errors, prune_tree
from .tree import C45DecisionTree, Node

__all__ = ["C45DecisionTree", "Node", "added_errors", "confidence_coefficient", "estimated_errors",
           "prune_tree", "trace_case", "tree_to_dict", "tree_to_dot"]


In [ ]:
%%writefile src/tp_ia/c45/criteria.py
"""
Criterios de C4.5: entropía, ganancia de información, split info y gain ratio.

GUÍA DE LECTURA (para el coloquio)
----------------------------------
Todo se calcula sobre *conteos ponderados de clases*: un vector de largo k (k = nº de clases)
donde cada posición es la suma de pesos de los casos de esa clase.

  - Usamos pesos y no simples cuentas porque C4.5 reparte "fracciones de caso" cuando un valor
    falta (ver tree.py). Sin faltantes, cada caso pesa 1 y todo coincide con las fórmulas del libro.

Fórmulas (Quinlan, 1993):
  Entropía        H(S)          = - sum_c p_c * log2(p_c)
  Ganancia        Gain(S, A)    = H(S) - sum_b (|S_b| / |S|) * H(S_b)
  Split info      SplitInfo(A)  = - sum_b (|S_b| / |S|) * log2(|S_b| / |S|)
  Gain ratio      GainRatio(A)  = Gain(S, A) / SplitInfo(A)

Por qué el gain ratio: la ganancia "premia" a los atributos con muchas ramas (un atributo tipo ID
separa todo y gana mucho sin servir). Dividir por SplitInfo penaliza justamente eso.
"""
from __future__ import annotations

import numpy as np

EPS = 1e-12  # tolerancia numérica para comparar con cero


def entropy(counts) -> float:
    """Entropía de Shannon (base 2) de un vector de conteos (pueden ser pesos fraccionarios)."""
    c = np.asarray(counts, dtype=float)
    total = c.sum()
    if total <= EPS:
        return 0.0
    p = c[c > 0] / total  # solo las clases presentes: 0 * log2(0) se toma como 0
    return float(-(p * np.log2(p)).sum())


def entropy_rows(count_matrix) -> np.ndarray:
    """Entropía de cada fila de una matriz de conteos (m filas x k clases), sin bucles de Python.

    Se usa para evaluar *todos* los umbrales de un atributo numérico de una sola vez.
    """
    m = np.asarray(count_matrix, dtype=float)
    total = m.sum(axis=1, keepdims=True)
    p = m / np.where(total > EPS, total, 1.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        logp = np.where(p > 0, np.log2(p), 0.0)
    return -(p * logp).sum(axis=1)


def information_gain(parent_counts, child_counts) -> float:
    """Ganancia de información de una partición.

    parent_counts: conteos de clases del nodo padre (vector de largo k).
    child_counts:  matriz (ramas x k) con los conteos de cada rama hija.
    """
    parent = np.asarray(parent_counts, dtype=float)
    child = np.asarray(child_counts, dtype=float)
    total = parent.sum()
    if total <= EPS:
        return 0.0
    branch_share = child.sum(axis=1) / total
    gain = entropy(parent) - float((branch_share * entropy_rows(child)).sum())
    return max(gain, 0.0)  # evita -1e-16 por redondeo


def split_info(branch_weights) -> float:
    """Entropía de los *tamaños* de las ramas (ignora las clases). Es 0 si hay una sola rama."""
    return entropy(branch_weights)


def gain_ratio(gain: float, split_information: float) -> float:
    """Gain / SplitInfo. Si SplitInfo es ~0 la división es degenerada y devolvemos 0."""
    if split_information <= EPS:
        return 0.0
    return gain / split_information


In [ ]:
%%writefile src/tp_ia/c45/splits.py
"""
Evaluación de divisiones candidatas para un nodo: atributos numéricos y nominales.

GUÍA DE LECTURA (para el coloquio)
----------------------------------
Para cada atributo A calculamos *la mejor división posible* y la resumimos en un SplitCandidate:

  - Atributo NUMÉRICO  -> división binaria  "A <= t"  /  "A > t".
        Se ordenan los valores y se prueban todos los cortes entre dos valores distintos
        consecutivos. Con sumas acumuladas por clase, probar todos los cortes cuesta O(n log n)
        en vez de O(n^2). El corte elegido es el de mayor ganancia (como hace C4.5).
  - Atributo NOMINAL   -> división multirrama: una rama por cada valor observado.

VALORES FALTANTES (idea de C4.5)
  - La ganancia se calcula solo con los casos que tienen el valor conocido y se multiplica por F, la
    fracción (en peso) de casos conocidos: Gain = F * Gain_conocidos.
  - Para SplitInfo los faltantes cuentan como una rama más.

MÍNIMO POR RAMA (parámetro -m de C4.5)
  - Una división solo es válida si al menos DOS ramas tienen peso >= min_leaf. Evita divisiones que
    dejan una hoja con un solo caso suelto.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from .criteria import EPS, entropy, entropy_rows, gain_ratio, split_info


@dataclass
class SplitCandidate:
    """Mejor división encontrada para un atributo."""

    feature: int                    # índice de columna
    kind: str                       # "numeric" | "nominal"
    gain: float                     # ganancia de C4.5 (ya multiplicada por la fracción de conocidos)
    gain_ratio: float
    split_info: float
    branch_weights: np.ndarray      # peso de los casos con valor conocido en cada rama
    threshold: float | None = None  # solo numérico: corte "x <= threshold"
    values: list | None = None      # solo nominal: valor que corresponde a cada rama


def _factorize(values: np.ndarray):
    """Devuelve (códigos, valores únicos ordenados). El orden fijo hace el árbol determinista."""
    uniques = list(dict.fromkeys(values.tolist()))
    try:
        uniques.sort()
    except TypeError:  # tipos mezclados que no se pueden comparar
        uniques.sort(key=str)
    index = {v: i for i, v in enumerate(uniques)}
    codes = np.fromiter((index[v] for v in values.tolist()), dtype=int, count=len(values))
    return codes, uniques


def evaluate_numeric(feature, x, y, w, n_classes, total_weight, min_leaf,
                     threshold_mode="c45", gain_correction=False):
    """Mejor corte binario de un atributo numérico (NaN = faltante). None si no hay corte válido."""
    known = ~np.isnan(x)
    if known.sum() < 2:
        return None
    xs, ys, ws = x[known], y[known], w[known]

    # Paso 1: ordenar por valor del atributo.
    order = np.argsort(xs, kind="stable")
    xs, ys, ws = xs[order], ys[order], ws[order]
    n = len(xs)

    # Paso 2: conteos acumulados por clase. cum[i] = conteos de los casos 0..i (los de la izquierda
    # si cortamos justo después de i). Así cada corte candidato se evalúa sin recorrer los datos.
    onehot = np.zeros((n, n_classes))
    onehot[np.arange(n), ys] = ws
    cum = np.cumsum(onehot, axis=0)
    total_counts = cum[-1]
    known_weight = float(total_counts.sum())

    # Paso 3: solo tiene sentido cortar entre dos valores DISTINTOS consecutivos.
    cut = np.nonzero(xs[:-1] < xs[1:])[0]
    if cut.size == 0:
        return None
    left_counts = cum[cut]
    right_counts = total_counts - left_counts
    left_w = left_counts.sum(axis=1)
    right_w = known_weight - left_w

    # Paso 4: descartar cortes que dejan una rama con menos de min_leaf de peso.
    valid = (left_w >= min_leaf - EPS) & (right_w >= min_leaf - EPS)
    if not valid.any():
        return None

    # Paso 5: ganancia de cada corte = H(padre) - promedio ponderado de H(hijos).
    child_info = (left_w * entropy_rows(left_counts) + right_w * entropy_rows(right_counts)) / known_weight
    gains_known = np.where(valid, entropy(total_counts) - child_info, -np.inf)
    best = int(np.argmax(gains_known))  # primer máximo -> desempate determinista (corte más bajo)
    gain_known = float(gains_known[best])

    # (Opcional) corrección de C4.5 release 8: penaliza a los numéricos por la cantidad de cortes
    # que "tuvieron oportunidad" de ganar, log2(N-1)/|D|. Evita preferirlos solo por tener muchos valores.
    if gain_correction:
        gain_known -= float(np.log2(cut.size)) / known_weight
    gain_known = max(gain_known, 0.0)

    # Paso 6: umbral. "c45" = mayor valor observado <= punto medio (el valor inferior del par);
    # "midpoint" = punto medio. Ambos separan exactamente los mismos casos de entrenamiento.
    i = int(cut[best])
    lower, upper = float(xs[i]), float(xs[i + 1])
    threshold = lower if threshold_mode == "c45" else (lower + upper) / 2.0

    gain = (known_weight / total_weight) * gain_known
    branch_weights = np.array([left_w[best], right_w[best]])
    missing_weight = total_weight - known_weight
    si_weights = np.append(branch_weights, missing_weight) if missing_weight > EPS else branch_weights
    si = split_info(si_weights)
    return SplitCandidate(feature, "numeric", gain, gain_ratio(gain, si), si, branch_weights,
                          threshold=threshold)


def evaluate_nominal(feature, x, y, w, n_classes, total_weight, min_leaf):
    """División multirrama de un atributo nominal (None = faltante). None si no hay división válida."""
    known = np.array([v is not None for v in x], dtype=bool)
    if known.sum() < 2:
        return None
    codes, uniques = _factorize(x[known])
    if len(uniques) < 2:
        return None  # un solo valor: no separa nada
    ys, ws = y[known], w[known]

    # Conteos (valor x clase) en una sola pasada.
    counts = np.zeros((len(uniques), n_classes))
    np.add.at(counts, (codes, ys), ws)
    branch_w = counts.sum(axis=1)
    if (branch_w >= min_leaf - EPS).sum() < 2:
        return None  # regla de C4.5: al menos dos ramas "con cuerpo"

    known_weight = float(branch_w.sum())
    child_info = float((branch_w * entropy_rows(counts)).sum()) / known_weight
    gain_known = max(entropy(counts.sum(axis=0)) - child_info, 0.0)
    gain = (known_weight / total_weight) * gain_known

    missing_weight = total_weight - known_weight
    si_weights = np.append(branch_w, missing_weight) if missing_weight > EPS else branch_w
    si = split_info(si_weights)
    return SplitCandidate(feature, "nominal", gain, gain_ratio(gain, si), si, branch_w,
                          values=uniques)


In [ ]:
%%writefile src/tp_ia/c45/tree.py
"""
Árbol de decisión C4.5 - implementación propia (sin ninguna librería de árboles de decisión).

GUÍA DE LECTURA (para el coloquio)
----------------------------------
Idea general (Quinlan, 1993): construir el árbol de arriba hacia abajo. En cada nodo:

  1. Si el nodo es "puro" (una sola clase), o tiene muy pocos casos, o llegó a max_depth -> HOJA.
  2. Si no, para cada atributo se calcula su mejor división (ver splits.py) y se elige una con la
     regla de C4.5:
        a) quedarse con los atributos cuya ganancia es >= la ganancia PROMEDIO, y
        b) entre ellos, el de mayor GAIN RATIO.
     (La parte (a) evita que el gain ratio elija un atributo de ganancia ínfima solo porque su
      SplitInfo es chico.)
  3. Si la mejor ganancia no supera min_gain -> HOJA (no vale la pena dividir).
  4. Se reparten los casos entre las ramas y se repite recursivamente.

VALORES FALTANTES (C4.5)
  - Entrenamiento: un caso al que le falta el atributo elegido se manda a TODAS las ramas con peso
    fraccionario (proporcional al peso de cada rama). Por eso todo el código trabaja con pesos.
  - Predicción: igual, el caso viaja por todas las ramas y se mezclan las distribuciones de clases
    de las hojas, ponderadas por la probabilidad de cada rama.
  - Un valor nominal NO visto en el entrenamiento se trata como faltante.

SALIDA
  - Cada hoja guarda la distribución ponderada de clases. predict_proba la normaliza y predict
    devuelve la clase más probable (empates: la primera en orden -> resultado determinista).

DIFERENCIAS CON C4.5 COMPLETO (para la tabla "implementado vs omitido" del informe)
  - Implementado: gain ratio + heurística de ganancia promedio, umbrales numéricos, ramas
    multivalor, faltantes con reparto fraccionario, min_samples_leaf (-m), corrección release 8
    (opcional), poda por error pesimista (prune=True, ver pruning.py). Pendiente: agrupación de
    valores nominales y "subtree raising".
  - Diferencia conocida: la ganancia PROMEDIO se calcula solo sobre los atributos con ganancia
    real (> min_gain); C4.5 promedia sobre todos los atributos evaluados.
  - min_samples_split=2 por defecto casi no actúa: min_samples_leaf=2 ya exige peso >= 4 para dividir.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from .criteria import EPS
from .export import trace_case, tree_to_dict, tree_to_dot
from .pruning import DEFAULT_CONFIDENCE, prune_tree
from .splits import SplitCandidate, evaluate_nominal, evaluate_numeric

CRITERIA = ("gain_ratio", "gain")
THRESHOLD_MODES = ("c45", "midpoint")
FEATURE_KINDS = ("numeric", "nominal")
_PARAM_NAMES = ("criterion", "max_depth", "min_samples_split", "min_samples_leaf", "min_gain",
                "use_avg_gain", "threshold_mode", "numeric_gain_correction", "prune", "confidence")


def _tidy(value):
    """Un nominal numérico como 3.0 (la columna era float porque tenía NaN) se compara y se muestra como 3."""
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return int(value)
    return value


@dataclass
class Node:
    """Nodo del árbol. Es hoja si no tiene hijos."""

    n_weight: float                       # peso total de casos que llegaron al nodo
    class_counts: np.ndarray              # distribución ponderada de clases en el nodo
    depth: int
    feature: int | None = None            # índice de la columna usada para dividir (nodos internos)
    kind: str | None = None               # "numeric" | "nominal"
    threshold: float | None = None        # numérico: rama 0 si x <= threshold, rama 1 si x > threshold
    values: list = field(default_factory=list)   # nominal: valor que corresponde a cada rama
    children: list = field(default_factory=list)
    branch_probs: np.ndarray | None = None       # probabilidad de cada rama (reparte los faltantes)
    gain: float = 0.0
    gain_ratio: float = 0.0

    @property
    def is_leaf(self) -> bool:
        return not self.children

    @property
    def proba(self) -> np.ndarray:
        total = float(self.class_counts.sum())
        if total <= EPS:
            return np.full(len(self.class_counts), 1.0 / len(self.class_counts))
        return self.class_counts / total

    @property
    def errors(self) -> float:
        """Peso mal clasificado si este nodo fuera hoja (base de la poda por error pesimista)."""
        return float(self.n_weight - self.class_counts.max())


class C45DecisionTree:
    """Clasificador C4.5.

    Parámetros (todos ajustables desde el "laboratorio de parámetros" de la interfaz):
      criterion              "gain_ratio" (C4.5) o "gain" (ganancia de información pura, estilo ID3).
      max_depth              profundidad máxima (None = sin límite).
      min_samples_split      peso mínimo de un nodo para intentar dividirlo.
      min_samples_leaf       peso mínimo que deben tener al menos dos ramas (parámetro -m de C4.5).
      min_gain               ganancia mínima para aceptar una división (pre-poda).
      use_avg_gain           aplicar la heurística de ganancia promedio de C4.5.
      threshold_mode         "c45" (valor observado <= punto medio) o "midpoint" (punto medio).
      numeric_gain_correction  corrección log2(N-1)/|D| de C4.5 release 8 para atributos numéricos.
      prune                  aplicar la poda por error pesimista de C4.5 (post-poda; ver pruning.py).
      confidence             nivel de confianza CF de la poda, en (0, 0.5]: más chico poda más (C4.5: 0.25).
    """

    def __init__(self, criterion="gain_ratio", max_depth=None, min_samples_split=2,
                 min_samples_leaf=2, min_gain=0.0, use_avg_gain=True, threshold_mode="c45",
                 numeric_gain_correction=False, prune=False, confidence=DEFAULT_CONFIDENCE):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_gain = min_gain
        self.use_avg_gain = use_avg_gain
        self.threshold_mode = threshold_mode
        self.numeric_gain_correction = numeric_gain_correction
        self.prune = prune
        self.confidence = confidence

    # ------------------------------------------------------------------ parámetros
    def get_params(self, deep=True):
        return {name: getattr(self, name) for name in _PARAM_NAMES}

    def set_params(self, **params):
        for name, value in params.items():
            if name not in _PARAM_NAMES:
                raise ValueError(f"Parámetro desconocido: {name!r}. Válidos: {_PARAM_NAMES}")
            setattr(self, name, value)
        return self

    def __repr__(self):
        args = ", ".join(f"{k}={v!r}" for k, v in self.get_params().items())
        return f"C45DecisionTree({args})"

    def _validate_params(self):
        if self.criterion not in CRITERIA:
            raise ValueError(f"criterion debe ser uno de {CRITERIA}, no {self.criterion!r}")
        if self.threshold_mode not in THRESHOLD_MODES:
            raise ValueError(f"threshold_mode debe ser uno de {THRESHOLD_MODES}, no {self.threshold_mode!r}")
        if self.max_depth is not None and self.max_depth < 0:
            raise ValueError("max_depth debe ser >= 0 o None")
        if self.min_samples_split < 1:
            raise ValueError("min_samples_split debe ser >= 1")
        if self.min_samples_leaf < 1:
            raise ValueError("min_samples_leaf debe ser >= 1")
        if self.min_gain < 0:
            raise ValueError("min_gain debe ser >= 0")
        if not isinstance(self.prune, (bool, np.bool_)):
            raise ValueError(f"prune debe ser True o False, no {self.prune!r}")
        if not 0 < self.confidence <= 0.5:
            raise ValueError("confidence debe estar en (0, 0.5]")

    # ------------------------------------------------------------------ entrenamiento
    def fit(self, X, y, feature_types=None, sample_weight=None):
        """Entrena el árbol.

        X: DataFrame (o matriz) con los atributos. Los faltantes se marcan con NaN/None/pd.NA.
        y: etiquetas (cualquier tipo comparable: texto o números).
        feature_types: dict opcional {columna: "numeric" | "nominal"}. Si se omite, se infiere por
            dtype. IMPORTANTE: una columna numérica que en realidad es una categoría (p. ej. un
            código 1/2/3/4) debe declararse "nominal" para que el árbol ramifique por valor.
        sample_weight: pesos por caso (opcional).
        """
        self._validate_params()
        X = self._as_frame(X, fitting=True)
        y = np.asarray(y)
        if len(X) == 0:
            raise ValueError("No hay datos para entrenar (X está vacío).")
        if len(X) != len(y):
            raise ValueError(f"X tiene {len(X)} filas pero y tiene {len(y)}.")
        if pd.isna(y).any():
            raise ValueError("y tiene valores faltantes; elimine esas filas antes de entrenar.")
        weights = np.ones(len(X)) if sample_weight is None else np.asarray(sample_weight, dtype=float)
        if weights.shape != (len(X),) or (weights < 0).any():
            raise ValueError("sample_weight debe tener un valor >= 0 por fila.")

        self.feature_names_ = list(X.columns)
        self.feature_types_ = self._resolve_types(X, feature_types)
        self.classes_, y_idx = np.unique(y, return_inverse=True)
        y_idx = np.asarray(y_idx).reshape(-1)
        self.n_classes_ = len(self.classes_)

        keep = np.nonzero(weights > 0)[0]
        if keep.size == 0:
            raise ValueError("Todos los pesos son cero.")
        cols = self._columns(X)
        self.tree_ = self._grow(cols, y_idx, keep, weights[keep], depth=0)
        self.n_nodes_unpruned_ = self.n_nodes_
        self.n_pruned_ = prune_tree(self.tree_, self.confidence) if self.prune else 0   # subárboles -> hoja
        self._compute_importances()                    # después de podar: solo cuentan los cortes que quedaron
        return self

    def _grow(self, cols, y, idx, w, depth) -> Node:
        """Construye recursivamente el subárbol de los casos `idx` (con pesos `w`)."""
        counts = np.bincount(y[idx], weights=w, minlength=self.n_classes_)
        node = Node(n_weight=float(w.sum()), class_counts=counts, depth=depth)

        # Paso 1: criterios de parada -> hoja.
        pure = np.count_nonzero(counts > EPS) <= 1
        too_small = node.n_weight < self.min_samples_split - EPS
        too_deep = self.max_depth is not None and depth >= self.max_depth
        if pure or too_small or too_deep:
            return node

        # Paso 2: elegir la mejor división; si ninguna sirve -> hoja.
        cand = self._best_split(cols, y, idx, w, node.n_weight)
        if cand is None:
            return node

        # Paso 3: repartir los casos (los faltantes van a todas las ramas, con peso fraccionario).
        parts, probs = self._partition(cand, cols[cand.feature][idx], idx, w)
        node.feature, node.kind, node.threshold = cand.feature, cand.kind, cand.threshold
        node.values = list(cand.values) if cand.values is not None else []
        node.branch_probs = probs
        node.gain, node.gain_ratio = cand.gain, cand.gain_ratio

        # Paso 4: recursión sobre cada rama.
        node.children = [self._grow(cols, y, ci, cw, depth + 1) for ci, cw in parts]
        return node

    def _best_split(self, cols, y, idx, w, total_weight) -> SplitCandidate | None:
        """Regla de selección de C4.5 entre las mejores divisiones de cada atributo."""
        candidates = []
        for j, kind in enumerate(self.feature_types_):
            x = cols[j][idx]
            if kind == "numeric":
                cand = evaluate_numeric(j, x, y[idx], w, self.n_classes_, total_weight,
                                        self.min_samples_leaf, self.threshold_mode,
                                        self.numeric_gain_correction)
            else:
                cand = evaluate_nominal(j, x, y[idx], w, self.n_classes_, total_weight,
                                        self.min_samples_leaf)
            # Solo cuentan las divisiones con ganancia real y que no sean degeneradas (SplitInfo > 0).
            if cand is not None and cand.gain > self.min_gain + EPS and cand.split_info > EPS:
                candidates.append(cand)
        if not candidates:
            return None

        if self.criterion == "gain":                      # ganancia pura (estilo ID3)
            pool, key = candidates, (lambda c: c.gain)
        else:                                             # gain ratio (C4.5)
            pool = candidates
            if self.use_avg_gain:                         # heurística: ganancia >= promedio
                average = float(np.mean([c.gain for c in candidates]))
                pool = [c for c in candidates if c.gain >= average - 1e-9]
            key = lambda c: c.gain_ratio

        best = pool[0]                                    # empate -> gana el atributo de menor índice
        for c in pool[1:]:
            if key(c) > key(best) + 1e-12:
                best = c
        return best

    @staticmethod
    def _partition(cand, x, idx, w):
        """Reparte (idx, w) entre las ramas. Devuelve ([(idx_rama, w_rama), ...], prob. de cada rama)."""
        if cand.kind == "numeric":
            known = ~np.isnan(x)
            masks = [known & (x <= cand.threshold), known & (x > cand.threshold)]
        else:
            known = np.array([v is not None for v in x], dtype=bool)
            masks = [known & (x == v) for v in cand.values]
        probs = cand.branch_weights / cand.branch_weights.sum()
        missing = ~known
        parts = []
        for mask, p in zip(masks, probs):
            child_idx, child_w = idx[mask], w[mask]
            if missing.any():  # los faltantes pasan por todas las ramas con peso w * p
                child_idx = np.concatenate([child_idx, idx[missing]])
                child_w = np.concatenate([child_w, w[missing] * p])
            parts.append((child_idx, child_w))
        return parts, probs

    # ------------------------------------------------------------------ predicción
    def predict_proba(self, X) -> np.ndarray:
        """Probabilidad de cada clase (columnas en el orden de `classes_`)."""
        self._check_fitted()
        X = self._as_frame(X)
        cols = self._columns(X)
        n = len(X)
        out = np.zeros((n, self.n_classes_))
        self._route(self.tree_, cols, np.arange(n), np.ones(n), out)
        return out

    def predict(self, X):
        self._check_fitted()
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

    def score(self, X, y) -> float:
        return float(np.mean(self.predict(X) == np.asarray(y)))

    def _route(self, node, cols, idx, w, out):
        """Lleva un lote de casos por el árbol acumulando las probabilidades en `out`.

        Un caso con valor conocido sigue UNA rama (peso completo). Uno con valor faltante (o nominal
        no visto) sigue TODAS las ramas con peso w * prob_rama: así se mezclan las hojas alcanzadas.
        """
        if idx.size == 0:
            return
        if node.is_leaf:
            out[idx] += w[:, None] * node.proba
            return
        x = cols[node.feature][idx]
        if node.kind == "numeric":
            known = ~np.isnan(x)
            masks = [known & (x <= node.threshold), known & (x > node.threshold)]
        else:
            lookup = {v: i for i, v in enumerate(node.values)}
            branch = np.array([lookup.get(v, -1) if v is not None else -1 for v in x])
            known = branch >= 0
            masks = [branch == i for i in range(len(node.values))]
        missing = ~known
        for mask, child, p in zip(masks, node.children, node.branch_probs):
            take = mask | missing
            weights = np.where(mask[take], w[take], w[take] * p)
            self._route(child, cols, idx[take], weights, out)

    # ------------------------------------------------------------------ interpretación
    def export_text(self) -> str:
        """El árbol como texto (estilo J48/Weka). Entre paréntesis: casos (y errores) de la hoja."""
        self._check_fitted()

        def leaf_text(node):
            label = self.classes_[int(np.argmax(node.class_counts))]
            errors = f"/{node.errors:g}" if node.errors > 1e-9 else ""
            return f"{label} ({node.n_weight:g}{errors})"

        def conditions(node):
            name = self.feature_names_[node.feature]
            if node.kind == "numeric":
                return [f"{name} <= {node.threshold:g}", f"{name} > {node.threshold:g}"]
            return [f"{name} = {v}" for v in node.values]

        if self.tree_.is_leaf:
            return f": {leaf_text(self.tree_)}"
        lines = []

        def walk(node, depth):
            for cond, child in zip(conditions(node), node.children):
                prefix = "|   " * depth
                if child.is_leaf:
                    lines.append(f"{prefix}{cond}: {leaf_text(child)}")
                else:
                    lines.append(f"{prefix}{cond}")
                    walk(child, depth + 1)

        walk(self.tree_, 0)
        return "\n".join(lines)

    def explain(self, row) -> dict:
        """Explica la predicción de UN caso: reglas recorridas, clase y probabilidades.

        Si en algún nodo el valor falta (o es un nominal no visto), el recorrido se corta ahí y se
        avisa: la predicción final igual mezcla las ramas (ver _route).
        """
        self._check_fitted()
        frame = self._as_frame(pd.DataFrame([dict(row)]))
        cols = self._columns(frame)
        node, rules = self.tree_, []
        while not node.is_leaf:
            name = self.feature_names_[node.feature]
            value = cols[node.feature][0]
            if node.kind == "numeric":
                if np.isnan(value):
                    rules.append(f"{name}: valor faltante -> se reparte entre las ramas")
                    break
                go_left = value <= node.threshold
                rules.append(f"{name} {'<=' if go_left else '>'} {node.threshold:g}")
                node = node.children[0 if go_left else 1]
            else:
                if value is None or value not in node.values:
                    rules.append(f"{name}: valor faltante o no visto -> se reparte entre las ramas")
                    break
                rules.append(f"{name} = {value}")
                node = node.children[node.values.index(value)]
        proba = self.predict_proba(frame)[0]
        return {"rules": rules, "label": self.classes_[int(np.argmax(proba))],
                "proba": {str(c): float(p) for c, p in zip(self.classes_, proba)}}

    def to_dict(self, class_names=None) -> dict:
        """El árbol como estructura anidada con solo tipos de Python (ver export.py)."""
        return tree_to_dict(self, class_names)

    def to_dot(self, class_names=None) -> str:
        """El árbol en formato DOT de Graphviz, con el estilo de las láminas de la cátedra."""
        return tree_to_dot(self, class_names)

    def trace(self, row) -> dict:
        """{id de nodo: peso} con que un caso pasa por cada nodo (los ids son los de to_dict)."""
        return trace_case(self, row)

    # ------------------------------------------------------------------ estructura
    def _iter_nodes(self):
        stack = [self.tree_]
        while stack:
            node = stack.pop()
            yield node
            stack.extend(node.children)

    @property
    def depth_(self) -> int:
        self._check_fitted()
        return max(n.depth for n in self._iter_nodes())

    @property
    def n_nodes_(self) -> int:
        self._check_fitted()
        return sum(1 for _ in self._iter_nodes())

    @property
    def n_leaves_(self) -> int:
        self._check_fitted()
        return sum(1 for n in self._iter_nodes() if n.is_leaf)

    def _compute_importances(self):
        """Importancia de un atributo = suma de (peso del nodo x ganancia) en los nodos que lo usan."""
        importance = np.zeros(len(self.feature_names_))
        for node in self._iter_nodes():
            if not node.is_leaf:
                importance[node.feature] += node.n_weight * node.gain
        total = importance.sum()
        self.feature_importances_ = importance / total if total > EPS else importance

    # ------------------------------------------------------------------ utilidades internas
    def _check_fitted(self):
        if not hasattr(self, "tree_"):
            raise RuntimeError("El árbol no está entrenado: llame a fit() primero.")

    def _as_frame(self, X, fitting=False) -> pd.DataFrame:
        if isinstance(X, pd.DataFrame):
            frame = X.reset_index(drop=True).copy()
        else:
            frame = pd.DataFrame(X, columns=None if fitting else self.feature_names_)
        frame.columns = [str(c) for c in frame.columns]
        if not fitting:
            absent = [c for c in self.feature_names_ if c not in frame.columns]
            if absent:
                raise ValueError(f"Faltan columnas en X: {absent}")
            frame = frame[self.feature_names_]
        return frame

    @staticmethod
    def _resolve_types(X, feature_types):
        given = {str(k): v for k, v in (feature_types or {}).items()}
        unknown = set(given) - set(X.columns)
        if unknown:
            raise ValueError(f"feature_types menciona columnas inexistentes: {sorted(unknown)}")
        types = []
        for column in X.columns:
            if column in given:
                kind = given[column]
                if kind not in FEATURE_KINDS:
                    raise ValueError(f"Tipo inválido para {column!r}: {kind!r}. Use {FEATURE_KINDS}.")
            else:
                s = X[column]
                is_number = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)
                kind = "numeric" if is_number else "nominal"
            types.append(kind)
        return types

    def _columns(self, X):
        """Una columna por atributo: numéricos como float (NaN = faltante), nominales como objetos
        (None = faltante). Un texto no numérico en una columna numérica (p. ej. '?') pasa a NaN."""
        cols = []
        for column, kind in zip(self.feature_names_, self.feature_types_):
            s = X[column]
            if kind == "numeric":
                cols.append(pd.to_numeric(s, errors="coerce").to_numpy(dtype=float))
            else:
                cols.append(np.array([None if pd.isna(v) else _tidy(v) for v in s], dtype=object))
        return cols


In [ ]:
%%writefile src/tp_ia/c45/pruning.py
"""
Poda por error pesimista de C4.5 (Quinlan, 1993, cap. 4).

GUÍA DE LECTURA (para el coloquio)
----------------------------------
El problema: un árbol que crece hasta clasificar bien TODOS los casos de entrenamiento memoriza el ruido.
Cada hoja "perfecta" que cubre pocos casos es en realidad una apuesta frágil.

La idea de Quinlan: el error medido sobre los mismos casos con que se construyó el árbol es OPTIMISTA.
Si una hoja cubre N casos y falla en E, el error verdadero probablemente sea mayor. Se lo estima con el
LÍMITE SUPERIOR de un intervalo de confianza binomial (nivel CF, 25 % por defecto): U_CF(E, N). Los
"errores estimados" de la hoja son N * U_CF(E, N) = E + AddErrs(N, E).
  - Con pocos casos el límite es muy holgado (una hoja de 1 caso sin errores estima 0.75 errores); con
    muchos casos se acerca al error observado. Por eso la poda castiga sobre todo a las hojas chicas.
  - Bajar CF poda MÁS (el límite superior se vuelve más pesimista); subirlo poda menos.

El algoritmo, de abajo hacia arriba, en cada nodo interno:
  (a) errores estimados si el nodo fuera una HOJA   -> E_nodo + AddErrs(N_nodo, E_nodo)
  (b) errores estimados del SUBÁRBOL                -> suma de lo estimado en cada hijo (ya podado)
  Si (a) <= (b) + 0.1 el subárbol se reemplaza por una hoja. El 0.1 es la tolerancia de C4.5: ante un
  empate técnico se prefiere el árbol más chico.
No hacen falta datos aparte: se usa la distribución de clases de entrenamiento que ya guarda cada nodo
(con los pesos fraccionarios de los faltantes, igual que en la construcción).

AddErrs (versión del código de C4.5 release 8)
  - E = 0:          N * (1 - CF^(1/N))     límite binomial exacto cuando no hubo errores.
  - 0 < E < 1:      interpolación lineal entre el caso E = 0 y el caso E = 1 (hay errores fraccionarios
                    cuando los faltantes reparten casos).
  - E + 0.5 >= N:   0.67 * (N - E)         caso de borde.
  - resto:          aproximación normal con corrección de continuidad. El coeficiente z sale de interpolar
                    una tabla de desvíos normales (para CF = 25 % da z = 0.6925; el cuantil exacto sería 0.6745).

QUÉ ESTÁ VERIFICADO Y QUÉ NO (para la tabla del informe)
  - U_25%(0, 6) = 0.206 coincide con el valor que cita Quinlan en su libro (visto en una cita pública).
    U_25%(0, 9) = 0.143, U_25%(0, 1) = 0.750 y U_25%(1, 16) = 0.157 son los otros valores de su ejemplo,
    tomados de memoria: esta fórmula los reproduce a tres decimales, pero no se contrastaron con el libro.
  - La tolerancia 0.1 y la tabla de desvíos son las de C4.5 tal como las recordamos (Weka/J48 usa también
    0.1, pero con z = 0.6745 exacto y otro caso de borde, así que sus podas pueden diferir apenas).
  - La constante 0.67 del caso de borde NO se pudo contrastar con el código original. Solo pesa en nodos
    con peso fraccionario muy chico, y se deja como constante con nombre (EDGE_FACTOR) para poder cambiarla.

DIFERENCIA CON C4.5 COMPLETO
  - No se implementa "subtree raising" (reemplazar un nodo por su rama más poblada), que C4.5 aplica
    además de reemplazar un subárbol por una hoja.
"""
from __future__ import annotations

import math

import numpy as np

from .criteria import EPS

CF_TABLE = (0.0, 0.001, 0.005, 0.01, 0.05, 0.10, 0.20, 0.40, 1.00)    # nivel de confianza (cola superior)
Z_TABLE = (4.0, 3.09, 2.58, 2.33, 1.65, 1.28, 0.84, 0.25, 0.00)       # desvío normal correspondiente
EDGE_FACTOR = 0.67        # caso e + 0.5 >= N (sin contrastar con el código original: ver arriba)
TOLERANCE = 0.1           # el subárbol sobrevive solo si mejora a la hoja por más de esto
DEFAULT_CONFIDENCE = 0.25


def confidence_coefficient(confidence: float = DEFAULT_CONFIDENCE) -> float:
    """z² para el nivel de confianza dado, interpolando linealmente la tabla de C4.5."""
    if not 0 < confidence <= 1:
        raise ValueError("confidence debe estar en (0, 1]")
    z = float(np.interp(confidence, CF_TABLE, Z_TABLE))
    return z * z


def added_errors(n: float, e: float, confidence: float = DEFAULT_CONFIDENCE) -> float:
    """AddErrs de C4.5: errores que se SUMAN a los E observados en N casos para obtener el límite superior."""
    if n <= EPS:
        return 0.0
    if e < 1e-6:                                    # sin errores: límite binomial exacto
        return n * (1.0 - confidence ** (1.0 / n))
    if e < 0.9999:                                  # error fraccionario: interpolar entre e = 0 y e = 1
        base = n * (1.0 - confidence ** (1.0 / n))
        return base + e * (added_errors(n, 1.0, confidence) - base)
    if e + 0.5 >= n:                                # casi todos mal clasificados
        return EDGE_FACTOR * (n - e)
    z2 = confidence_coefficient(confidence)
    upper = (e + 0.5 + z2 / 2 + math.sqrt(z2 * ((e + 0.5) * (1 - (e + 0.5) / n) + z2 / 4))) / (n + z2)
    return n * upper - e


def estimated_errors(n: float, e: float, confidence: float = DEFAULT_CONFIDENCE) -> float:
    """Errores estimados de una hoja con N casos y E errores observados: E + AddErrs(N, E)."""
    return e + added_errors(n, e, confidence)


def _make_leaf(node) -> None:
    """Convierte un nodo interno en hoja conservando su distribución de clases."""
    node.children = []
    node.feature = node.kind = node.threshold = node.branch_probs = None
    node.values = []
    node.gain = node.gain_ratio = 0.0


def prune_tree(root, confidence: float = DEFAULT_CONFIDENCE, tolerance: float = TOLERANCE) -> int:
    """Poda `root` en el lugar (de abajo hacia arriba). Devuelve cuántos subárboles se reemplazaron por una hoja."""
    if not 0 < confidence <= 0.5:
        raise ValueError("confidence debe estar en (0, 0.5]")
    collapsed = 0

    def visit(node) -> float:
        nonlocal collapsed
        as_leaf = estimated_errors(node.n_weight, node.errors, confidence)
        if node.is_leaf:
            return as_leaf
        as_subtree = sum(visit(child) for child in node.children)    # los hijos ya vienen podados
        if as_leaf <= as_subtree + tolerance:
            _make_leaf(node)
            collapsed += 1
            return as_leaf
        return as_subtree

    visit(root)
    return collapsed


In [ ]:
%%writefile src/tp_ia/c45/export.py
"""
Exportadores del árbol: estructura anidada (para la interfaz web), DOT de Graphviz y traza de un caso.

GUÍA DE LECTURA
---------------
  - tree_to_dict(): el árbol como diccionario anidado con solo tipos de Python (se puede pasar a json.dumps).
    Cada nodo lleva un `id` en PREORDEN (raíz = 0, luego el subárbol del primer hijo, etc.): la interfaz y
    trace_case() usan ese mismo id para hablar del mismo nodo.
  - tree_to_dot(): el árbol en el lenguaje de Graphviz con el formato de las láminas de la cátedra: nodos
    internos en amarillo con entropía, ganancia, razón de ganancia y n; hojas coloreadas por clase.
  - trace_case(): por qué nodos pasa UN caso y con qué peso. Si un valor falta (o es un nominal que el árbol
    no vio), el caso se reparte entre las ramas según branch_probs, igual que en predict_proba.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .criteria import entropy

LEAF_COLORS = ("#9ECAE1", "#B7E2A3", "#F4B8A0", "#D6C2E8", "#F2D78C", "#BFE3DC")   # una por clase, en orden
INTERNAL_COLOR = "#FFF2AE"


def _py(value):
    """Escalar de numpy -> escalar de Python (para que json.dumps no falle)."""
    return value.item() if hasattr(value, "item") else value


def _preorder(root) -> list:
    nodes, stack = [], [root]
    while stack:
        node = stack.pop()
        nodes.append(node)
        stack.extend(reversed(node.children))      # el primer hijo se visita primero
    return nodes


def _class_names(tree, class_names) -> list:
    names = class_names or {}
    return [str(names.get(_py(c), _py(c))) for c in tree.classes_]


def _edge_labels(node) -> list:
    if node.kind == "numeric":
        return [f"≤ {node.threshold:g}", f"> {node.threshold:g}"]
    return [str(_py(v)) for v in node.values]


def tree_to_dict(tree, class_names=None) -> dict:
    """El árbol entrenado como estructura anidada JSON-segura. `class_names` = {clase: nombre para mostrar}."""
    tree._check_fitted()
    names = _class_names(tree, class_names)
    ids = {id(node): i for i, node in enumerate(_preorder(tree.tree_))}

    def build(node) -> dict:
        total = float(node.class_counts.sum())
        out = {
            "id": ids[id(node)], "depth": int(node.depth), "n": float(node.n_weight), "leaf": node.is_leaf,
            "counts": [float(c) for c in node.class_counts],
            "proba": [float(p) for p in node.proba],
            "label": names[int(np.argmax(node.class_counts))],
            "entropy": float(entropy(node.class_counts)) if total > 0 else 0.0,
            "errors": float(node.errors),
        }
        if not node.is_leaf:
            out.update({
                "feature": tree.feature_names_[node.feature], "feature_index": int(node.feature), "kind": node.kind,
                "threshold": None if node.threshold is None else float(node.threshold),
                "edges": _edge_labels(node),
                "branch_probs": [float(p) for p in node.branch_probs],
                "gain": float(node.gain), "gain_ratio": float(node.gain_ratio),
                "children": [build(child) for child in node.children],
            })
        return out

    return {"classes": [{"value": _py(c), "name": n} for c, n in zip(tree.classes_, names)],
            "root": build(tree.tree_), "n_nodes": int(tree.n_nodes_), "n_leaves": int(tree.n_leaves_),
            "depth": int(tree.depth_), "n_pruned": int(getattr(tree, "n_pruned_", 0)),
            "n_nodes_unpruned": int(getattr(tree, "n_nodes_unpruned_", tree.n_nodes_))}


def _dot_escape(text) -> str:
    return str(text).replace("\\", "\\\\").replace('"', '\\"').replace("\n", "\\n")


def tree_to_dot(tree, class_names=None) -> str:
    """El árbol en formato DOT (Graphviz), con el estilo de las láminas de la cátedra."""
    data = tree_to_dict(tree, class_names)
    names = [c["name"] for c in data["classes"]]
    lines = ["digraph Arbol {",
             '  graph [rankdir=TB, fontname="Helvetica"];',
             '  node [shape=box, style="rounded,filled", fontname="Helvetica", color="#222222"];',
             '  edge [fontname="Helvetica", fontsize=10];']
    pending = [data["root"]]
    while pending:
        node = pending.pop(0)
        if node["leaf"]:
            label = f"{node['label']}\nn = {node['n']:g}"
            if node["errors"] > 1e-9:
                label += f"\nerrores = {node['errors']:g}"
            color = LEAF_COLORS[names.index(node["label"]) % len(LEAF_COLORS)]
        else:
            label = (f"{node['feature']}\nentropía = {node['entropy']:.3f}\nganancia = {node['gain']:.3f}\n"
                     f"razón ganancia = {node['gain_ratio']:.3f}\nn = {node['n']:g}")
            color = INTERNAL_COLOR
        lines.append(f'  n{node["id"]} [label="{_dot_escape(label)}", fillcolor="{color}"];')
        for edge, child in zip(node.get("edges", []), node.get("children", [])):
            lines.append(f'  n{node["id"]} -> n{child["id"]} [label="{_dot_escape(edge)}"];')
            pending.append(child)
    lines.append("}")
    return "\n".join(lines)


def trace_case(tree, row) -> dict:
    """{id de nodo: peso} con que UN caso pasa por cada nodo (un caso completo recorre un solo camino).

    `row`: dict {atributo: valor}; un valor faltante es None o NaN. Si falta (o es un nominal no visto) en un
    nodo, el caso se reparte entre sus ramas según branch_probs: los pesos de los hijos suman el del padre.
    """
    tree._check_fitted()
    ids = {id(node): i for i, node in enumerate(_preorder(tree.tree_))}
    frame = tree._as_frame(pd.DataFrame([dict(row)]))
    cols = tree._columns(frame)
    weights: dict = {}

    def visit(node, weight: float) -> None:
        weights[ids[id(node)]] = weight
        if node.is_leaf:
            return
        value = cols[node.feature][0]
        branch = None
        if node.kind == "numeric":
            if not np.isnan(value):
                branch = 0 if value <= node.threshold else 1
        elif value is not None and value in node.values:
            branch = node.values.index(value)
        for i, child in enumerate(node.children):
            if branch is None:
                visit(child, weight * float(node.branch_probs[i]))
            elif i == branch:
                visit(child, weight)
            else:
                skip(child)

    def skip(node) -> None:                   # las ramas que el caso no toca figuran con peso 0
        weights[ids[id(node)]] = 0.0
        for child in node.children:
            skip(child)

    visit(tree.tree_, 1.0)
    return weights


### 1.2 Los datos (`tp_ia.data`)

Cargan y revisan los datos, y dejan escrito qué se decidió sobre ellos.

- `__init__.py`: lo que se puede importar del paquete de datos.
- `schema.py`: cómo se describe un dataset: columnas, tipos de atributo y escenarios de la variable objetivo.
- `loader.py`: un cargador que sirve para cualquier dataset: detecta el separador, los decimales y los faltantes.
- `quality.py`: revisa la calidad de los datos: faltantes, duplicados y valores extremos.
- `decisions.py`: el registro de las decisiones que se toman sobre los datos (D-01 a D-07).
- `fixtures.py`: ejemplos chicos para probar el árbol: Play Tennis y el ejemplo de Liu.
- `iris.py`: carga Iris.
- `heart.py`: carga Heart Disease (Cleveland), con nombres en español y los códigos reemplazados por su significado.

In [ ]:
%%writefile src/tp_ia/data/__init__.py
"""Datos: cargador agnóstico, auditoría, registro de decisiones, Iris, enfermedad cardíaca y fixtures de juguete."""
from .decisions import DecisionLog
from .fixtures import liu_loans, play_tennis
from .heart import load_heart_dataset
from .iris import load_iris_dataset
from .loader import Dataset, build_dataset, dataset_from_frame, load_dataset, read_table
from .quality import audit
from .schema import DatasetSpec, Scenario

__all__ = ["Dataset", "DatasetSpec", "DecisionLog", "Scenario", "audit", "build_dataset",
           "dataset_from_frame", "liu_loans", "load_dataset", "load_heart_dataset", "load_iris_dataset", "play_tennis",
           "read_table"]


In [ ]:
%%writefile src/tp_ia/data/schema.py
"""
Descripción DECLARATIVA de un dataset: qué columnas hay, cuál es el objetivo, cuáles son nominales,
qué valores significan "faltante" y qué escenarios de objetivo se pueden armar.

GUÍA DE LECTURA
---------------
Cuando llegue un dataset nuevo (el de la cátedra, uno subido desde la interfaz) no se toca el código:
se escribe un DatasetSpec y listo. El cargador (loader.py) hace el resto y deja registrado en un
DecisionLog cada decisión que tomó (qué se trató como faltante, qué columnas son nominales, etc.).
"""
from __future__ import annotations

from dataclasses import dataclass, field

# Textos que, en un CSV, suelen querer decir "no hay dato".
DEFAULT_MISSING_TOKENS = ("?", "", "NA", "N/A", "NaN", "nan", "null", "NULL", "None", "-", "--")


@dataclass(frozen=True)
class Scenario:
    """Una forma de definir la variable objetivo a partir de la columna original.

    mapping:      valor original -> clase nueva (None = se deja la columna como viene).
    class_names:  clase nueva -> nombre legible para tablas y gráficos.
    goal:         qué se predice, como acción ("predecir si ..."); es el título de la decisión en el registro.
    """

    name: str
    mapping: dict | None = None
    class_names: dict | None = None
    description: str = ""
    goal: str = ""


@dataclass
class DatasetSpec:
    name: str
    target: str                                   # nombre de la columna objetivo
    columns: list | None = None                   # nombres, si el archivo NO trae encabezado
    header: bool = True                           # ¿la primera fila es el encabezado?
    nominal: list = field(default_factory=list)   # columnas nominales (códigos sin orden)
    zero_is_missing: list = field(default_factory=list)  # columnas donde un 0 es un faltante disfrazado
    drop: list = field(default_factory=list)      # columnas a descartar (IDs, nombres, ...)
    missing_tokens: tuple = DEFAULT_MISSING_TOKENS
    scenarios: dict = field(default_factory=dict)  # nombre -> Scenario
    source: str = ""                              # de dónde salen los datos; completa la frase "Los datos provienen de ..."
    citation: str = ""
    purpose: str = ""                             # por qué se eligió este dataset (aparece en la primera decisión)
    value_labels: dict = field(default_factory=dict)  # columna -> {código: texto}; por ejemplo {"sexo": {0: "mujer"}}
    descriptions: dict = field(default_factory=dict)  # columna -> qué mide, en lenguaje sencillo (se ve en la interfaz)


In [ ]:
%%writefile src/tp_ia/data/loader.py
"""
Cargador de datasets AGNÓSTICO: lee casi cualquier CSV y deja registrado qué decidió.

GUÍA DE LECTURA
---------------
Dos etapas, a propósito separadas:
  1. read_table(): archivo -> DataFrame. Adivina separador (, ; tab |), coma decimal y codificación
     (utf-8 o latin-1), recorta espacios y convierte los textos de "falta dato" (?, NA, vacío, ...)
     en faltantes reales. Una columna se vuelve numérica solo si TODOS sus valores lo son.
  2. build_dataset(): DataFrame + DatasetSpec -> Dataset (X, y, tipos, auditoría, registro de
     decisiones). Aplica el escenario de objetivo, los ceros-faltantes, la política de faltantes
     y de duplicados.

El dataset de la cátedra, uno subido desde la interfaz o los archivos de UCI pasan por el mismo camino.
"""
from __future__ import annotations

import codecs
import re
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

from .decisions import DecisionLog
from .quality import audit
from .schema import DEFAULT_MISSING_TOKENS, DatasetSpec

_SEP_CANDIDATES = (",", ";", "\t", "|")
MISSING_POLICIES = ("keep", "drop_rows")
DUPLICATE_POLICIES = ("keep", "drop")


# ------------------------------------------------------------------ etapa 1: archivo -> DataFrame
def _read_sample(path, nbytes: int = 65536):
    raw = Path(path).read_bytes()[:nbytes]
    try:  # decodificador incremental: tolera que la muestra corte un carácter a la mitad
        return codecs.getincrementaldecoder("utf-8-sig")().decode(raw, final=False), "utf-8-sig"
    except UnicodeDecodeError:
        return raw.decode("latin-1"), "latin-1"


def _detect_sep(sample: str) -> str:
    """El separador es el que aparece la MISMA cantidad de veces en todas las primeras líneas."""
    lines = [line for line in sample.splitlines() if line.strip()][:20]
    if not lines:
        return ","
    best, best_count = ",", 0
    for sep in _SEP_CANDIDATES:
        counts = [line.count(sep) for line in lines]
        if counts[0] > 0 and len(set(counts)) == 1 and counts[0] > best_count:
            best, best_count = sep, counts[0]
    if best_count == 0:  # ninguno es consistente: nos quedamos con el más frecuente
        totals = {sep: sum(line.count(sep) for line in lines) for sep in _SEP_CANDIDATES}
        best = max(totals, key=totals.get) if max(totals.values()) > 0 else ","
    return best


def _detect_decimal(sample: str, sep: str) -> str:
    if sep != "," and re.search(r"\d,\d", sample) and not re.search(r"\d\.\d", sample):
        return ","
    return "."


def _convert_column(series: pd.Series, tokens: set, decimal: str) -> pd.Series:
    # Se trabaja con listas de Python (None = faltante): pandas 3 representa los faltantes de una columna
    # de texto como NaN y pandas 2 como None; así el resultado no depende de la versión.
    values = [None if (pd.isna(v) or str(v).strip() in tokens) else str(v).strip() for v in series]
    numeric_text = values if decimal == "." else [None if v is None else v.replace(",", ".") for v in values]
    try:
        return pd.Series(pd.to_numeric(pd.Series(numeric_text, dtype=object)).to_numpy(), index=series.index)
    except (ValueError, TypeError):
        return pd.Series(values, index=series.index, dtype=object)   # hay texto: queda como texto (nominal)


def read_table(path, header: bool = True, columns=None, missing_tokens=DEFAULT_MISSING_TOKENS,
               sep=None, decimal=None, encoding=None) -> pd.DataFrame:
    """Lee un CSV/TSV 'sucio' y lo devuelve con faltantes reales (NaN/None) y tipos razonables."""
    sample, detected = _read_sample(path)
    encoding = encoding or detected
    sep = sep or _detect_sep(sample)
    decimal = decimal or _detect_decimal(sample, sep)

    def read(enc):
        return pd.read_csv(path, sep=sep, encoding=enc, header=0 if header else None,
                           names=None if header else columns, dtype=str, keep_default_na=False,
                           skipinitialspace=True)
    try:
        raw = read(encoding)
    except UnicodeDecodeError:   # había bytes inválidos más allá de la muestra
        raw = read("latin-1")
    raw.columns = [str(c).strip() for c in raw.columns]
    tokens = {t.strip() for t in missing_tokens}
    return pd.DataFrame({c: _convert_column(raw[c], tokens, decimal) for c in raw.columns})


# ------------------------------------------------------------------ etapa 2: DataFrame -> Dataset
@dataclass
class Dataset:
    name: str
    X: pd.DataFrame
    y: pd.Series
    feature_types: dict
    target: str
    scenario: str | None
    class_names: dict
    classes: list
    log: DecisionLog
    report: dict
    source: str = ""
    descriptions: dict = field(default_factory=dict)   # columna -> qué mide (para la interfaz)

    def summary(self) -> dict:
        return {"name": self.name, "scenario": self.scenario, "n_rows": len(self.X),
                "n_features": self.X.shape[1], "classes": self.report["class_counts"],
                "missing_cells": int(self.X.isna().sum().sum()), "duplicates": self.report["duplicates"],
                "warnings": len(self.report["warnings"])}


def _resolve_types(X: pd.DataFrame, nominal) -> dict:
    unknown = set(nominal) - set(map(str, X.columns))
    if unknown:
        raise ValueError(f"Columnas nominales que no existen en los datos: {sorted(unknown)}")
    types = {}
    for column in X.columns:
        series = X[column]
        if str(column) in nominal:
            types[str(column)] = "nominal"
        elif pd.api.types.is_numeric_dtype(series) and not pd.api.types.is_bool_dtype(series):
            types[str(column)] = "numeric"
        else:
            types[str(column)] = "nominal"
    return types


def build_dataset(df: pd.DataFrame, spec: DatasetSpec, scenario=None, missing_policy="keep",
                  duplicates="keep", log=None, source="") -> Dataset:
    if missing_policy not in MISSING_POLICIES:
        raise ValueError(f"missing_policy debe ser uno de {MISSING_POLICIES}, no {missing_policy!r}")
    if duplicates not in DUPLICATE_POLICIES:
        raise ValueError(f"duplicates debe ser uno de {DUPLICATE_POLICIES}, no {duplicates!r}")
    log = log if log is not None else DecisionLog()
    df = df.copy()
    df.columns = [str(c) for c in df.columns]
    if spec.target not in df.columns:
        raise ValueError(f"La columna objetivo {spec.target!r} no está en los datos. Columnas: {list(df.columns)}")

    file_name = Path(str(source)).name
    chosen = f"usar el dataset «{spec.name}»" + (f" (archivo {file_name})" if "." in file_name else "")
    why = [spec.purpose, f"Los datos provienen de {spec.source}." if spec.source else ""]
    log.add("D-01", "Fuente de los datos", chosen, rationale=" ".join(p for p in why if p),
            evidence={"filas": len(df), "columnas": df.shape[1], "referencia": spec.citation or "—"})

    if spec.drop:
        df = df.drop(columns=[c for c in spec.drop if c in df.columns])

    # --- códigos con nombre: 4 -> "sin síntomas" (se registra en D-05) ---
    translated = {}
    for column, labels in spec.value_labels.items():
        if column in df.columns:
            df[column] = df[column].map(lambda v, labels=labels: _label(v, labels))
            translated[column] = len(labels)

    # --- D-03: ceros que en realidad son faltantes ---
    for column in spec.zero_is_missing:
        if column in df.columns:
            zeros = df[column] == 0
            if int(zeros.sum()):
                df[column] = df[column].astype(float).mask(zeros)
                log.add("D-03", f"Ceros en «{column}»", f"tratar los ceros de «{column}» como valores faltantes",
                        rationale="En esta variable, un 0 es un valor imposible. Por eso se interpreta como un dato faltante que se registró como 0.",
                        alternatives=["considerarlos valores reales"], evidence={"filas afectadas": int(zeros.sum())})

    # --- D-02: objetivo y escenario ---
    y_raw = df[spec.target]
    has_target = y_raw.notna()
    if (~has_target).any():
        log.add("D-02", "Filas sin objetivo", "eliminar las filas que no tienen la clase", rationale="No se puede entrenar ni evaluar un modelo con casos que no tienen la clase.",
                evidence={"filas eliminadas": int((~has_target).sum())})
        df, y_raw = df[has_target], y_raw[has_target]
    scenario_obj = None
    y = y_raw
    if scenario is not None:
        scenario_obj = spec.scenarios.get(scenario)
        if scenario_obj is None:
            raise ValueError(f"Escenario {scenario!r} desconocido. Disponibles: {sorted(spec.scenarios)}")
        if scenario_obj.mapping is not None:
            unknown = sorted(set(y_raw.unique()) - set(scenario_obj.mapping), key=str)
            if unknown:
                raise ValueError(f"Valores del objetivo sin mapeo en el escenario {scenario!r}: {unknown}")
            y = y_raw.map(scenario_obj.mapping)
    X = df.drop(columns=[spec.target])

    # --- D-05: tipos de atributo ---
    feature_types = _resolve_types(X, set(spec.nominal))
    log.add("D-05", "Tipos de atributo", "dividir los atributos nominales por valor y los numéricos por umbral",
            rationale="Un código sin orden, como 1, 2, 3 o 4 para distintas categorías, no debe tratarse como un número. "
                      "Si se tratara así, el árbol haría cortes sin sentido, como «categoría ≤ 2».",
            alternatives=["deducir el tipo de cada atributo a partir del tipo de dato"],
            evidence={"atributos nominales": [c for c, t in feature_types.items() if t == "nominal"],
                      "atributos numéricos": [c for c, t in feature_types.items() if t == "numeric"],
                      **({"códigos reemplazados por su significado": list(translated)} if translated else {})})

    # --- D-03: política de faltantes ---
    n_missing_rows = int(X.isna().any(axis=1).sum())
    per_column = {c: int(k) for c, k in X.isna().sum().items() if k}
    X, y = X.reset_index(drop=True), y.reset_index(drop=True)
    if missing_policy == "drop_rows":
        keep = ~X.isna().any(axis=1)
        X, y = X[keep].reset_index(drop=True), y[keep].reset_index(drop=True)
        log.add("D-03", "Valores faltantes", "eliminar las filas con algún valor faltante",
                rationale="Es la opción más simple, pero se pierden datos y los resultados pueden sesgarse si los "
                          "faltantes no aparecen al azar.",
                alternatives=["conservarlas, ya que el árbol puede repartir esos casos entre las ramas y el bosque puede "
                              "completar los datos usando solo el entrenamiento"],
                evidence={"filas con faltantes": n_missing_rows, "filas eliminadas": int((~keep).sum()),
                          "faltantes por columna": per_column})
    else:
        log.add("D-03", "Valores faltantes", "conservar las filas con valores faltantes",
                rationale="El árbol propio maneja los faltantes como C4.5, repartiendo el caso entre las ramas. El bosque, en "
                          "cambio, completa los datos faltantes con valores calculados solo con el conjunto de entrenamiento, "
                          "para no usar información de las pruebas.",
                alternatives=["eliminar las filas con valores faltantes"],
                evidence={"filas con faltantes": n_missing_rows, "faltantes por columna": per_column})

    # --- D-04: duplicados ---
    both = pd.concat([X, y.rename("__objetivo__")], axis=1)
    n_dup = int(both.duplicated().sum())
    if duplicates == "drop" and n_dup:
        keep = ~both.duplicated()
        X, y = X[keep].reset_index(drop=True), y[keep].reset_index(drop=True)
    dropped = duplicates == "drop"
    log.add("D-04", "Filas duplicadas", "eliminar las filas duplicadas" if dropped else "conservar las filas duplicadas",
            rationale=("Si una fila repetida queda a la vez en el entrenamiento y en la prueba, el modelo ya «vio» ese caso "
                       "y las métricas salen mejores de lo que realmente son."
                       if dropped else
                       "Pueden ser casos reales que se repiten, así que borrarlos cambiaría los datos originales. Hay que "
                       "tener en cuenta que, si una copia queda en el entrenamiento y otra en la prueba, las métricas "
                       "pueden salir algo mejores de lo que realmente son."),
            alternatives=["conservarlas" if dropped else "eliminarlas", "mantener las copias juntas al dividir los datos"],
            evidence={"filas duplicadas": n_dup})

    classes = sorted(y.unique().tolist())
    report = audit(X, y, feature_types)
    log.add("D-06", "Valores extremos", "conservar los valores extremos",
            rationale="Un valor extremo pero posible suele ser una medición real, y borrarlo o recortarlo sin conocer el "
                      "dominio sería inventar datos. Además, los árboles dividen por umbrales, así que un valor muy "
                      "grande cae en la misma rama que uno grande y no distorsiona el modelo. Los valores imposibles, "
                      "como un 0 donde no puede haberlo, no se consideran extremos sino faltantes, y se tratan en la "
                      "decisión sobre valores faltantes.",
            alternatives=["eliminar las filas con valores extremos", "recortarlos hasta un valor límite (winsorización)"],
            evidence={"criterio": f"se considera extremo un valor fuera de [Q1 - {report['outlier_factor']}·IQR, "
                                  f"Q3 + {report['outlier_factor']}·IQR] (regla de Tukey)",
                      "valores extremos": sum(i["count"] for i in report["outliers"].values()),
                      "por columna": {c: i["count"] for c, i in report["outliers"].items()}})
    log.add("D-02", "Variable objetivo",
            (scenario_obj.goal or f"usar el escenario «{scenario}» como variable objetivo") if scenario
            else f"predecir la columna «{spec.target}» con sus clases originales",
            rationale=scenario_obj.description if scenario_obj
            else "Las clases originales ya sirven para el problema, así que no hace falta agruparlas ni transformarlas.",
            alternatives=[f"usar el escenario «{s}»" for s in (sorted(spec.scenarios) if spec.scenarios else []) if s != scenario],
            evidence={"casos por clase": report["class_counts"]})
    return Dataset(name=spec.name, X=X, y=y, feature_types=feature_types, target=spec.target, scenario=scenario,
                   class_names=dict(scenario_obj.class_names or {}) if scenario_obj else {}, classes=classes,
                   log=log, report=report, source=str(source),
                   descriptions={c: d for c, d in spec.descriptions.items() if c in X.columns})


def _label(value, labels: dict):
    """Reemplaza un código por su significado. 4.0 busca la clave 4; un código desconocido o un faltante quedan igual."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return value
    key = int(value) if isinstance(value, float) and value.is_integer() else value
    return labels.get(key, value)


def load_dataset(path, spec: DatasetSpec, scenario=None, missing_policy="keep", duplicates="keep",
                 log=None) -> Dataset:
    """Archivo + especificación -> Dataset listo para entrenar."""
    df = read_table(path, header=spec.header, columns=spec.columns, missing_tokens=spec.missing_tokens)
    return build_dataset(df, spec, scenario, missing_policy, duplicates, log, source=str(path))


def dataset_from_frame(df: pd.DataFrame, target: str, nominal=None, name="datos", **kwargs) -> Dataset:
    """Atajo para un DataFrame que ya está en memoria (por ejemplo, un archivo subido desde la interfaz)."""
    spec = DatasetSpec(name=name, target=target, nominal=list(nominal or []))
    return build_dataset(df, spec, **kwargs)


In [ ]:
%%writefile src/tp_ia/data/quality.py
"""
Auditoría de calidad de un dataset ya cargado: faltantes, duplicados, clases, columnas raras.

GUÍA DE LECTURA
---------------
audit() devuelve un diccionario con los números y una lista de `warnings` en castellano. Es lo que
muestra la pantalla "Datos" de la interfaz y lo que respalda las decisiones D-03 (faltantes), D-04
(duplicados) y D-06 (valores extremos, por la regla de Tukey: solo se informan, no se tocan). Los
umbrales son parámetros: no hay números mágicos escondidos.
"""
from __future__ import annotations

import pandas as pd


def _py(value):
    return value.item() if hasattr(value, "item") else value


def _tukey_outliers(X: pd.DataFrame, feature_types: dict, factor: float) -> dict:
    """Valores fuera de [Q1 - factor*IQR, Q3 + factor*IQR] por columna numérica (regla de Tukey).

    Solo informa: no es un error ni se descarta nada (decisión D-06). Si Q1 == Q3 no hay dispersión
    central y la regla marcaría como 'extremo' a cualquier valor distinto, así que esa columna se saltea.
    """
    found = {}
    for column in X.columns:
        if feature_types.get(column) != "numeric":
            continue
        values = pd.to_numeric(X[column], errors="coerce").dropna()
        if len(values) < 4:
            continue
        q1, q3 = values.quantile(0.25), values.quantile(0.75)
        spread = q3 - q1
        if spread <= 0:
            continue
        low, high = q1 - factor * spread, q3 + factor * spread
        count = int(((values < low) | (values > high)).sum())
        if count:
            found[str(column)] = {"count": count, "low": round(float(low), 3), "high": round(float(high), 3)}
    return found


def audit(X: pd.DataFrame, y: pd.Series, feature_types: dict, small_class: int = 15,
          high_missing: float = 0.30, imbalance_ratio: float = 5.0, outlier_factor: float = 1.5) -> dict:
    n = len(X)
    counts = y.value_counts().sort_index()
    class_counts = {_py(k): int(v) for k, v in counts.items()}

    missing = {}
    for column in X.columns:
        k = int(X[column].isna().sum())
        if k:
            missing[str(column)] = {"count": k, "pct": round(100.0 * k / n, 1)}
    rows_with_missing = int(X.isna().any(axis=1).sum())

    # Un duplicado es una fila idéntica en atributos Y clase.
    both = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True).rename("__objetivo__")], axis=1)
    duplicates = int(both.duplicated().sum())

    constant = [str(c) for c in X.columns if X[c].nunique(dropna=True) <= 1]
    id_like = []
    for column in X.columns:
        values = X[column].dropna()
        if n >= 10 and values.nunique() == n:
            integer_valued = pd.api.types.is_numeric_dtype(values) and bool((values % 1 == 0).all())
            if feature_types.get(column) == "nominal" or integer_valued:
                id_like.append(str(column))

    numeric_summary = {}
    for column in X.columns:
        if feature_types.get(column) == "numeric":
            values = pd.to_numeric(X[column], errors="coerce").dropna()
            if len(values):
                numeric_summary[str(column)] = {"min": round(float(values.min()), 3),
                                                "max": round(float(values.max()), 3),
                                                "mean": round(float(values.mean()), 3)}
    cardinality = {str(c): int(X[c].nunique(dropna=True)) for c in X.columns if feature_types.get(c) == "nominal"}

    warnings = []
    for label, count in class_counts.items():
        if count < small_class:
            warnings.append(f"La clase «{label}» tiene solo {count} casos. Por eso, la validación cruzada no puede usar más de "
                            f"{count} pliegues y las métricas de esa clase van a variar mucho. Conviene agrupar clases o usar menos pliegues.")
    if len(class_counts) > 1:
        ratio = max(class_counts.values()) / max(min(class_counts.values()), 1)
        if ratio > imbalance_ratio:
            warnings.append(f"Las clases están desbalanceadas: la más frecuente tiene {ratio:.1f} veces más casos que la menos frecuente. "
                            "En estos casos conviene mirar el F1 macro y la exactitud balanceada, y no solo la exactitud.")
    for column, info in missing.items():
        if info["count"] / n > high_missing:
            warnings.append(f"La columna «{column}» tiene un {info['pct']} % de valores faltantes.")
    if duplicates:
        rows = "Hay 1 fila duplicada" if duplicates == 1 else f"Hay {duplicates} filas duplicadas"
        warnings.append(f"{rows}. Si una copia queda en el entrenamiento y otra en la prueba, las métricas van a "
                        "dar mejor de lo que realmente son.")
    if constant:
        warnings.append(f"Estas columnas tienen siempre el mismo valor, así que no aportan información: {', '.join(map(str, constant))}.")
    if id_like:
        warnings.append(f"Estas columnas parecen identificadores, porque tienen un valor distinto en cada fila: {', '.join(map(str, id_like))}.")

    return {
        "n_rows": n, "n_features": X.shape[1], "class_counts": class_counts,
        "missing": missing, "rows_with_missing": rows_with_missing, "duplicates": duplicates,
        "constant_columns": constant, "id_like_columns": id_like,
        "numeric_summary": numeric_summary, "cardinality": cardinality, "warnings": warnings,
        "outliers": _tukey_outliers(X, feature_types, outlier_factor), "outlier_factor": outlier_factor,
    }


In [ ]:
%%writefile src/tp_ia/data/decisions.py
"""
Registro de decisiones de datos.

GUÍA DE LECTURA
---------------
La cátedra pide que las decisiones técnicas queden anotadas en el informe. En vez de reconstruirlas
de memoria, cada paso del cargador las va registrando solo: qué se decidió, por qué, qué alternativas
había y con qué evidencia (conteos reales). Se puede mostrar en la interfaz o pegar en el informe.

Numeración (la misma del plan): D-01 fuente, D-02 objetivo, D-03 faltantes, D-04 duplicados,
D-05 tipos de atributo, D-06 valores extremos, D-07 evaluación.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from datetime import datetime


@dataclass
class Decision:
    id: str
    title: str
    chosen: str
    rationale: str = ""
    alternatives: list = field(default_factory=list)
    evidence: dict = field(default_factory=dict)
    when: str = field(default_factory=lambda: datetime.now().isoformat(timespec="seconds"))


class DecisionLog:
    def __init__(self):
        self._items: list[Decision] = []

    def add(self, id, title, chosen, rationale="", alternatives=None, evidence=None) -> Decision:
        decision = Decision(id, title, chosen, rationale, list(alternatives or []), dict(evidence or {}))
        self._items.append(decision)
        return decision

    def replace(self, id, title, chosen, rationale="", alternatives=None, evidence=None) -> Decision:
        """Como add(), pero descarta antes lo registrado con ese id (para decisiones que se re-registran)."""
        self._items = [d for d in self._items if d.id != id]
        return self.add(id, title, chosen, rationale, alternatives, evidence)

    def __len__(self):
        return len(self._items)

    def __iter__(self):
        return iter(self._items)

    def ids(self) -> list:
        return [d.id for d in self._items]

    def _ordered(self) -> list:
        """Por número de decisión (D-01, D-02, ...); a igual id se conserva el orden en que se registraron."""
        return sorted(self._items, key=lambda d: d.id)

    def to_records(self) -> list:
        return [asdict(d) for d in self._ordered()]

    def to_markdown(self) -> str:
        """El registro como texto para el informe: cada decisión numerada en palabras, con su motivo y sus datos."""
        lines = ["# Registro de decisiones sobre los datos", "",
                 "Mientras se cargan y se evalúan los datos, el programa anota cada decisión que se tomó, junto con el "
                 "motivo y los números que la respaldan.", ""]
        for number, d in enumerate(self._ordered(), start=1):
            lines += [f"## {decision_label(number)}: {d.chosen}", f"*Tema: {d.title} (código {d.id})*", ""]
            if d.rationale:
                lines += [f"**Por qué:** {d.rationale}", ""]
            if d.alternatives:
                lines += [f"**Otras opciones posibles:** {join_or(d.alternatives)}.", ""]
            if d.evidence:
                lines.append("**En qué se basa:**")
                lines += [f"- {k[:1].upper()}{k[1:]}: {_friendly(v)}" for k, v in d.evidence.items()]
                lines.append("")
            lines += [f"<sub>Se registró el {_friendly_date(d.when)}.</sub>", ""]
        return "\n".join(lines)


ORDINALS = ["Primera", "Segunda", "Tercera", "Cuarta", "Quinta", "Sexta", "Séptima", "Octava", "Novena", "Décima",
            "Undécima", "Duodécima"]


def decision_label(number: int) -> str:
    """1 -> "Primera decisión", 2 -> "Segunda decisión", ...; desde la 13, "Decisión n.º 13"."""
    return f"{ORDINALS[number - 1]} decisión" if 1 <= number <= len(ORDINALS) else f"Decisión n.º {number}"


def join_or(items: list) -> str:
    """["a", "b", "c"] -> "a, b o c" (como se diría en voz alta)."""
    items = [str(i) for i in items]
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " o " + items[-1]


def _friendly(value) -> str:
    """Muestra listas y diccionarios como texto corrido, sin corchetes ni llaves."""
    if isinstance(value, dict):
        parts = [f"{k}: {_friendly(v)}" for k, v in value.items()]
        # si algún valor ya tiene comas, se separa con punto y coma para que no se mezclen
        return ("; " if any("," in p for p in parts) else ", ").join(parts) if parts else "ninguno"
    if isinstance(value, (list, tuple, set)):
        return ", ".join(map(str, value)) if value else "ninguno"
    return str(value)


def _friendly_date(iso: str) -> str:
    """'2026-11-11T02:08:13' -> '11/11/2026 a las 02:08'."""
    try:
        when = datetime.fromisoformat(iso)
    except ValueError:
        return iso
    return when.strftime("%d/%m/%Y a las %H:%M")


In [ ]:
%%writefile src/tp_ia/data/fixtures.py
"""
Datasets de juguete con resultado conocido. Sirven para TESTS y DEMOS: los números se pueden
verificar a mano contra el libro. No son datos del trabajo práctico.
"""
from __future__ import annotations

import pandas as pd


def play_tennis():
    """Play Tennis (Quinlan 1986; Mitchell 1997, cap. 3): 14 casos, 4 atributos nominales, 2 clases.

    Ganancias esperadas en la raíz (en bits): Outlook 0.246, Humidity 0.151, Wind 0.048,
    Temperature 0.029. Árbol esperado: Outlook en la raíz; Overcast -> Yes; Sunny -> Humidity;
    Rain -> Wind.
    """
    rows = [
        ("Sunny", "Hot", "High", "Weak", "No"),
        ("Sunny", "Hot", "High", "Strong", "No"),
        ("Overcast", "Hot", "High", "Weak", "Yes"),
        ("Rain", "Mild", "High", "Weak", "Yes"),
        ("Rain", "Cool", "Normal", "Weak", "Yes"),
        ("Rain", "Cool", "Normal", "Strong", "No"),
        ("Overcast", "Cool", "Normal", "Strong", "Yes"),
        ("Sunny", "Mild", "High", "Weak", "No"),
        ("Sunny", "Cool", "Normal", "Weak", "Yes"),
        ("Rain", "Mild", "Normal", "Weak", "Yes"),
        ("Sunny", "Mild", "Normal", "Strong", "Yes"),
        ("Overcast", "Mild", "High", "Strong", "Yes"),
        ("Overcast", "Hot", "Normal", "Weak", "Yes"),
        ("Rain", "Mild", "High", "Strong", "No"),
    ]
    df = pd.DataFrame(rows, columns=["Outlook", "Temperature", "Humidity", "Wind", "PlayTennis"])
    return df.drop(columns="PlayTennis"), df["PlayTennis"]


def liu_loans():
    """Préstamos (Liu, Web Data Mining, tabla 3.1): 15 casos, 4 atributos nominales, 2 clases.

    Ganancias esperadas en la raíz (en bits): Age 0.083, Has_job 0.324, Own_house 0.420,
    Credit_rating 0.363. Árbol esperado: Own_house en la raíz; true -> Yes; false -> Has_job
    (true -> Yes, false -> No).
    """
    rows = [
        ("young", "false", "false", "fair", "No"),
        ("young", "false", "false", "good", "No"),
        ("young", "true", "false", "good", "Yes"),
        ("young", "true", "true", "fair", "Yes"),
        ("young", "false", "false", "fair", "No"),
        ("middle", "false", "false", "fair", "No"),
        ("middle", "false", "false", "good", "No"),
        ("middle", "true", "true", "good", "Yes"),
        ("middle", "false", "true", "excellent", "Yes"),
        ("middle", "false", "true", "excellent", "Yes"),
        ("old", "false", "true", "excellent", "Yes"),
        ("old", "false", "true", "good", "Yes"),
        ("old", "true", "false", "good", "Yes"),
        ("old", "true", "false", "excellent", "Yes"),
        ("old", "false", "false", "fair", "No"),
    ]
    df = pd.DataFrame(rows, columns=["Age", "Has_job", "Own_house", "Credit_rating", "Class"])
    return df.drop(columns="Class"), df["Class"]


In [ ]:
%%writefile src/tp_ia/data/iris.py
"""
Iris (Fisher, 1936): dataset de DESARROLLO y validación del pipeline.

GUÍA DE LECTURA
---------------
150 flores, 4 atributos numéricos, 3 especies (50 de cada una), sin faltantes y con 1 fila duplicada.
Sirve para probar todo el camino (cargar -> particionar -> entrenar DT y RF -> métricas) antes de usar
el dataset real. Se guarda como CSV en data/iris/ y se lee con el mismo cargador que cualquier otro archivo.
"""
from __future__ import annotations

from pathlib import Path

from .loader import Dataset, load_dataset
from .schema import DatasetSpec

CITATION = ("Fisher, R. A. (1936). The use of multiple measurements in taxonomic problems. "
            "Annals of Eugenics, 7(2), 179-188.")

IRIS_SPEC = DatasetSpec(
    name="Iris", target="species",
    source="la copia que viene incluida en scikit-learn (datasets.load_iris), guardada como CSV en la carpeta data/iris/",
    citation=CITATION,
    purpose="Es un conjunto de datos clásico y pequeño, con tres clases bien diferenciadas. Por eso es ideal para "
            "desarrollar el algoritmo y comparar el árbol con el bosque sin que los tiempos de cálculo sean un problema.")


def default_data_dir() -> Path:
    return Path(__file__).resolve().parents[3] / "data" / "iris"


def load_iris_dataset(data_dir=None, **kwargs) -> Dataset:
    path = (Path(data_dir) if data_dir else default_data_dir()) / "iris.csv"
    return load_dataset(path, IRIS_SPEC, **kwargs)


In [ ]:
%%writefile src/tp_ia/data/heart.py
"""
Enfermedad cardíaca (UCI Heart Disease, Cleveland).

GUÍA DE LECTURA
---------------
De los archivos de data/uci_heart/ solo se usa processed.cleveland.data: según el aviso de los autores (WARNING),
el archivo crudo cleveland.data quedó dañado y los de Hungría, Suiza y Long Beach tienen muchísimos faltantes
(ver la auditoría en DATOS.md). El archivo no trae encabezado: 14 columnas separadas por coma y «?» para los faltantes.

Para que el árbol se pueda leer sin un diccionario al lado:
  - las columnas se renombran en español (el nombre original de UCI queda en la descripción de cada atributo);
  - los códigos de los atributos nominales se traducen (por ejemplo, tipo_dolor = 4 pasa a «sin síntomas»).
El significado de cada código sale de heart-disease.names, la documentación oficial del dataset.
"""
from __future__ import annotations

from pathlib import Path

from .loader import Dataset, load_dataset
from .schema import DatasetSpec, Scenario

CITATION = ("Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). Heart Disease [Dataset]. "
            "UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X")

# (nombre en español, nombre original en UCI, descripción para la interfaz)
ATTRIBUTES = [
    ("edad", "age", "Edad del paciente, en años."),
    ("sexo", "sex", "Sexo del paciente."),
    ("tipo_dolor", "cp", "Tipo de dolor de pecho que presenta el paciente."),
    ("presion_reposo", "trestbps", "Presión arterial en reposo, en mm Hg, medida al ingresar al hospital."),
    ("colesterol", "chol", "Colesterol en sangre, en mg/dl."),
    ("glucemia_ayunas", "fbs", "Indica si el azúcar en sangre en ayunas supera los 120 mg/dl."),
    ("ecg_reposo", "restecg", "Resultado del electrocardiograma en reposo."),
    ("frecuencia_max", "thalach", "Frecuencia cardíaca máxima alcanzada durante la prueba de esfuerzo, en latidos por minuto."),
    ("angina_ejercicio", "exang", "Indica si el ejercicio le provoca angina (dolor de pecho) al paciente."),
    ("depresion_st", "oldpeak", "Cuánto baja el segmento ST del electrocardiograma con el ejercicio, comparado con el reposo."),
    ("pendiente_st", "slope", "Forma que toma el segmento ST en el momento de máximo esfuerzo."),
    ("vasos_coloreados", "ca", "Cantidad de vasos sanguíneos principales (de 0 a 3) que se ven en la fluoroscopía."),
    ("talio", "thal", "Resultado de la prueba de esfuerzo con talio, que muestra cómo llega la sangre al corazón."),
]
TARGET = "diagnostico"
COLUMNS = [name for name, _, _ in ATTRIBUTES] + [TARGET]
DESCRIPTIONS = {name: f"{text} En UCI se llama «{original}»." for name, original, text in ATTRIBUTES}

# Qué significa cada código (según heart-disease.names). vasos_coloreados es un conteo, así que queda numérico.
VALUE_LABELS = {
    "sexo": {0: "mujer", 1: "hombre"},
    "tipo_dolor": {1: "angina típica", 2: "angina atípica", 3: "dolor no anginoso", 4: "sin síntomas"},
    "glucemia_ayunas": {0: "120 mg/dl o menos", 1: "más de 120 mg/dl"},
    "ecg_reposo": {0: "normal", 1: "anomalía de la onda ST-T", 2: "hipertrofia ventricular"},
    "angina_ejercicio": {0: "no", 1: "sí"},
    "pendiente_st": {1: "ascendente", 2: "plana", 3: "descendente"},
    "talio": {3: "normal", 6: "defecto fijo", 7: "defecto reversible"},
}

SCENARIOS = {
    "binario": Scenario(
        "binario", mapping={0: 0, 1: 1, 2: 1, 3: 1, 4: 1},
        class_names={0: "sin enfermedad", 1: "con enfermedad"},
        goal="predecir si el paciente tiene o no una enfermedad cardíaca",
        description="La columna original va de 0 a 4: el 0 indica que no hay enfermedad y los valores de 1 a 4 indican "
                    "que sí la hay. Según la documentación del dataset, los experimentos con estos datos se concentraron "
                    "en distinguir la presencia de la enfermedad (valores 1 a 4) de su ausencia (valor 0), así que se "
                    "agrupan los valores de 1 a 4 en una sola clase. Además, así las dos clases quedan parejas (164 y "
                    "139 casos), mientras que con las cinco clases originales la más chica tendría solo 13 casos."),
    "multiclase": Scenario(
        "multiclase", mapping=None,
        class_names={0: "sin enfermedad", 1: "valor 1", 2: "valor 2", 3: "valor 3", 4: "valor 4"},
        goal="predecir el valor original del diagnóstico, de 0 a 4",
        description="Se predice la columna original, con sus cinco valores: el 0 indica que no hay enfermedad y los "
                    "valores de 1 a 4, que sí la hay. La documentación del dataset no explica qué distingue a cada uno "
                    "de esos cuatro valores."),
}

HEART_SPEC = DatasetSpec(
    name="Enfermedad cardíaca", target=TARGET, columns=COLUMNS, header=False,
    nominal=list(VALUE_LABELS), value_labels=VALUE_LABELS, descriptions=DESCRIPTIONS, scenarios=SCENARIOS,
    source="el archivo processed.cleveland.data del UCI Machine Learning Repository (Heart Disease), que reúne los "
           "pacientes de la Cleveland Clinic Foundation. Es el único archivo de la colección que los autores "
           "señalan como confiable y que está prácticamente completo",
    citation=CITATION,
    purpose="Es un problema real de diagnóstico médico que mezcla atributos numéricos (como la edad o el colesterol) "
            "con atributos nominales (como el tipo de dolor de pecho), e incluye algunos valores faltantes. Por eso "
            "permite probar todas las capacidades del árbol C4.5 en un caso realista.")


def default_data_dir() -> Path:
    return Path(__file__).resolve().parents[3] / "data" / "uci_heart"


def load_heart_dataset(data_dir=None, scenario: str = "binario", **kwargs) -> Dataset:
    """Carga processed.cleveland.data como Dataset. Por defecto, con el objetivo binario (con o sin enfermedad)."""
    path = (Path(data_dir) if data_dir else default_data_dir()) / "processed.cleveland.data"
    return load_dataset(path, HEART_SPEC, scenario=scenario, **kwargs)


### 1.3 La comparación (`tp_ia.eval`)

Entrenan el Random Forest y comparan los dos modelos sobre las mismas particiones.

- `__init__.py`: lo que se puede importar de la evaluación.
- `models.py`: el Random Forest de scikit-learn.
- `splits.py`: las particiones de entrenamiento y prueba (validación cruzada estratificada repetida).
- `metrics.py`: las métricas: exactitud, exactitud balanceada, F1 macro, AUC y matriz de confusión.
- `runner.py`: corre los experimentos con las mismas particiones para los dos modelos.
- `experiments.py`: el barrido de un parámetro, la validación cruzada anidada y la robustez a datos faltantes.

In [ ]:
%%writefile src/tp_ia/eval/__init__.py
"""Evaluación: particiones, métricas, modelos comparables y corredor de experimentos."""
from .experiments import (inject_missing, missing_robustness, nested_cv, parameter_sweep, robustness_table,
                          selection_frequency)
from .metrics import confusion, evaluate
from .models import RandomForestModel, make_model
from .runner import (environment_info, paired_comparison, register_evaluation, run_experiment, save_results,
                     summarize)
from .splits import Split, make_splits, max_stratified_splits

__all__ = ["Split", "RandomForestModel", "confusion", "environment_info", "evaluate", "inject_missing",
           "make_model", "make_splits", "max_stratified_splits", "missing_robustness", "nested_cv",
           "paired_comparison", "parameter_sweep", "register_evaluation", "robustness_table", "run_experiment",
           "save_results", "selection_frequency", "summarize"]


In [ ]:
%%writefile src/tp_ia/eval/models.py
"""
Modelos con una interfaz común para compararlos: fit(X, y, feature_types) / predict / predict_proba.

GUÍA DE LECTURA
---------------
  - El árbol propio (C45DecisionTree) ya tiene esa interfaz.
  - RandomForestModel envuelve el RandomForestClassifier de scikit-learn (que el enunciado permite) con
    el preprocesamiento que necesita, ajustado SOLO con el train de cada pliegue (sin filtrar el test):
        numéricos -> imputación por mediana
        nominales -> imputación por la moda + one-hot (los valores nuevos en test se ignoran)
    Así el RF no se beneficia de información del test y el DT y el RF ven los mismos datos.
  - Diferencia a documentar en el informe: el RF de sklearn usa árboles binarios tipo CART; nuestro DT
    usa divisiones multirrama C4.5.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..c45 import C45DecisionTree

_RF_PARAMS = ("n_estimators", "max_depth", "max_features", "min_samples_split", "min_samples_leaf",
              "criterion", "class_weight", "random_state")


class RandomForestModel:
    def __init__(self, n_estimators=100, max_depth=None, max_features="sqrt", min_samples_split=2,
                 min_samples_leaf=1, criterion="gini", class_weight=None, random_state=0):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.class_weight = class_weight
        self.random_state = random_state

    def get_params(self, deep=True):
        return {name: getattr(self, name) for name in _RF_PARAMS}

    def set_params(self, **params):
        for name, value in params.items():
            if name not in _RF_PARAMS:
                raise ValueError(f"Parámetro desconocido: {name!r}. Válidos: {_RF_PARAMS}")
            setattr(self, name, value)
        return self

    def _prepare(self, X):
        frame = X[self.feature_names_].copy().reset_index(drop=True)
        for column in self.numeric_:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        for column in self.nominal_:
            frame[column] = frame[column].astype(object).where(frame[column].notna(), np.nan)
        return frame

    def fit(self, X, y, feature_types=None, sample_weight=None):
        from sklearn.compose import ColumnTransformer
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.impute import SimpleImputer
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import OneHotEncoder

        types = feature_types or {}
        self.feature_names_ = [str(c) for c in X.columns]
        X = X.copy()
        X.columns = self.feature_names_
        self.nominal_ = [c for c in self.feature_names_ if types.get(c) == "nominal" or
                         (c not in types and not pd.api.types.is_numeric_dtype(X[c]))]
        self.numeric_ = [c for c in self.feature_names_ if c not in self.nominal_]
        preprocess = ColumnTransformer([
            ("num", SimpleImputer(strategy="median", keep_empty_features=True), self.numeric_),
            ("nom", Pipeline([("imp", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
                              ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), self.nominal_),
        ])
        forest = RandomForestClassifier(**self.get_params())
        self.pipeline_ = Pipeline([("pre", preprocess), ("rf", forest)])
        fit_params = {} if sample_weight is None else {"rf__sample_weight": np.asarray(sample_weight)}
        self.pipeline_.fit(self._prepare(X), np.asarray(y), **fit_params)
        self.classes_ = self.pipeline_.classes_
        self.feature_importances_ = self._importances_by_feature()
        return self

    def _importances_by_feature(self) -> dict:
        """Importancia por atributo ORIGINAL: se suman las columnas one-hot de cada nominal."""
        raw = self.pipeline_.named_steps["rf"].feature_importances_
        result = dict(zip(self.numeric_, raw[:len(self.numeric_)]))
        position = len(self.numeric_)
        if self.nominal_:
            encoder = self.pipeline_.named_steps["pre"].named_transformers_["nom"].named_steps["oh"]
            for column, categories in zip(self.nominal_, encoder.categories_):
                result[column] = raw[position:position + len(categories)].sum()
                position += len(categories)
        return {c: float(result[c]) for c in self.feature_names_}

    def predict_proba(self, X):
        return self.pipeline_.predict_proba(self._prepare(X))

    def predict(self, X):
        return self.pipeline_.predict(self._prepare(X))


def make_model(kind: str, **params):
    """'dt' = árbol C4.5 propio; 'rf' = Random Forest de scikit-learn."""
    if kind == "dt":
        return C45DecisionTree(**params)
    if kind == "rf":
        return RandomForestModel(**params)
    raise ValueError(f"Modelo desconocido {kind!r}: use 'dt' o 'rf'.")


In [ ]:
%%writefile src/tp_ia/eval/splits.py
"""
Particiones train/test reproducibles y estratificadas.

GUÍA DE LECTURA
---------------
  - Con pocos datos, UNA sola partición es ruido: un error de más mueve la accuracy varios puntos.
    Por eso el esquema por defecto es "repeated_kfold": k pliegues, repetidos n_repeats veces con
    semillas distintas (seed + repetición), y se reporta media ± desvío.
  - ESTRATIFICADO: cada pliegue conserva la proporción de clases. Requiere k <= casos de la clase
    más chica (si no, se avisa con un error claro en vez de fallar adentro de sklearn).
  - groups: si hay filas repetidas (o del mismo paciente), pasarlas con el mismo grupo evita que una
    caiga en train y su copia en test, que infla las métricas.
  - Las MISMAS particiones se usan para DT y RF: la comparación es sobre los mismos datos.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, StratifiedShuffleSplit

SCHEMES = ("repeated_kfold", "holdout")


@dataclass(frozen=True)
class Split:
    id: int
    repeat: int
    fold: int
    train: np.ndarray
    test: np.ndarray


def max_stratified_splits(y) -> int:
    """Cantidad máxima de pliegues estratificados que admite y (casos de la clase más chica)."""
    return int(pd.Series(np.asarray(y)).value_counts().min())


def make_splits(y, scheme="repeated_kfold", n_splits=5, n_repeats=3, test_size=0.3, seed=0, groups=None):
    y = np.asarray(y)
    if scheme not in SCHEMES:
        raise ValueError(f"scheme debe ser uno de {SCHEMES}, no {scheme!r}")
    placeholder = np.zeros(len(y))
    splits: list = []
    if scheme == "repeated_kfold":
        smallest = max_stratified_splits(y)
        if n_splits < 2:
            raise ValueError("n_splits debe ser >= 2")
        if n_splits > smallest:
            raise ValueError(f"n_splits={n_splits} supera los casos de la clase más chica ({smallest}). "
                             f"Use n_splits <= {smallest} o agrupe clases.")
        for repeat in range(n_repeats):
            state = seed + repeat
            if groups is None:
                folds = StratifiedKFold(n_splits, shuffle=True, random_state=state).split(placeholder, y)
            else:
                folds = StratifiedGroupKFold(n_splits, shuffle=True, random_state=state).split(placeholder, y, groups)
            for fold, (train, test) in enumerate(folds):
                splits.append(Split(len(splits), repeat, fold, train, test))
    else:
        if groups is not None:
            raise ValueError("holdout no admite groups; use repeated_kfold con groups.")
        shuffle = StratifiedShuffleSplit(n_splits=n_repeats, test_size=test_size, random_state=seed)
        for repeat, (train, test) in enumerate(shuffle.split(placeholder, y)):
            splits.append(Split(repeat, repeat, 0, train, test))
    return splits


In [ ]:
%%writefile src/tp_ia/eval/metrics.py
"""
Métricas de clasificación (binaria y multiclase) con las precauciones que importan con pocos datos.

GUÍA DE LECTURA
---------------
  - accuracy sola engaña con clases desbalanceadas: se reportan también balanced_accuracy, macro-F1,
    kappa y MCC. Las métricas "macro" promedian las clases por igual (una clase rara pesa lo mismo).
  - Por clase: precision, recall, F1 y soporte. En binario: sensibilidad (recall de la clase 1) y
    especificidad (recall de la clase 0).
  - ROC-AUC uno-contra-resto (OvR) promediado, calculado SOLO sobre las clases que tienen positivos y
    negativos en el test (en un pliegue chico puede faltar una clase; en ese caso se la omite en vez de fallar).
  - Los pesos de las métricas "macro" se calculan sobre las clases presentes en y_true o y_pred.
"""
from __future__ import annotations

import warnings

import numpy as np
from sklearn import metrics as skm


def confusion(y_true, y_pred, classes):
    """Matriz de confusión con filas = clase real y columnas = predicha, en el orden de `classes`."""
    return skm.confusion_matrix(y_true, y_pred, labels=list(classes))


def ovr_auc(y_true, proba, classes) -> float:
    y_true = np.asarray(y_true)
    aucs = []
    for k, c in enumerate(classes):
        positive = y_true == c
        if 0 < positive.sum() < len(y_true):
            aucs.append(skm.roc_auc_score(positive, proba[:, k]))
    return float(np.mean(aucs)) if aucs else float("nan")


def evaluate(y_true, y_pred, proba=None, classes=None) -> dict:
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    classes = list(classes) if classes is not None else sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    seen = set(y_true.tolist()) | set(y_pred.tolist())
    present = [c for c in classes if c in seen]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        out = {
            "accuracy": skm.accuracy_score(y_true, y_pred),
            "balanced_accuracy": skm.balanced_accuracy_score(y_true, y_pred),
            "macro_f1": skm.f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0),
            "weighted_f1": skm.f1_score(y_true, y_pred, labels=present, average="weighted", zero_division=0),
            "macro_precision": skm.precision_score(y_true, y_pred, labels=present, average="macro", zero_division=0),
            "macro_recall": skm.recall_score(y_true, y_pred, labels=present, average="macro", zero_division=0),
            "kappa": skm.cohen_kappa_score(y_true, y_pred, labels=present),
            "mcc": skm.matthews_corrcoef(y_true, y_pred),
        }
        precision, recall, f1, support = skm.precision_recall_fscore_support(
            y_true, y_pred, labels=classes, zero_division=0)
    for i, c in enumerate(classes):
        out[f"precision_{c}"], out[f"recall_{c}"] = precision[i], recall[i]
        out[f"f1_{c}"], out[f"support_{c}"] = f1[i], support[i]
    if len(classes) == 2:
        out["sensitivity"], out["specificity"] = recall[1], recall[0]
    if proba is not None:
        out["roc_auc"] = ovr_auc(y_true, np.asarray(proba), classes)
    return {k: float(v) for k, v in out.items()}


In [ ]:
%%writefile src/tp_ia/eval/runner.py
"""
Corredor de experimentos: entrena cada modelo en cada partición y devuelve UNA tabla.

GUÍA DE LECTURA
---------------
  - Entrada: un Dataset, un diccionario {nombre: (tipo, parámetros)} y las particiones (splits.py).
  - Salida: un DataFrame con una fila por (partición, modelo): parámetros, tiempos de fit y de predict,
    métricas en TEST y en TRAIN (la diferencia entre ambas es la medida de sobreajuste) y la matriz de
    confusión. Se guarda como CSV con la versión de las librerías: con eso el experimento se reproduce.
  - summarize(): media y desvío por modelo. paired_comparison(): diferencia entre dos modelos sobre las
    MISMAS particiones, con la prueba de Wilcoxon. Las particiones de una validación cruzada se solapan,
    así que ese p-valor es DESCRIPTIVO (optimista), no una prueba formal.
  - Los tiempos del DT (Python puro) y del RF (código compilado) no son comparables en valor absoluto.
"""
from __future__ import annotations

import json
import platform
import time

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from .metrics import confusion, evaluate
from .models import make_model


def environment_info() -> dict:
    import scipy
    import sklearn
    return {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "sklearn": sklearn.__version__}


def _align_proba(proba, model_classes, classes) -> np.ndarray:
    """Lleva las probabilidades al conjunto GLOBAL de clases (un pliegue de train puede no tener una clase rara)."""
    out = np.zeros((len(proba), len(classes)))
    index = {c: i for i, c in enumerate(classes)}
    for j, c in enumerate(model_classes):
        out[:, index[c]] = proba[:, j]
    return out


PARAM_NAMES = {"criterion": "criterio", "max_depth": "profundidad máxima", "min_samples_leaf": "mínimo de casos por hoja",
               "min_samples_split": "mínimo de casos para dividir", "min_gain": "ganancia mínima",
               "use_avg_gain": "heurística de ganancia promedio", "threshold_mode": "umbral numérico",
               "numeric_gain_correction": "corrección de la versión 8", "prune": "poda", "confidence": "confianza de la poda",
               "n_estimators": "cantidad de árboles", "random_state": "semilla"}
PARAM_VALUES = {None: "sin límite", True: "sí", False: "no", "gain_ratio": "razón de ganancia",
                "gain": "ganancia de información", "c45": "valor observado", "midpoint": "punto medio"}


def _param_text(key, value) -> str:
    """('max_depth', None) -> 'profundidad máxima: sin límite', como se lee en la interfaz."""
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    shown = PARAM_VALUES.get(value, value) if isinstance(value, (str, bool, type(None))) else value
    return f"{PARAM_NAMES.get(key, key)}: {shown}"


def register_evaluation(dataset, splits, models: dict):
    """Anota en el registro de decisiones del dataset cómo se evalúa (D-07).

    Se puede llamar las veces que haga falta: reemplaza la anotación anterior en vez de duplicarla.
    """
    if not splits:
        return None
    repeats = len({s.repeat for s in splits})
    folds = len(splits) // repeats
    train, test = [len(s.train) for s in splits], [len(s.test) for s in splits]
    if folds > 1:
        chosen = f"evaluar con validación cruzada estratificada de {folds} pliegues, repetida {repeats} veces ({len(splits)} particiones en total)"
    else:
        chosen = f"evaluar con una división estratificada en entrenamiento y prueba, repetida {repeats} veces ({len(splits)} particiones en total)"
    span = lambda sizes: f"{min(sizes)}" if min(sizes) == max(sizes) else f"entre {min(sizes)} y {max(sizes)}"
    names = {"dt": "árbol propio C4.5", "rf": "Random Forest de scikit-learn"}
    describe = lambda kind, params: (f"{names.get(kind, kind)}, con " + ", ".join(_param_text(k, v) for k, v in params.items())
                                     if params else f"{names.get(kind, kind)}, con los parámetros por defecto")
    return dataset.log.replace(
        "D-07", "Esquema de evaluación", chosen,
        rationale="Con pocos datos, una sola partición da resultados poco confiables: un error más o menos cambia la "
                  "exactitud en varios puntos. Por eso se repite la validación cruzada. Además, se estratifica para "
                  "mantener la proporción de clases en cada partición, y los dos modelos se evalúan sobre las mismas "
                  "particiones para que la comparación sea justa. Se informan la exactitud, la exactitud balanceada y el "
                  "F1 macro, porque con clases desbalanceadas la exactitud sola puede engañar. También se compara el "
                  "entrenamiento con la prueba para detectar sobreajuste. Como las particiones comparten datos entre sí, "
                  "el valor p de la prueba de Wilcoxon es solo orientativo.",
        alternatives=["usar una sola división en entrenamiento y prueba", "usar validación cruzada sin repeticiones"],
        evidence={"particiones": len(splits), "repeticiones": repeats, "pliegues por repetición": folds,
                  "casos de entrenamiento por partición": span(train), "casos de prueba por partición": span(test),
                  "modelos": {name: describe(kind, params) for name, (kind, params) in models.items()}})


def run_experiment(dataset, models: dict, splits, register_decision: bool = True) -> pd.DataFrame:
    """Entrena cada modelo en cada partición. Con register_decision anota D-07 en dataset.log."""
    if register_decision:
        register_evaluation(dataset, splits, models)
    X, y = dataset.X, dataset.y.to_numpy()
    classes = list(dataset.classes)
    labels = np.asarray(classes)
    rows = []
    for split in splits:
        X_train, X_test = X.iloc[split.train], X.iloc[split.test]
        y_train, y_test = y[split.train], y[split.test]
        for name, (kind, params) in models.items():
            model = make_model(kind, **params)
            start = time.perf_counter()
            model.fit(X_train, y_train, feature_types=dataset.feature_types)
            fit_time = time.perf_counter() - start
            start = time.perf_counter()
            proba_test = _align_proba(model.predict_proba(X_test), model.classes_, classes)
            predict_time = time.perf_counter() - start
            proba_train = _align_proba(model.predict_proba(X_train), model.classes_, classes)
            pred_test, pred_train = labels[proba_test.argmax(axis=1)], labels[proba_train.argmax(axis=1)]
            test = evaluate(y_test, pred_test, proba_test, classes)
            train = evaluate(y_train, pred_train, proba_train, classes)
            row = {"dataset": dataset.name, "scenario": dataset.scenario, "split": split.id,
                   "repeat": split.repeat, "fold": split.fold, "model": name, "kind": kind,
                   "params": json.dumps(params, sort_keys=True, default=str),
                   "n_train": len(split.train), "n_test": len(split.test),
                   "fit_time": fit_time, "predict_time": predict_time,
                   "confusion": json.dumps(confusion(y_test, pred_test, classes).tolist())}
            row.update({f"test_{k}": v for k, v in test.items()})
            row.update({f"train_{k}": train[k] for k in ("accuracy", "balanced_accuracy", "macro_f1")})
            rows.append(row)
    results = pd.DataFrame(rows)
    results.attrs["env"] = environment_info()
    return results


def _column(results, metric):
    return f"test_{metric}" if f"test_{metric}" in results.columns else metric


def summarize(results, metrics=("accuracy", "balanced_accuracy", "macro_f1", "roc_auc", "fit_time")) -> pd.DataFrame:
    """Media y desvío por modelo (en el orden en que se definieron)."""
    order = list(dict.fromkeys(results["model"]))
    grouped = results.groupby("model")
    table = pd.DataFrame(index=order)
    for metric in metrics:
        column = _column(results, metric)
        if column in results.columns:
            table[f"{metric}_mean"] = grouped[column].mean()
            table[f"{metric}_std"] = grouped[column].std(ddof=1)
    return table


def paired_comparison(results, a: str, b: str, metric: str = "accuracy") -> dict:
    """Diferencia a - b sobre las mismas particiones, con Wilcoxon (descriptivo: ver nota del módulo)."""
    column = _column(results, metric)
    a_scores = results[results["model"] == a].set_index("split")[column]
    b_scores = results[results["model"] == b].set_index("split")[column]
    diff = (a_scores - b_scores).dropna()
    p_value = 1.0 if (len(diff) == 0 or np.allclose(diff, 0)) else float(wilcoxon(diff).pvalue)
    return {"metric": metric, "a": a, "b": b, "n": int(len(diff)), "mean_diff": float(diff.mean()),
            "wins_a": int((diff > 0).sum()), "wins_b": int((diff < 0).sum()), "ties": int((diff == 0).sum()),
            "p_wilcoxon": p_value}


def save_results(results, path) -> None:
    results.to_csv(path, index=False)


In [ ]:
%%writefile src/tp_ia/eval/experiments.py
"""
Experimentos: barrido de un parámetro, selección con validación cruzada anidada y robustez a faltantes.

GUÍA DE LECTURA
---------------
  - parameter_sweep(): varía UN parámetro y deja los demás fijos. Devuelve, por valor, el rendimiento en test
    y en train: la brecha train - test mide el sobreajuste (es la "curva de complejidad" del informe).
  - nested_cv(): elegir los parámetros con los mismos datos con que después se mide el resultado lo infla,
    porque el test "participó" de la elección. Por eso, dentro de CADA partición externa se elige la mejor
    combinación de la grilla con una validación cruzada interna que ve SOLO el train externo; recién después
    se evalúa en el test externo. Ante un empate gana la primera combinación de la grilla: poner primero la
    más simple.
  - inject_missing() / missing_robustness(): borra al azar una fracción de los valores (faltantes completamente
    al azar) y mide cuánto cae cada modelo. Es donde más se diferencian los dos: el árbol C4.5 reparte fracciones
    de caso por las ramas, mientras que el Random Forest tiene que imputar con la mediana o la moda.
  - Todo se apoya en run_experiment y make_splits: mismas particiones, mismas métricas, mismo formato de tabla.
"""
from __future__ import annotations

import copy
import itertools
import json
from dataclasses import replace

import numpy as np
import pandas as pd

from .runner import run_experiment
from .splits import make_splits

METRICS = ("accuracy", "balanced_accuracy", "macro_f1")   # las que se calculan en train y en test


def _check_metric(metric: str) -> None:
    if metric not in METRICS:
        raise ValueError(f"metric debe ser una de {METRICS}, no {metric!r}")


# ------------------------------------------------------------------ un parámetro a la vez
def parameter_sweep(dataset, kind: str, param: str, values, splits, base_params=None,
                    metric: str = "macro_f1") -> pd.DataFrame:
    """Rendimiento en test y en train al variar UN parámetro. `gap` = train - test (sobreajuste)."""
    _check_metric(metric)
    values = list(values)
    labels = [str(v) for v in values]
    if not values or len(set(labels)) != len(labels):
        raise ValueError("values no puede estar vacío ni repetir valores")
    base = dict(base_params or {})
    models = {label: (kind, {**base, param: value}) for label, value in zip(labels, values)}
    results = run_experiment(dataset, models, splits, register_decision=False)
    rows = []
    for label, value in zip(labels, values):
        part = results[results["model"] == label]
        test, train = part[f"test_{metric}"], part[f"train_{metric}"]
        rows.append({"param": param, "value": value, "n_partitions": len(part),
                     f"test_{metric}_mean": test.mean(), f"test_{metric}_std": test.std(ddof=1),
                     f"train_{metric}_mean": train.mean(), "gap": train.mean() - test.mean(),
                     "fit_time_mean": part["fit_time"].mean()})
    table = pd.DataFrame(rows)
    table["value"] = pd.Series(values, dtype=object)    # sin esto 1 pasa a 1.0 y None a NaN
    return table


# ------------------------------------------------------------------ validación cruzada anidada
def _combinations(grid: dict) -> list:
    names = list(grid)
    return [dict(zip(names, combo)) for combo in itertools.product(*(list(grid[n]) for n in names))]


def nested_cv(dataset, kind: str, grid: dict, outer_splits, inner_splits: int = 3, inner_repeats: int = 1,
              metric: str = "macro_f1", base_params=None, seed: int = 0) -> pd.DataFrame:
    """Una fila por partición externa: combinación elegida con la CV interna y su resultado en el test externo."""
    _check_metric(metric)
    if not grid or any(len(list(v)) == 0 for v in grid.values()):
        raise ValueError("grid no puede estar vacía ni tener parámetros sin valores")
    base = dict(base_params or {})
    combos = _combinations(grid)
    candidates = {str(i): (kind, {**base, **combo}) for i, combo in enumerate(combos)}
    rows = []
    for outer in outer_splits:
        # La CV interna trabaja con una copia que contiene SOLO el train externo: el test externo no existe para ella.
        train_only = replace(dataset, X=dataset.X.iloc[outer.train].reset_index(drop=True),
                             y=dataset.y.iloc[outer.train].reset_index(drop=True))
        inner = make_splits(train_only.y, n_splits=inner_splits, n_repeats=inner_repeats, seed=seed + outer.id)
        inner_results = run_experiment(train_only, candidates, inner, register_decision=False)
        scores = inner_results.groupby("model")[f"test_{metric}"].mean().reindex(list(candidates))
        best = int(np.argmax(scores.to_numpy()))        # ante un empate, la primera de la grilla
        final = run_experiment(dataset, {"elegido": (kind, {**base, **combos[best]})}, [outer],
                               register_decision=False).iloc[0]
        row = {"outer_split": outer.id, "repeat": outer.repeat, "fold": outer.fold,
               "chosen": json.dumps(combos[best], sort_keys=True, default=str),
               "inner_score": float(scores.iloc[best])}
        row.update({k: final[k] for k in final.index if k.startswith(("test_", "train_"))})
        rows.append(row)
    return pd.DataFrame(rows)


def selection_frequency(nested: pd.DataFrame) -> pd.Series:
    """Cuántas veces eligió cada combinación la CV interna: si cambia mucho, la elección es inestable."""
    return nested["chosen"].value_counts()


# ------------------------------------------------------------------ robustez a faltantes
def inject_missing(dataset, fraction: float, seed: int = 0, columns=None):
    """Copia del dataset con una fracción de los valores de los atributos borrada al azar. No toca la clase."""
    if not 0 <= fraction < 1:
        raise ValueError("fraction debe estar en [0, 1)")
    columns = list(columns) if columns is not None else list(dataset.X.columns)
    unknown = set(columns) - set(dataset.X.columns)
    if unknown:
        raise ValueError(f"Columnas que no existen en los datos: {sorted(map(str, unknown))}")
    mask = np.random.default_rng(seed).random((len(dataset.X), len(columns))) < fraction
    X = dataset.X.copy()
    X[columns] = X[columns].mask(pd.DataFrame(mask, index=X.index, columns=columns))
    return replace(dataset, name=f"{dataset.name} (+{fraction:.0%} faltantes)", X=X, log=copy.deepcopy(dataset.log))


def missing_robustness(dataset, models: dict, fractions, splits, seeds=(0,)) -> pd.DataFrame:
    """run_experiment repetido con fracciones crecientes de faltantes (en train y en test), con las mismas particiones.

    Cada semilla de `seeds` es una máscara de faltantes distinta: promediar varias evita que el resultado dependa
    de qué celdas cayeron justo. Con la misma semilla las máscaras son anidadas (las celdas borradas al 10 % siguen
    borradas al 20 %), así que las fracciones son comparables entre sí.
    """
    frames = []
    for seed in seeds:
        for fraction in fractions:
            results = run_experiment(inject_missing(dataset, fraction, seed=seed), models, splits,
                                     register_decision=False)
            results["missing_fraction"] = fraction
            results["mask_seed"] = seed
            frames.append(results)
    return pd.concat(frames, ignore_index=True)


def robustness_table(results: pd.DataFrame, metric: str = "macro_f1") -> pd.DataFrame:
    """Media de la métrica en test: una fila por fracción de faltantes y una columna por modelo."""
    table = results.pivot_table(index="missing_fraction", columns="model", values=f"test_{metric}", aggfunc="mean")
    return table[list(dict.fromkeys(results["model"]))]


### 1.4 La interfaz (`tp_ia.web`)

Una parte en Python (los cálculos) y otra en JavaScript y CSS (lo que se ve).

- `__init__.py`: lo que se puede importar de la interfaz.
- `service.py`: toda la lógica de la interfaz, sin necesidad de un navegador.
- `widget.py`: el widget de anywidget que se muestra en la celda.
- `app.js`: la interfaz que se dibuja en el navegador (JavaScript).
- `app.css`: los estilos de la interfaz.
- `devserver.py`: un servidor local para probar la interfaz fuera de Colab. En este cuaderno no se usa.
- `dev.html`: la página que usa ese servidor local.

In [ ]:
%%writefile src/tp_ia/web/__init__.py
"""Interfaz web del laboratorio: servicio en Python (service.py) y widget de Colab (widget.py, requiere anywidget)."""
from .service import DatasetEntry, LabService, default_entries, to_jsonable

__all__ = ["DatasetEntry", "LabService", "default_entries", "to_jsonable"]


In [ ]:
%%writefile src/tp_ia/web/service.py
"""
Capa de servicio de la interfaz web: toda la lógica del laboratorio, sin nada de interfaz.

GUÍA DE LECTURA
---------------
La interfaz (JavaScript dentro de un widget de Colab) solo pide cosas y dibuja lo que vuelve. Esta clase hace el
trabajo: elige el dataset, entrena el árbol propio (C4.5) y el Random Forest, compara, barre parámetros y mide la
robustez a faltantes. Cada método devuelve diccionarios con tipos de Python puros (se pasan a JSON sin cambios),
así que se puede probar con pytest sin abrir ningún navegador.

  - LabService.call(método, parámetros) es el único punto de entrada para la interfaz: despacha a un método
    permitido y limpia el resultado para JSON. Un método que no está en la lista no se ejecuta.
  - El árbol de la pestaña "Árbol" se entrena con una partición estratificada train/test (por defecto 30 % de test;
    Play Tennis usa todos los casos, como en el libro). La pestaña "Comparar" usa validación cruzada repetida.
  - Los datasets se registran con register_dataset(): así se puede sumar otro dataset o un CSV subido sin tocar nada más.
    El primero de default_entries() es el primero de la lista desplegable de la interfaz.
"""
from __future__ import annotations

import json
import math
import tempfile
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd

from ..c45 import C45DecisionTree
from ..data import load_heart_dataset, load_iris_dataset
from ..data.fixtures import play_tennis
from ..data.loader import Dataset, build_dataset, read_table
from ..data.schema import DatasetSpec
from ..eval import (RandomForestModel, evaluate, make_splits, max_stratified_splits, missing_robustness,
                    paired_comparison, parameter_sweep, robustness_table, run_experiment)
from ..eval.metrics import confusion
from ..eval.runner import _align_proba

DT_PARAMS = ("criterion", "max_depth", "min_samples_split", "min_samples_leaf", "min_gain", "use_avg_gain",
             "threshold_mode", "numeric_gain_correction", "prune", "confidence")
RF_PARAMS = ("n_estimators", "max_depth", "max_features", "min_samples_split", "min_samples_leaf", "criterion",
             "class_weight", "random_state")
FOREST_PRESET = {"n_estimators": 100, "max_depth": 10, "random_state": 42}      # el RF de ejemplo de la cátedra
SWEEPABLE = {"dt": ("max_depth", "min_samples_leaf", "min_samples_split", "min_gain", "confidence"),
             "rf": ("n_estimators", "max_depth", "min_samples_leaf", "min_samples_split")}
INT_PARAMS = ("max_depth", "min_samples_split", "n_estimators", "random_state")
METHODS = ("datasets", "select", "set_split", "train", "train_forest", "predict", "compare", "sweep",
           "robustness", "decisions", "load_csv")


# ------------------------------------------------------------------ utilidades
def to_jsonable(obj):
    """Convierte (recursivamente) tipos de numpy/pandas a tipos de Python; NaN e infinitos pasan a None."""
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    if isinstance(obj, pd.DataFrame):
        return to_jsonable(obj.to_dict("records"))
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        value = float(obj)
        return value if math.isfinite(value) else None
    if obj is None or obj is pd.NA or obj is pd.NaT:
        return None
    return obj if isinstance(obj, str) else str(obj)


def _tidy(value):
    """Un nominal como 3.0 se muestra como 3; los escalares de numpy pasan a Python."""
    if hasattr(value, "item"):
        value = value.item()
    if isinstance(value, float) and value.is_integer():
        return int(value)
    return value


@dataclass
class DatasetEntry:
    key: str
    title: str
    description: str
    loader: Callable[[], Dataset]
    source: str = ""
    default_test_size: float = 0.3
    disclaimer: str = ""


# ------------------------------------------------------------------ datasets incluidos
def _play_tennis() -> Dataset:
    X, y = play_tennis()
    frame = X.copy()
    frame["Play"] = np.asarray(y)
    spec = DatasetSpec(name="Play Tennis", target="Play", nominal=list(X.columns),
                       source="el ejemplo clásico de Quinlan (1986)",
                       purpose="Es el ejemplo con el que Quinlan presenta los árboles de decisión. Como tiene solo 14 "
                               "casos con atributos nominales, permite comparar el árbol obtenido con el del libro.",
                       citation="Quinlan, J. R. (1986). Induction of decision trees. Machine Learning, 1(1), 81-106.")
    return build_dataset(frame, spec, source="Play Tennis")


def _sklearn_dataset(title: str, loader, target_label: str, source: str, purpose: str) -> Dataset:
    bunch = loader(as_frame=True)
    frame = bunch.frame.copy()
    frame[target_label] = [bunch.target_names[int(i)] for i in frame.pop("target")]
    return build_dataset(frame, DatasetSpec(name=title, target=target_label, source=source, purpose=purpose), source=title)


def _wine() -> Dataset:
    from sklearn.datasets import load_wine
    return _sklearn_dataset("Wine", load_wine, "variedad",
                            "el UCI Machine Learning Repository, a través de la copia incluida en scikit-learn (load_wine)",
                            "Tiene más atributos que Iris (13 medidas químicas), así que sirve para ver cómo se comportan "
                            "los dos modelos cuando tienen más información disponible.")


def _breast_cancer() -> Dataset:
    from sklearn.datasets import load_breast_cancer
    return _sklearn_dataset("Breast Cancer Wisconsin", load_breast_cancer, "diagnóstico",
                            "el UCI Machine Learning Repository, a través de la copia incluida en scikit-learn (load_breast_cancer)",
                            "Es un problema real, con 30 atributos numéricos y dos clases de distinto tamaño, así que "
                            "sirve para probar los modelos en un caso más exigente.")


def default_entries() -> list:
    return [
        DatasetEntry("heart", "Enfermedad cardíaca", "303 pacientes de la Cleveland Clinic, descritos con 13 atributos "
                     "clínicos. Se predice si tienen o no una enfermedad cardíaca.",
                     load_heart_dataset, "Janosi y otros (1989), UCI",
                     disclaimer="Esta es una herramienta educativa. Sus resultados no constituyen un diagnóstico médico."),
        DatasetEntry("play_tennis", "Play Tennis", "14 días descritos por el clima. Es el ejemplo clásico del libro de Quinlan.",
                     _play_tennis, "Quinlan (1986)", default_test_size=0.0),
        DatasetEntry("iris", "Iris", "150 flores de 3 especies, descritas con 4 medidas numéricas. Es el dataset principal de desarrollo.",
                     load_iris_dataset, "Fisher (1936)"),
        DatasetEntry("wine", "Wine", "178 vinos de 3 variedades, descritos con 13 atributos químicos.", _wine, "UCI vía scikit-learn"),
        DatasetEntry("breast_cancer", "Breast Cancer Wisconsin", "569 casos con 30 medidas numéricas, clasificados en 2 diagnósticos.",
                     _breast_cancer, "UCI vía scikit-learn",
                     disclaimer="Esta es una herramienta educativa. Sus resultados no constituyen un diagnóstico médico."),
    ]


# ------------------------------------------------------------------ el servicio
class LabService:
    def __init__(self, entries=None, seed: int = 0):
        self.seed = seed
        self.entries: dict = {}
        for entry in (entries if entries is not None else default_entries()):
            self.register_dataset(entry)
        self.dataset: Dataset | None = None
        self.entry: DatasetEntry | None = None
        self.tree: C45DecisionTree | None = None
        self.forest: RandomForestModel | None = None
        self.test_size = 0.3

    # ------------------------------------------------------------ registro y selección
    def register_dataset(self, entry: DatasetEntry) -> None:
        self.entries[entry.key] = entry

    def datasets(self) -> dict:
        return {"datasets": [{"key": e.key, "title": e.title, "description": e.description, "source": e.source,
                              "disclaimer": e.disclaimer} for e in self.entries.values()]}

    def select(self, key: str) -> dict:
        if key not in self.entries:
            raise ValueError(f"Dataset desconocido {key!r}. Disponibles: {sorted(self.entries)}")
        self.entry = self.entries[key]
        self.dataset = self.entry.loader()
        self.tree = self.forest = None
        self.test_size = self.entry.default_test_size
        self._split()
        return self.info()

    def load_csv(self, text: str, target: str, name: str = "Mi archivo", nominal=()) -> dict:
        """Carga un CSV subido desde la interfaz (texto) como dataset actual."""
        with tempfile.TemporaryDirectory() as folder:
            path = Path(folder) / "datos.csv"
            path.write_text(text, encoding="utf-8")
            frame = read_table(path)
        spec = DatasetSpec(name=name, target=target, nominal=list(nominal), source="un archivo CSV que cargó el usuario")
        entry = DatasetEntry("subido", name, "Archivo subido por el usuario.",
                             lambda: build_dataset(frame, spec, source=name))
        self.register_dataset(entry)
        return self.select("subido")

    def _need_dataset(self) -> Dataset:
        if self.dataset is None:
            raise RuntimeError("Primero elegí un dataset.")
        return self.dataset

    def _split(self) -> None:
        ds = self._need_dataset()
        n = len(ds.y)
        if self.test_size and self.test_size > 0:
            split = make_splits(ds.y, scheme="holdout", n_repeats=1, test_size=self.test_size, seed=self.seed)[0]
            self.train_idx, self.test_idx = split.train, split.test
        else:
            self.train_idx, self.test_idx = np.arange(n), np.array([], dtype=int)
        self.tree = self.forest = None

    def set_split(self, test_size: float = 0.3, seed: int = 0) -> dict:
        if not 0 <= test_size < 0.9:
            raise ValueError("test_size debe estar entre 0 y 0.9")
        self.test_size, self.seed = float(test_size), int(seed)
        self._split()
        return {"n_train": int(len(self.train_idx)), "n_test": int(len(self.test_idx)), "test_size": self.test_size}

    # ------------------------------------------------------------ información del dataset
    def class_names(self) -> dict:
        return {c: n for c, n in self._need_dataset().class_names.items()}

    def info(self) -> dict:
        ds = self._need_dataset()
        report = ds.report
        names = self.class_names()
        classes = [{"value": _tidy(c), "name": str(names.get(c, c)), "count": int(report["class_counts"].get(c, 0))}
                   for c in ds.classes]
        head = ds.X.head(8)
        return {
            "key": self.entry.key, "title": self.entry.title, "description": self.entry.description,
            "source": self.entry.source, "disclaimer": self.entry.disclaimer,
            "n_rows": int(len(ds.X)), "n_features": int(ds.X.shape[1]), "classes": classes,
            "target": ds.target, "features": self._features(ds),
            "quality": {"duplicates": report["duplicates"], "rows_with_missing": report["rows_with_missing"],
                        "missing": report["missing"], "warnings": report["warnings"],
                        "constant_columns": report["constant_columns"], "id_like_columns": report["id_like_columns"]},
            "split": {"n_train": int(len(self.train_idx)), "n_test": int(len(self.test_idx)),
                      "test_size": self.test_size},
            "preview": head.astype(object).where(head.notna(), None).to_dict("records"),
            "decisions": ds.log.to_records(),
        }

    def _features(self, ds: Dataset) -> list:
        out = []
        for column in ds.X.columns:
            series, kind = ds.X[column], ds.feature_types[column]
            item = {"name": str(column), "kind": kind, "missing": int(series.isna().sum()),
                    "description": ds.descriptions.get(str(column), "")}
            if kind == "numeric":
                values = pd.to_numeric(series, errors="coerce").dropna()
                edges = np.histogram_bin_edges(values, bins=12)
                by_class = {}
                for c in ds.classes:
                    in_class = pd.to_numeric(series[ds.y == c], errors="coerce").dropna()
                    by_class[str(_tidy(c))] = np.histogram(in_class, bins=edges)[0].tolist()
                item.update(min=float(values.min()), max=float(values.max()), mean=float(values.mean()),
                            median=float(values.median()), q1=float(values.quantile(0.25)),
                            q3=float(values.quantile(0.75)), default=float(values.median()),
                            outliers=int(ds.report["outliers"].get(str(column), {}).get("count", 0)),
                            hist={"edges": edges.tolist(), "by_class": by_class})
            else:
                counts = series.dropna().map(_tidy).value_counts()
                values = sorted(counts.index, key=str)
                tidy = series.map(lambda v: None if pd.isna(v) else _tidy(v))
                by_class = {str(v): [int(((tidy == v) & (ds.y == c)).sum()) for c in ds.classes] for v in values}
                item.update(values=[_tidy(v) for v in values], counts={str(_tidy(v)): int(counts[v]) for v in values},
                            by_class=by_class, default=_tidy(counts.index[0]) if len(counts) else None)
            out.append(item)
        return out

    def decisions(self) -> dict:
        ds = self._need_dataset()
        return {"records": ds.log.to_records(), "markdown": ds.log.to_markdown()}

    # ------------------------------------------------------------ entrenamiento
    @staticmethod
    def _coerce(params: dict | None, allowed: tuple) -> dict:
        clean = {}
        for key, value in (params or {}).items():
            if key not in allowed:
                continue
            if key in INT_PARAMS and value is not None:
                value = int(value)
            elif key in ("min_samples_leaf", "min_gain", "confidence") and value is not None:
                value = float(value)
            clean[key] = value
        return clean

    def _train_test(self):
        ds = self._need_dataset()
        X, y = ds.X, ds.y.to_numpy()
        return (X.iloc[self.train_idx], y[self.train_idx], X.iloc[self.test_idx], y[self.test_idx])

    def _metrics(self, model, X, y) -> dict | None:
        if len(X) == 0:
            return None
        ds = self._need_dataset()
        proba = _align_proba(model.predict_proba(X), model.classes_, ds.classes)
        labels = np.asarray(ds.classes)
        pred = labels[proba.argmax(axis=1)]
        scores = evaluate(y, pred, proba, ds.classes)
        keep = {k: scores[k] for k in ("accuracy", "balanced_accuracy", "macro_f1", "roc_auc") if k in scores}
        keep["confusion"] = confusion(y, pred, ds.classes).tolist()
        keep["n"] = int(len(X))
        return keep

    def train(self, params: dict | None = None) -> dict:
        """Entrena el árbol C4.5 propio con la partición actual y devuelve el árbol y sus métricas."""
        ds = self._need_dataset()
        X_train, y_train, X_test, y_test = self._train_test()
        clean = self._coerce(params, DT_PARAMS)
        tree = C45DecisionTree(**clean)
        start = time.perf_counter()
        tree.fit(X_train, y_train, feature_types=ds.feature_types)
        fit_time = time.perf_counter() - start
        self.tree = tree
        names = self.class_names()
        order = np.argsort(-tree.feature_importances_)
        return {"params": {**C45DecisionTree().get_params(), **clean}, "tree": tree.to_dict(names),
                "fit_time": fit_time, "text": tree.export_text(), "dot": tree.to_dot(names),
                "metrics": {"train": self._metrics(tree, X_train, y_train),
                            "test": self._metrics(tree, X_test, y_test)},
                "importances": [{"feature": tree.feature_names_[i], "value": float(tree.feature_importances_[i])}
                                for i in order if tree.feature_importances_[i] > 0],
                "classes": [{"value": _tidy(c), "name": str(names.get(c, c))} for c in ds.classes]}

    def train_forest(self, params: dict | None = None) -> dict:
        ds = self._need_dataset()
        X_train, y_train, X_test, y_test = self._train_test()
        clean = {**FOREST_PRESET, **self._coerce(params, RF_PARAMS)}
        forest = RandomForestModel(**clean)
        start = time.perf_counter()
        forest.fit(X_train, y_train, feature_types=ds.feature_types)
        fit_time = time.perf_counter() - start
        self.forest = forest
        order = sorted(forest.feature_importances_.items(), key=lambda kv: -kv[1])
        return {"params": forest.get_params(), "fit_time": fit_time,
                "metrics": {"train": self._metrics(forest, X_train, y_train),
                            "test": self._metrics(forest, X_test, y_test)},
                "importances": [{"feature": f, "value": float(v)} for f, v in order if v > 0]}

    # ------------------------------------------------------------ predecir un caso
    def _clean_row(self, row: dict) -> dict:
        ds = self._need_dataset()
        clean = {}
        for column in ds.X.columns:
            value = (row or {}).get(str(column))
            if value is None or (isinstance(value, float) and math.isnan(value)) or value == "":
                clean[str(column)] = None
            elif ds.feature_types[column] == "numeric":
                clean[str(column)] = float(value)
            else:
                clean[str(column)] = value
        return clean

    def predict(self, row: dict) -> dict:
        """Qué dicen el árbol y el bosque de UN caso; con el camino del árbol (los ids son los de `train`)."""
        ds = self._need_dataset()
        if self.tree is None:
            self.train({})
        if self.forest is None:
            self.train_forest({})
        clean = self._clean_row(row)
        frame = pd.DataFrame([clean])
        names = self.class_names()
        labels = [str(names.get(c, c)) for c in ds.classes]
        out = {"classes": labels, "row": clean, "missing": [k for k, v in clean.items() if v is None]}
        for key, model in (("dt", self.tree), ("rf", self.forest)):
            proba = _align_proba(model.predict_proba(frame), model.classes_, ds.classes)[0]
            out[key] = {"proba": proba.tolist(), "label": labels[int(np.argmax(proba))]}
        out["dt"]["rules"] = self.tree.explain(clean)["rules"]
        out["dt"]["trace"] = self.tree.trace(clean)
        return out

    # ------------------------------------------------------------ comparar y experimentar
    def _splits(self, n_splits: int, n_repeats: int, seed: int):
        ds = self._need_dataset()
        k = max(2, min(int(n_splits), max_stratified_splits(ds.y)))
        return make_splits(ds.y, n_splits=k, n_repeats=max(1, min(int(n_repeats), 10)), seed=int(seed)), k

    def compare(self, dt_params=None, rf_params=None, n_splits: int = 5, n_repeats: int = 3, seed: int = 0) -> dict:
        """Árbol propio contra Random Forest sobre las MISMAS particiones (validación cruzada repetida)."""
        ds = self._need_dataset()
        splits, k = self._splits(n_splits, n_repeats, seed)
        dt = self._coerce(dt_params, DT_PARAMS)
        rf = {**FOREST_PRESET, **self._coerce(rf_params, RF_PARAMS)}
        results = run_experiment(ds, {"DT": ("dt", dt), "RF": ("rf", rf)}, splits)
        out = {"n_splits": k, "n_repeats": len(splits) // k, "n_partitions": len(splits),
               "params": {"DT": dt, "RF": rf}, "models": {}}
        for model in ("DT", "RF"):
            part = results[results["model"] == model]
            stats = {}
            for column, label in (("test_accuracy", "accuracy"), ("test_balanced_accuracy", "balanced_accuracy"),
                                  ("test_macro_f1", "macro_f1"), ("test_roc_auc", "roc_auc"), ("fit_time", "fit_time"),
                                  ("predict_time", "predict_time"), ("train_accuracy", "train_accuracy")):
                if column in part:
                    stats[label] = {"mean": float(part[column].mean()), "std": float(part[column].std(ddof=1))}
            gap = part["train_accuracy"] - part["test_accuracy"]
            stats["gap"] = {"mean": float(gap.mean()), "std": float(gap.std(ddof=1))}
            matrix = np.sum([np.array(json.loads(text)) for text in part["confusion"]], axis=0)
            out["models"][model] = {"stats": stats, "per_split": part["test_accuracy"].tolist(),
                                    "per_split_f1": part["test_macro_f1"].tolist(), "confusion": matrix.tolist()}
        out["paired"] = {"accuracy": paired_comparison(results, "RF", "DT", "accuracy"),
                         "macro_f1": paired_comparison(results, "RF", "DT", "macro_f1")}
        names = self.class_names()
        out["classes"] = [str(names.get(c, c)) for c in ds.classes]
        out["decision"] = next((d for d in reversed(ds.log.to_records()) if d["id"] == "D-07"), None)
        return out

    def sweep(self, kind: str, param: str, values, base_params=None, metric: str = "macro_f1",
              n_splits: int = 5, n_repeats: int = 2, seed: int = 0) -> dict:
        """Cómo cambia el rendimiento al mover UN parámetro (los demás quedan fijos)."""
        ds = self._need_dataset()
        if kind not in SWEEPABLE or param not in SWEEPABLE[kind]:
            raise ValueError(f"Parámetro no permitido: {kind}/{param}. Permitidos: {SWEEPABLE}")
        splits, _ = self._splits(n_splits, n_repeats, seed)
        allowed = DT_PARAMS if kind == "dt" else RF_PARAMS
        base = self._coerce(base_params, allowed)
        if kind == "rf":
            base = {**FOREST_PRESET, **base}
            base.pop(param, None)
        if kind == "dt" and param == "confidence":
            base["prune"] = True                         # la confianza solo actúa si se poda
        coerced = [self._coerce({param: v}, allowed)[param] for v in values]
        table = parameter_sweep(ds, kind, param, coerced, splits, base_params=base, metric=metric)
        rows = [{"value": _tidy(r["value"]), "test_mean": float(r[f"test_{metric}_mean"]),
                 "test_std": float(r[f"test_{metric}_std"]), "train_mean": float(r[f"train_{metric}_mean"]),
                 "gap": float(r["gap"]), "fit_time": float(r["fit_time_mean"])} for _, r in table.iterrows()]
        best = max(rows, key=lambda r: r["test_mean"])
        return {"kind": kind, "param": param, "metric": metric, "rows": rows, "best": best["value"],
                "n_partitions": len(splits), "base": base}

    def robustness(self, fractions=(0, 0.1, 0.2, 0.3, 0.4), seeds=(0,), n_splits: int = 5, n_repeats: int = 1,
                   dt_params=None, rf_params=None, metric: str = "macro_f1") -> dict:
        """Borra al azar una fracción de los valores (en train y en test) y mide cuánto cae cada modelo."""
        ds = self._need_dataset()
        splits, _ = self._splits(n_splits, n_repeats, 0)
        dt = self._coerce(dt_params, DT_PARAMS)
        rf = {**FOREST_PRESET, **self._coerce(rf_params, RF_PARAMS)}
        results = missing_robustness(ds, {"DT": ("dt", dt), "RF": ("rf", rf)}, [float(f) for f in fractions], splits,
                                     seeds=[int(s) for s in seeds])
        table = robustness_table(results, metric)
        spread = results.groupby(["missing_fraction", "model"])[f"test_{metric}"].std(ddof=1).unstack()
        rows = [{"fraction": float(f), "DT": float(table.loc[f, "DT"]), "RF": float(table.loc[f, "RF"]),
                 "DT_std": float(spread.loc[f, "DT"]), "RF_std": float(spread.loc[f, "RF"])} for f in table.index]
        return {"metric": metric, "rows": rows, "n_partitions": len(splits), "seeds": [int(s) for s in seeds]}

    # ------------------------------------------------------------ punto de entrada para la interfaz
    def call(self, method: str, params: dict | None = None):
        """Despacha un método permitido y devuelve el resultado listo para JSON."""
        if method not in METHODS:
            raise ValueError(f"Método desconocido: {method!r}")
        return to_jsonable(getattr(self, method)(**(params or {})))

    def handle(self, message: dict) -> dict:
        """Un pedido de la interfaz {id, method, params} -> {id, ok, result} o {id, ok: False, error}.

        Es el protocolo que comparten el widget de Colab y el servidor de desarrollo: la interfaz no sabe
        con cuál está hablando. Un error nunca tumba la sesión: vuelve como texto para mostrarlo en pantalla.
        """
        request_id = message.get("id") if isinstance(message, dict) else None
        try:
            if not isinstance(message, dict):
                raise ValueError("El pedido debe ser un objeto {id, method, params}")
            return {"id": request_id, "ok": True, "result": self.call(message.get("method", ""), message.get("params"))}
        except Exception as error:                                   # se informa en la interfaz, no se propaga
            return {"id": request_id, "ok": False, "error": f"{type(error).__name__}: {error}"}


In [ ]:
%%writefile src/tp_ia/web/widget.py
"""
Widget de anywidget: la interfaz del laboratorio dentro de un notebook (Colab o Jupyter).

GUÍA DE LECTURA
---------------
anywidget deja escribir la parte visual en JavaScript y CSS propios (static/app.js y static/app.css) y la parte de
cálculo en Python. Se comunican con mensajes: la interfaz manda {id, method, params} y este widget le contesta
{id, ok, result} o {id, ok: False, error}. Quien realmente responde es LabService.handle (service.py), el mismo
que usa el servidor de desarrollo, así que la interfaz funciona igual en los dos lados.

  - show(): muestra el laboratorio. En Colab hay que activar antes el gestor de widgets personalizados;
    show() lo hace solo (enable_colab()).
  - Los cálculos largos (comparar, barrer parámetros) bloquean el kernel mientras corren y la interfaz muestra
    "calculando"; cada pedido se contesta cuando termina.
"""
from __future__ import annotations

from pathlib import Path

import anywidget

from .service import LabService

STATIC = Path(__file__).resolve().parent / "static"


def enable_colab() -> bool:
    """Activa los widgets personalizados en Colab. Devuelve False si no hace falta o no se puede.

    Hace falta estar dentro del kernel de un notebook de Colab, no basta con tener instalado google.colab: en Colab,
    un subproceso (por ejemplo el pytest de la última celda) también puede importarlo, pero no tiene shell de
    IPython y el gestor de widgets no tiene dónde engancharse.
    """
    try:
        from google.colab import output
        from IPython import get_ipython
    except ImportError:
        return False
    if get_ipython() is None:
        return False
    output.enable_custom_widget_manager()
    return True


class TreeLab(anywidget.AnyWidget):
    _esm = STATIC / "app.js"
    _css = STATIC / "app.css"

    def __init__(self, service: LabService | None = None, **kwargs):
        super().__init__(**kwargs)
        self.service = service if service is not None else LabService()
        self.on_msg(self._on_message)

    def _on_message(self, _widget, content, _buffers) -> None:
        self.send(self.service.handle(content))


def show(service: LabService | None = None) -> TreeLab:
    """Crea el laboratorio y lo muestra en la celda."""
    from IPython.display import display
    enable_colab()
    widget = TreeLab(service)
    display(widget)
    return widget


In [ ]:
%%writefile src/tp_ia/web/static/app.js
/*
 * Laboratorio de árboles de decisión: C4.5 propio contra Random Forest.
 *
 * GUÍA DE LECTURA
 * ---------------
 * Es un módulo de anywidget: exporta { render({ model, el }) }. Todo el cálculo se hace en Python; acá solo se piden
 * cosas con model.send({ id, method, params }) y se dibuja lo que vuelve ({ id, ok, result } o { id, ok: false, error }).
 * No usa ninguna librería externa: los gráficos y el árbol son SVG armado a mano, así que funciona sin conexión.
 *
 * Partes: utilidades de DOM y de formato -> comunicación con Python -> controles -> gráficos -> una vista por pestaña
 * (Datos, Árbol, Comparar, Experimentos) -> armazón de la aplicación.
 */

const SVGNS = "http://www.w3.org/2000/svg";
const LOCALE = "es-AR";
const DEFAULT_PARAMS = {
  criterion: "gain_ratio", max_depth: null, min_samples_leaf: 2, min_gain: 0, use_avg_gain: true,
  threshold_mode: "c45", numeric_gain_correction: false, prune: false, confidence: 0.25,
};

/* ============================ utilidades de DOM y de formato ============================ */
function applyAttrs(el, attrs) {
  for (const [key, value] of Object.entries(attrs || {})) {
    if (value === null || value === undefined || value === false) continue;
    if (key.startsWith("on") && typeof value === "function") el.addEventListener(key.slice(2).toLowerCase(), value);
    else if (key === "style" && typeof value === "object") Object.assign(el.style, value);
    else if (value === true) el.setAttribute(key, "");
    else el.setAttribute(key, value);
  }
}
function append(el, kids) {
  for (const kid of kids.flat(Infinity)) {
    if (kid === null || kid === undefined || kid === false) continue;
    el.append(kid.nodeType ? kid : document.createTextNode(String(kid)));
  }
}
function h(tag, attrs = {}, ...kids) { const el = document.createElement(tag); applyAttrs(el, attrs); append(el, kids); return el; }
function s(tag, attrs = {}, ...kids) { const el = document.createElementNS(SVGNS, tag); applyAttrs(el, attrs); append(el, kids); return el; }
function clear(el) { while (el.firstChild) el.removeChild(el.firstChild); return el; }

const isNil = (x) => x === null || x === undefined || Number.isNaN(x);
const num = (x, d = 3) => (isNil(x) ? "—" : Number(x).toLocaleString(LOCALE, { minimumFractionDigits: 0, maximumFractionDigits: d }));
const fix = (x, d = 3) => (isNil(x) ? "—" : Number(x).toLocaleString(LOCALE, { minimumFractionDigits: d, maximumFractionDigits: d }));
const pct = (x, d = 1) => (isNil(x) ? "—" : `${fix(x * 100, d)} %`);
const secs = (x) => (isNil(x) ? "—" : x < 1 ? `${fix(x * 1000, 0)} ms` : `${fix(x, 2)} s`);
const classColor = (i) => `var(--c${i % 7})`;
const sum = (xs) => xs.reduce((a, b) => a + b, 0);
const clamp = (x, lo, hi) => Math.min(hi, Math.max(lo, x));
function debounce(fn, ms) { let t; return (...args) => { clearTimeout(t); t = setTimeout(() => fn(...args), ms); }; }
function shorten(text, n) { text = String(text); return text.length > n ? `${text.slice(0, n - 1)}…` : text; }
const plural = (n, one, many) => (Number(n) === 1 ? one : many);

/* ============================ comunicación con Python ============================ */
function createRpc(model) {
  let seq = 0;
  const pending = new Map();
  const onMessage = (msg) => {
    const waiting = pending.get(msg && msg.id);
    if (!waiting) return;
    pending.delete(msg.id);
    if (msg.ok) waiting.resolve(msg.result);
    else waiting.reject(new Error(msg.error || "Error desconocido"));
  };
  model.on("msg:custom", onMessage);
  const call = (method, params = {}) => new Promise((resolve, reject) => {
    const id = ++seq;
    pending.set(id, { resolve, reject });
    model.send({ id, method, params });
  });
  call.dispose = () => model.off("msg:custom", onMessage);
  return call;
}

/* ============================ controles ============================ */
function slider({ label, help, min, max, step, value, format = (v) => num(v), disabled = false, onInput, onChange }) {
  const out = h("output", {}, format(value));
  const input = h("input", { type: "range", min, max, step, value, disabled, "aria-label": label });
  input.addEventListener("input", () => { out.textContent = format(Number(input.value)); if (onInput) onInput(Number(input.value)); });
  input.addEventListener("change", () => { if (onChange) onChange(Number(input.value)); });
  const el = h("div", { class: "tl-param" },
    h("label", { class: "tl-lab" }, h("span", {}, label), out), input, help ? h("p", { class: "tl-help" }, help) : null);
  return { el, input, set(v) { input.value = v; out.textContent = format(Number(v)); }, setDisabled(d) { input.disabled = d; } };
}
function checkbox(label, checked, onChange, help) {
  const input = h("input", { type: "checkbox", checked });
  input.addEventListener("change", () => onChange(input.checked));
  return { el: h("div", { class: "tl-param" }, h("label", { class: "tl-check" }, input, h("span", {}, label)),
    help ? h("p", { class: "tl-help" }, help) : null), input };
}
function segmented(options, value, onChange) {
  const buttons = options.map(([key, text]) => {
    const b = h("button", { type: "button", "aria-pressed": String(key === value) }, text);
    b.addEventListener("click", () => { set(key); onChange(key); });
    return [key, b];
  });
  function set(key) { for (const [k, b] of buttons) b.setAttribute("aria-pressed", String(k === key)); }
  return { el: h("div", { class: "tl-seg", role: "group" }, buttons.map(([, b]) => b)), set };
}
function selectBox(options, value, onChange, attrs = {}) {
  const el = h("select", { class: "tl-select", ...attrs }, options.map(([key, text]) =>
    h("option", { value: key, selected: String(key) === String(value) }, text)));
  el.addEventListener("change", () => onChange(el.value));
  return el;
}
function button(text, onClick, kind = "") {
  const b = h("button", { type: "button", class: `tl-btn ${kind}`.trim() }, text);
  b.addEventListener("click", onClick);
  return b;
}
function field(label, control, help) {
  return h("div", { class: "tl-param" }, h("label", { class: "tl-lab" }, h("span", {}, label)), control,
    help ? h("p", { class: "tl-help" }, help) : null);
}

/* ============================ gráficos pequeños ============================ */
function stackedBar(counts, { height = 10, gap = 1 } = {}) {
  const total = sum(counts) || 1;
  return h("div", { class: "tl-pbar", style: { height: `${height}px`, gap: `${gap}px` } },
    counts.map((c, i) => c > 0 ? h("div", { style: { flex: `${c / total} 1 0`, background: classColor(i), minWidth: "2px" },
      title: `${num(c, 2)}` }) : null));
}
function legend(classes) {
  return h("div", { class: "tl-legend" }, classes.map((c, i) =>
    h("span", {}, h("i", { style: { background: classColor(i) } }), c.name + (c.count !== undefined ? ` (${num(c.count, 0)})` : ""))));
}
function miniHistogram(feature, classes, width = 260, height = 52) {
  const edges = feature.hist.edges, bins = edges.length - 1;
  const stacks = Array.from({ length: bins }, (_, b) => classes.map((c) => (feature.hist.by_class[String(c.value)] || [])[b] || 0));
  const top = Math.max(...stacks.map(sum), 1), bw = width / bins;
  const svg = s("svg", { viewBox: `0 0 ${width} ${height}`, width: "100%", height, role: "img",
    "aria-label": `Distribución de ${feature.name} por clase` });
  stacks.forEach((stack, b) => {
    let y = height;
    stack.forEach((count, i) => {
      const hh = (count / top) * (height - 2);
      y -= hh;
      if (hh > 0) svg.append(s("rect", { x: b * bw + 0.5, y, width: Math.max(bw - 1, 1), height: hh, fill: classColor(i), opacity: 0.92 }));
    });
  });
  return svg;
}
/* Barras de un atributo nominal: una por valor, partida en colores según cuántos casos de cada clase lo tienen. */
function valueBars(feature) {
  const entries = Object.entries(feature.counts).sort((a, b) => b[1] - a[1]).slice(0, 6);
  const top = Math.max(...entries.map((e) => e[1]), 1);
  return h("div", { class: "tl-bars" }, entries.map(([value, count]) => {
    const split = feature.by_class && feature.by_class[value];
    const fill = split
      ? h("div", { class: "tl-fill tl-fill-split", style: { width: `${(count / top) * 100}%` } }, split.map((n, i) =>
        n > 0 ? h("div", { style: { flex: `${n} 1 0`, background: classColor(i) }, title: `${n} ${plural(n, "caso", "casos")}` }) : null))
      : h("div", { class: "tl-fill", style: { width: `${(count / top) * 100}%` } });
    return h("div", { class: "tl-bar" }, h("span", { title: value }, shorten(value, 22)), h("div", { class: "tl-track" }, fill),
      h("span", {}, num(count, 0)));
  }));
}

/* ============================ vista: Datos ============================ */
function buildDatos(ctx) {
  const { S } = ctx;
  const info = S.info;
  const classes = info.classes;
  const missingCells = sum(Object.values(info.quality.missing).map((m) => m.count));
  const root = h("div", {});

  root.append(h("section", {},
    h("h2", { class: "tl-h2" }, info.title),
    h("p", { class: "tl-muted" }, info.description, " ", h("span", { class: "tl-small" }, `Fuente: ${info.source}`)),
    info.disclaimer ? h("p", { class: "tl-warn", style: { marginTop: "8px" } }, info.disclaimer) : null,
    h("div", { class: "tl-facts", style: { marginTop: "14px" } },
      [[num(info.n_rows, 0), plural(info.n_rows, "caso", "casos")], [num(info.n_features, 0), plural(info.n_features, "atributo", "atributos")],
        [num(classes.length, 0), plural(classes.length, "clase", "clases")],
        [num(missingCells, 0), plural(missingCells, "valor faltante", "valores faltantes")],
        [num(info.quality.duplicates, 0), plural(info.quality.duplicates, "fila duplicada", "filas duplicadas")]]
        .map(([value, label]) => h("div", { class: "tl-fact" }, h("b", {}, value), h("span", {}, label)))),
    h("div", { class: "tl-classbar", role: "img", "aria-label": "Distribución de clases" }, classes.map((c, i) =>
      h("div", { style: { flex: `${c.count} 1 0`, background: classColor(i) }, title: `${c.name}: ${c.count}` }))),
    legend(classes)));

  if (info.quality.warnings.length) {
    root.append(h("div", { class: "tl-section", style: { display: "grid", gap: "8px" } },
      info.quality.warnings.map((w) => h("p", { class: "tl-warn" }, w))));
  }

  root.append(h("section", { class: "tl-section" }, h("h3", { class: "tl-h3" }, `Atributos (${info.features.length})`,
      h("span", { class: "tl-muted tl-small", style: { fontWeight: 400 } }, " · cada color representa una clase")),
    h("div", { class: "tl-features" }, info.features.map((f) => h("div", { class: "tl-feature" },
      h("header", {}, h("b", {}, f.name),
        h("span", { style: { display: "flex", gap: "6px" } },
          f.missing ? h("span", { class: "tl-tag tl-tag-warn" }, `${f.missing} ${plural(f.missing, "faltante", "faltantes")}`) : null,
          f.kind === "numeric" && f.outliers ? h("span", { class: "tl-tag tl-tag-warn", title: "Son valores que se alejan mucho del resto (más de 1,5 veces el rango intercuartil). Se conservan porque pueden ser mediciones reales." }, `${f.outliers} ${plural(f.outliers, "valor extremo", "valores extremos")}`) : null,
          h("span", { class: "tl-tag" }, f.kind === "numeric" ? "numérico" : "nominal"))),
      f.description ? h("p", { class: "tl-feature-desc" }, f.description) : null,
      f.kind === "numeric"
        ? [miniHistogram(f, classes), h("p", { class: "tl-muted tl-small" }, `Valores entre ${num(f.min)} y ${num(f.max)} · mediana ${num(f.median)}`)]
        : valueBars(f))))));

  root.append(h("section", { class: "tl-section tl-decisions" },
    h("h3", { class: "tl-h3" }, "Decisiones tomadas sobre los datos"),
    h("p", { class: "tl-muted tl-small", style: { marginBottom: "6px" } }, "Cada decisión explica qué se hizo con los datos y por qué."),
    info.decisions.map((d, i) => h("details", {},
      h("summary", {}, `${decisionLabel(i + 1)}: `, h("span", { class: "tl-muted" }, d.chosen), " ", h("span", { class: "tl-tag" }, d.id)),
      decisionBody(d)))));

  const keys = Object.keys(info.preview[0] || {});
  root.append(h("section", { class: "tl-section" }, h("details", {}, h("summary", { class: "tl-h3", style: { cursor: "pointer" } }, "Primeras filas"),
    h("div", { class: "tl-table-wrap" }, h("table", { class: "tl-table" },
      h("thead", {}, h("tr", {}, keys.map((k) => h("th", {}, k)))),
      h("tbody", {}, info.preview.map((row) => h("tr", {}, keys.map((k) => h("td", {}, isNil(row[k]) ? "—" : typeof row[k] === "number" ? num(row[k]) : row[k]))))))))));

  root.append(buildUpload(ctx));
  return { el: root };
}

/* Decisiones sobre los datos: el mismo formato que DecisionLog.to_markdown() en Python. */
const ORDINALS = ["Primera", "Segunda", "Tercera", "Cuarta", "Quinta", "Sexta", "Séptima", "Octava", "Novena", "Décima", "Undécima", "Duodécima"];
const decisionLabel = (n) => (n >= 1 && n <= ORDINALS.length ? `${ORDINALS[n - 1]} decisión` : `Decisión n.º ${n}`);
const joinOr = (xs) => (xs.length === 1 ? String(xs[0]) : `${xs.slice(0, -1).join(", ")} o ${xs[xs.length - 1]}`);
function friendly(value) {
  if (Array.isArray(value)) return value.length ? value.join(", ") : "ninguno";
  if (value && typeof value === "object") {
    const parts = Object.entries(value).map(([k, v]) => `${k}: ${friendly(v)}`);
    return parts.length ? parts.join(parts.some((p) => p.includes(",")) ? "; " : ", ") : "ninguno";
  }
  return String(value);
}
function decisionBody(d) {
  const evidence = Object.entries(d.evidence || {});
  return h("div", {},
    h("p", { class: "tl-small" }, `Tema: ${d.title}`),
    d.rationale ? h("p", {}, h("b", {}, "Por qué: "), d.rationale) : null,
    d.alternatives.length ? h("p", {}, h("b", {}, "Otras opciones posibles: "), `${joinOr(d.alternatives)}.`) : null,
    evidence.length ? [h("p", { style: { marginBottom: "2px" } }, h("b", {}, "En qué se basa:")),
      h("ul", { class: "tl-evidence" }, evidence.map(([k, v]) => h("li", {}, `${k[0].toUpperCase()}${k.slice(1)}: ${friendly(v)}`)))] : null);
}

function buildUpload(ctx) {
  const box = h("section", { class: "tl-section" }, h("h3", { class: "tl-h3" }, "Probar con otros datos"),
    h("p", { class: "tl-muted tl-small" }, "Podés cargar un archivo CSV propio. Tiene que incluir una columna con la clase que se quiere predecir. El separador, el formato de los decimales y los valores faltantes se detectan automáticamente."));
  const form = h("div", { class: "tl-row", style: { marginTop: "8px" } });
  const picker = h("input", { type: "file", accept: ".csv,.tsv,.txt", class: "tl-file-input", "aria-label": "Archivo CSV" });
  const pickerButton = h("label", { class: "tl-btn" }, picker, "Elegir un archivo CSV…");
  picker.addEventListener("change", async () => {
    const file = picker.files[0];
    if (!file) return;
    const text = await file.text();
    const first = text.split(/\r?\n/)[0] || "";
    const sep = [";", "\t", "|", ","].map((c) => [c, first.split(c).length]).sort((a, b) => b[1] - a[1])[0][0];
    const columns = first.split(sep).map((c) => c.trim().replace(/^"|"$/g, "")).filter(Boolean);
    const target = selectBox(columns.map((c) => [c, c]), columns[columns.length - 1], () => {}, { "aria-label": "Columna de la clase" });
    clear(form).append(h("span", {}, `¿Qué columna de «${file.name}» contiene la clase a predecir?`), target,
      button("Usar este archivo", () => ctx.loadCsv(text, target.value, file.name.replace(/\.\w+$/, "")), "tl-btn-primary"));
  });
  form.append(pickerButton);
  box.append(form);
  return box;
}

/* ============================ vistas pendientes (se completan más abajo) ============================ */
/* ============================ vista: Árbol ============================ */
const RIBBON_MAX = 30;

function ruleText(node, branch) {
  return node.kind === "numeric" ? `${node.feature} ${node.edges[branch]}` : `${node.feature} = ${node.edges[branch]}`;
}

/* Acomoda el árbol: las hojas en fila, cada nodo interno centrado sobre sus hijos y un nivel por profundidad. */
function layoutTree(data, { nodeW = 124, nodeH = 54, gapX = 16, gapY = 72 } = {}) {
  const nodes = [];
  let leaves = 0;
  const column = nodeW + gapX;
  function place(node, parent, branch) {
    const item = { id: node.id, data: node, parent, depth: node.depth, children: [],
      rule: parent ? ruleText(parent.data, branch) : null, path: parent ? [...parent.path, ruleText(parent.data, branch)] : [] };
    nodes.push(item);
    if (node.leaf) item.x = leaves++ * column + column / 2;
    else {
      item.children = node.children.map((child, i) => place(child, item, i));
      item.x = (item.children[0].x + item.children[item.children.length - 1].x) / 2;
    }
    item.y = node.depth * (nodeH + gapY) + nodeH / 2 + 10;
    return item;
  }
  const root = place(data, null, 0);
  const deepest = Math.max(...nodes.map((n) => n.depth));
  return { nodes, root, nodeW, nodeH, width: Math.max(leaves, 1) * column, height: (deepest + 1) * (nodeH + gapY) - gapY + 20 };
}

function argmax(xs) { let best = 0; xs.forEach((x, i) => { if (x > xs[best]) best = i; }); return best; }

/* Dibuja el árbol como flujo de casos: el ancho de cada cinta es la cantidad de casos y se apila por clase. */
function drawTree(host, layout, { classes, zoom, trace, pinnedId, onHover, onPin }) {
  const { nodes, root, nodeW, nodeH } = layout;
  const rootN = root.data.n || 1;
  const ribbon = (n) => Math.max(1.5, (n / rootN) * RIBBON_MAX);
  const svg = s("svg", { class: `tl-tree${trace ? " has-path" : ""}`, viewBox: `0 0 ${layout.width} ${layout.height}`,
    width: layout.width * zoom, height: layout.height * zoom, role: "img",
    "aria-label": "Árbol de decisión: el ancho de cada rama es la cantidad de casos que pasan por ella" });
  const linkLayer = s("g", {}), edgeLayer = s("g", {}), nodeLayer = s("g", {});
  svg.append(linkLayer, nodeLayer, edgeLayer);
  const elements = new Map();

  for (const item of nodes) {
    if (!item.children.length) continue;
    const widths = item.children.map((c) => ribbon(c.data.n));
    let cursor = item.x - sum(widths) / 2;
    item.children.forEach((child, i) => {
      const w = widths[i], sx = cursor + w / 2;
      cursor += w;
      const y1 = item.y + nodeH / 2, y2 = child.y - nodeH / 2, ym = (y1 + y2) / 2;
      const counts = child.data.counts, total = sum(counts) || 1;
      const onPath = trace && (trace[child.id] || 0) > 0;
      const group = s("g", { class: `tl-link${onPath ? " is-path" : ""}` });
      let offset = 0;
      counts.forEach((count, k) => {
        const wk = (count / total) * w;
        if (wk > 0.05) {
          const xs = sx - w / 2 + offset + wk / 2, xe = child.x - w / 2 + offset + wk / 2;
          group.append(s("path", { d: `M ${xs} ${y1} C ${xs} ${ym}, ${xe} ${ym}, ${xe} ${y2}`, stroke: classColor(k),
            "stroke-width": wk, opacity: 0.88, fill: "none" }));
        }
        offset += wk;
      });
      linkLayer.append(group);
      edgeLayer.append(s("text", { class: "tl-edge", x: (sx + child.x) / 2, y: ym, "text-anchor": "middle", "dominant-baseline": "middle" },
        shorten(item.data.edges[i], 18)));
    });
  }

  for (const item of nodes) {
    const d = item.data, weight = trace ? trace[item.id] || 0 : 0;
    const majority = argmax(d.counts);
    const left = item.x - nodeW / 2, top = item.y - nodeH / 2;
    const g = s("g", { class: `tl-node${weight > 0 && trace ? " is-path" : ""}${pinnedId === item.id ? " is-pinned" : ""}`,
      tabindex: 0, role: "button", "aria-label": d.leaf ? `Hoja ${d.label}, ${num(d.n, 1)} casos` : `Nodo ${d.feature}, ${num(d.n, 1)} casos` });
    const box = s("rect", { class: "tl-box", x: left, y: top, width: nodeW, height: nodeH, rx: 9 });
    if (d.leaf) { box.setAttribute("style", `fill: color-mix(in srgb, ${classColor(majority)} 20%, var(--surface)); stroke: ${classColor(majority)}`); }
    g.append(box);
    if (d.leaf) {
      g.append(s("text", { class: "tl-leaf-label", x: item.x, y: top + 22, "text-anchor": "middle" }, shorten(d.label, 17)),
        s("text", { class: "tl-n", x: item.x, y: top + 40, "text-anchor": "middle" },
          `n = ${num(d.n, 1)}${d.errors > 1e-9 ? ` · errores ${num(d.errors, 1)}` : ""}`));
    } else {
      g.append(s("text", { class: "tl-feat", x: item.x, y: top + 20, "text-anchor": "middle" }, shorten(d.feature, 17)),
        s("text", { class: "tl-n", x: item.x, y: top + 35, "text-anchor": "middle" }, `n = ${num(d.n, 1)}`));
      let cursor = left + 8;
      const total = sum(d.counts) || 1;
      d.counts.forEach((count, k) => {
        const w = (count / total) * (nodeW - 16);
        if (w > 0) g.append(s("rect", { x: cursor, y: top + nodeH - 11, width: w, height: 5, fill: classColor(k), rx: 1.5 }));
        cursor += w;
      });
    }
    if (trace && weight > 0 && weight < 0.9995) {
      g.append(s("text", { class: "tl-n", x: left + nodeW - 4, y: top - 4, "text-anchor": "end", style: "font-weight:640; fill: var(--ink)" }, `×${num(weight, 2)}`));
    }
    g.addEventListener("mouseenter", () => onHover(item));
    g.addEventListener("focus", () => onHover(item));
    g.addEventListener("click", () => onPin(item));
    g.addEventListener("keydown", (e) => { if (e.key === "Enter" || e.key === " ") { e.preventDefault(); onPin(item); } });
    nodeLayer.append(g);
    elements.set(item.id, g);
  }
  clear(host).append(svg);
  return { svg, elements };
}

function confusionTable(matrix, names) {
  const rowTotals = matrix.map(sum);
  return h("table", { class: "tl-matrix" },
    h("thead", {}, h("tr", {}, h("th", {}, "real \\ predicho"), names.map((n) => h("th", {}, shorten(n, 16))))),
    h("tbody", {}, matrix.map((row, i) => h("tr", {}, h("th", {}, shorten(names[i], 16)),
      row.map((count, j) => h("td", { style: { background: `color-mix(in srgb, ${i === j ? "var(--ok)" : "var(--warn)"} ${Math.round(clamp(count / (rowTotals[i] || 1), 0, 1) * 70)}%, var(--surface-2))` } },
        num(count, 0)))))));
}

function importanceBars(items, limit = 8) {
  if (!items.length) return h("p", { class: "tl-muted" }, "El árbol no usa ningún atributo, porque quedó formado por una sola hoja que predice siempre la misma clase.");
  const top = items[0].value || 1;
  return h("div", { class: "tl-bars" }, items.slice(0, limit).map((it) => h("div", { class: "tl-bar" },
    h("span", { title: it.feature }, shorten(it.feature, 16)),
    h("div", { class: "tl-track" }, h("div", { class: "tl-fill", style: { width: `${(it.value / top) * 100}%` } })), h("span", {}, pct(it.value, 0)))));
}

function buildArbol(ctx) {
  const { S, rpc, run } = ctx;
  const info = S.info;
  const classes = info.classes;
  const hasNumeric = info.features.some((f) => f.kind === "numeric");
  const V = { result: null, item: null, pinned: null, zoom: null, seq: 0, prediction: null, values: {}, missing: {}, touched: false, layout: null, handles: null };
  info.features.forEach((f) => { V.values[f.name] = f.default; V.missing[f.name] = false; });

  /* ---- panel de parámetros ---- */
  const P = S.params;
  const retrain = debounce(() => train(), 180);
  const set = (key, value) => { P[key] = value; refreshAdvanced(); retrain(); };

  // Profundidad: el extremo derecho del slider significa "sin límite" (max_depth = null).
  const DEPTH_UNLIMITED = 13;
  const depthSlider = slider({ label: "Profundidad máxima", min: 1, max: DEPTH_UNLIMITED, step: 1, value: P.max_depth === null ? DEPTH_UNLIMITED : P.max_depth,
    format: (v) => (v >= DEPTH_UNLIMITED ? "sin límite" : num(v, 0)),
    help: "Indica cuántas preguntas seguidas puede hacer el árbol, desde la raíz hasta una hoja. En el extremo derecho, la profundidad no tiene límite.",
    onInput: (v) => set("max_depth", v >= DEPTH_UNLIMITED ? null : v) });

  // Poda: la confianza solo se muestra cuando la poda está activada.
  const confidence = slider({ label: "Confianza de la poda", min: 0.01, max: 0.5, step: 0.01, value: P.confidence,
    help: "Cuanto más bajo es este valor, más ramas se podan y más simple queda el árbol. C4.5 usa 0,25 por defecto.", onInput: (v) => set("confidence", v) });
  confidence.el.hidden = !P.prune;
  const pruneBox = checkbox("Podar por error pesimista", P.prune, (on) => { confidence.el.hidden = !on; set("prune", on); },
    "Reemplaza por una hoja las ramas que no reducen el error estimado. Así se obtiene un árbol más simple, que suele funcionar mejor con casos nuevos.");

  const testSlider = slider({ label: "Casos de prueba", min: 0, max: 50, step: 5, value: Math.round(info.split.test_size * 100),
    format: (v) => `${num(v, 0)} %`, help: "Este porcentaje de los datos se usará para probar el árbol. Si se indica 0 %, se usarán todos los datos para el entrenamiento.",
    onChange: (v) => run("Repartiendo los casos", async () => { S.info.split = await rpc("set_split", { test_size: v / 100, seed: 0 }); await train(true); }).catch(() => {}) });
  const criterion = segmented([["gain_ratio", "Razón de ganancia"], ["gain", "Ganancia"]], P.criterion, (k) => set("criterion", k));
  const threshold = segmented([["c45", "Valor observado"], ["midpoint", "Punto medio"]], P.threshold_mode, (k) => set("threshold_mode", k));

  // Opciones avanzadas: plegadas, con un contador de cuántas difieren de los valores por defecto.
  const ADVANCED = ["min_gain", "use_avg_gain", "threshold_mode", "numeric_gain_correction"];
  const advancedCount = h("span", { class: "tl-tag" });
  function refreshAdvanced() {
    const changed = ADVANCED.filter((k) => P[k] !== DEFAULT_PARAMS[k]).length;
    advancedCount.textContent = `${changed} ${plural(changed, "cambiada", "cambiadas")}`;
    advancedCount.hidden = changed === 0;
  }
  const advanced = h("details", { class: "tl-advanced" },
    h("summary", {}, "Opciones avanzadas de C4.5", advancedCount),
    h("div", { class: "tl-advanced-body" },
      slider({ label: "Ganancia mínima", min: 0, max: 0.5, step: 0.01, value: P.min_gain, help: "Si una división mejora la ganancia menos que este valor, no se realiza. Sirve para evitar divisiones que aportan muy poco.", onInput: (v) => set("min_gain", v) }).el,
      checkbox("Heurística de ganancia promedio", P.use_avg_gain, (on) => set("use_avg_gain", on), "Solo se consideran los atributos cuya ganancia de información es igual o mayor que el promedio. C4.5 lo hace para no elegir atributos con una razón de ganancia alta pero una ganancia muy baja.").el,
      hasNumeric ? field("Umbral numérico", threshold.el, "Define dónde se corta un atributo numérico: en un valor que aparece en los datos, como hace C4.5, o en el punto medio entre dos valores consecutivos.") : null,
      hasNumeric ? checkbox("Corrección de la versión 8", P.numeric_gain_correction, (on) => set("numeric_gain_correction", on), "Penaliza las divisiones sobre atributos numéricos, como propone la versión 8 de C4.5. Compensa que estos atributos ofrecen muchos puntos de corte posibles.").el : null));
  refreshAdvanced();

  const rail = h("aside", { class: "tl-rail", "aria-label": "Parámetros del árbol" },
    h("div", { class: "tl-rail-head" }, h("h2", { class: "tl-h2" }, "Parámetros"),
      button("Restablecer", () => { Object.assign(P, DEFAULT_PARAMS); ctx.S.views.arbol = null; ctx.goto("arbol"); }, "tl-btn-link")),
    testSlider.el,
    field("Criterio de corte", criterion.el, "El algoritmo C4.5 utiliza la razón de ganancia en lugar de la ganancia de información, ya que esta última tiende a favorecer atributos con muchos valores distintos, lo que genera divisiones con muchas ramas."),
    depthSlider.el,
    slider({ label: "Mínimo de casos por hoja", min: 1, max: 20, step: 1, value: P.min_samples_leaf, help: "Para que el árbol haga una división, al menos dos de las ramas resultantes deben recibir esta cantidad de casos. Equivale al parámetro -m de C4.5.", onInput: (v) => set("min_samples_leaf", v) }).el,
    h("div", { class: "tl-rail-group" }, pruneBox.el, confidence.el),
    advanced);

  /* ---- zona principal ---- */
  const stats = h("div", { class: "tl-stats" });
  const treeHost = h("div", {});
  const tools = h("div", { class: "tl-treetools" },
    button("−", () => zoomBy(0.8)), button("+", () => zoomBy(1.25)), button("Ajustar", () => { V.zoom = null; drawNow(); }));
  tools.querySelectorAll("button").forEach((b, i) => b.setAttribute("aria-label", ["Alejar", "Acercar", "Ajustar al ancho"][i]));
  const treeWrap = h("div", { class: "tl-treewrap" }, tools, treeHost);
  const inspector = h("div", { class: "tl-inspector" });
  const caseHost = h("div", { class: "tl-predict" });
  const clearCase = button("Quitar el caso", () => { V.prediction = null; V.touched = false; drawPrediction(); drawNow(); }, "tl-btn-link");
  const importanceHost = h("div", {});
  const metricsHost = h("div", {});
  const sourceHost = h("div", {});
  const lower = h("div", { class: "tl-lower" },
    h("section", { class: "tl-card" }, h("h3", { class: "tl-h3" }, "Nodo"),
      h("p", { class: "tl-muted tl-small", style: { marginBottom: "8px" } }, "Pasá el mouse sobre un nodo del árbol para ver su detalle. Si hacés clic, el detalle queda fijo."),
      inspector),
    h("section", { class: "tl-card tl-card-accent" },
      h("div", { class: "tl-card-head" }, h("h3", { class: "tl-h3" }, "Predecir un caso"), clearCase),
      h("p", { class: "tl-muted tl-small", style: { marginBottom: "10px" } }, "Mové los valores para describir un caso. Los dos modelos lo clasifican al instante y el árbol resalta el camino que sigue."),
      caseHost));
  const lower2 = h("div", { class: "tl-lower tl-lower-even" },
    h("section", { class: "tl-card" }, h("h3", { class: "tl-h3" }, "Qué atributos usa el árbol"), importanceHost),
    h("section", { class: "tl-card" }, metricsHost));
  const main = h("div", {}, stats, treeWrap, lower, lower2, sourceHost);
  const el = h("div", { class: "tl-arbol" }, rail, main);

  function zoomBy(factor) { V.zoom = clamp((V.zoom || 1) * factor, 0.3, 2.5); drawNow(); }

  function drawInspector() {
    const item = V.item || V.layout.root;
    const d = item.data;
    clear(inspector);
    inspector.append(
      item.path.length ? h("p", { class: "tl-muted tl-small", style: { marginBottom: "6px" } }, item.path.join("  →  ")) : h("p", { class: "tl-muted tl-small", style: { marginBottom: "6px" } }, "Este es el nodo raíz, que contiene todos los casos de entrenamiento."),
      h("dl", {}, ...[
        d.leaf ? ["Predice la clase", d.label] : ["Divide según", d.feature],
        ["Casos", num(d.n, 2)], ["Entropía", fix(d.entropy)],
        d.leaf ? null : ["Ganancia de información", fix(d.gain)], d.leaf ? null : ["Razón de ganancia", fix(d.gain_ratio)],
        ["Errores si fuera una hoja", num(d.errors, 2)],
      ].filter(Boolean).flatMap(([k, v]) => [h("dt", {}, k), h("dd", {}, v)])),
      h("div", { class: "tl-bars", style: { marginTop: "10px" } }, classes.map((c, i) => h("div", { class: "tl-bar" },
        h("span", {}, h("i", { class: "tl-dot", style: { background: classColor(i) } }), shorten(c.name, 16)),
        h("div", { class: "tl-track" }, h("div", { class: "tl-fill", style: { width: `${(d.counts[i] / (d.n || 1)) * 100}%`, background: classColor(i) } })),
        h("span", {}, num(d.counts[i], 1))))));
  }

  function drawNow() {
    if (!V.result) return;
    V.layout = layoutTree(V.result.tree.root);
    const width = treeWrap.clientWidth || 760;
    const zoom = V.zoom || clamp((width - 8) / V.layout.width, 0.4, 1);
    V.handles = drawTree(treeHost, V.layout, {
      classes, zoom, trace: V.prediction ? V.prediction.dt.trace : null, pinnedId: V.pinned,
      onHover: (item) => { V.item = item; drawInspector(); },
      onPin: (item) => { V.pinned = V.pinned === item.id ? null : item.id; V.item = item; for (const [id, g] of V.handles.elements) g.classList.toggle("is-pinned", id === V.pinned); drawInspector(); },
    });
    drawInspector();
  }

  function drawStats() {
    const r = V.result, test = r.metrics.test, shown = test || r.metrics.train;
    clear(stats).append(...[
      [pct(shown.accuracy), test ? "exactitud en prueba" : "exactitud en entrenamiento"],
      [fix(shown.macro_f1), "F1 macro"], [num(r.tree.n_nodes, 0), r.tree.n_pruned ? `nodos (se podaron ${r.tree.n_pruned})` : "nodos"],
      [num(r.tree.n_leaves, 0), "hojas"], [num(r.tree.depth, 0), "profundidad"], [secs(r.fit_time), "tiempo de entrenamiento"],
    ].map(([value, label]) => h("div", { class: "tl-stat" }, h("b", {}, value), h("span", {}, label))));
  }

  function drawMetrics() {
    const r = V.result, test = r.metrics.test, names = classes.map((c) => c.name);
    const matrix = (test || r.metrics.train).confusion;
    clear(importanceHost).append(importanceBars(r.importances),
      h("p", { class: "tl-muted tl-small", style: { marginTop: "8px" } }, "Indica cuánto contribuye cada atributo a las divisiones del árbol. Se calcula sumando la ganancia de cada división que usa el atributo, ponderada por la cantidad de casos que pasan por ella."));
    clear(metricsHost).append(
      h("h3", { class: "tl-h3" }, test ? "Aciertos y errores en los casos de prueba" : "Aciertos y errores en entrenamiento"),
      h("div", { class: "tl-table-wrap" }, confusionTable(matrix, names)),
      test ? h("p", { class: "tl-muted tl-small", style: { marginTop: "8px" } }, `Con los datos de entrenamiento, la exactitud es de ${pct(r.metrics.train.accuracy)}. Si es mucho mayor que con los datos de prueba, el árbol está memorizando los casos en lugar de aprender reglas generales.`) : h("p", { class: "tl-muted tl-small", style: { marginTop: "8px" } }, "No hay casos de prueba, así que la exactitud se mide con los mismos datos de entrenamiento y resulta demasiado optimista."));
    clear(sourceHost).append(
      h("details", { class: "tl-source" }, h("summary", {}, "Ver el árbol como texto y como Graphviz"),
        h("pre", { class: "tl-code", style: { marginTop: "8px" } }, r.text),
        h("p", { class: "tl-muted tl-small", style: { margin: "10px 0 4px" } }, "Código DOT del árbol. Se puede dibujar con el paquete graphviz de Python o pegándolo en un visor de Graphviz en línea."),
        h("pre", { class: "tl-code" }, r.dot)));
  }

  /* ---- predecir un caso ---- */
  const predictSoon = debounce(() => predict(), 200);
  function buildCaseForm() {
    const rows = info.features.map((f) => {
      const missing = h("input", { type: "checkbox", "aria-label": `${f.name}: faltante` });
      const out = h("output", {}, "");
      let input;
      if (f.kind === "numeric") {
        const step = (f.max - f.min) / 200 || 0.01;
        input = h("input", { type: "range", min: f.min, max: f.max, step, value: V.values[f.name], "aria-label": f.name });
        out.textContent = num(V.values[f.name], 2);
        input.addEventListener("input", () => { V.values[f.name] = Number(input.value); out.textContent = num(V.values[f.name], 2); V.touched = true; predictSoon(); });
      } else {
        input = selectBox(f.values.map((v) => [v, v]), V.values[f.name], (v) => { V.values[f.name] = f.values.find((x) => String(x) === v); out.textContent = v; V.touched = true; predictSoon(); }, { "aria-label": f.name });
        out.textContent = String(V.values[f.name]);
      }
      missing.addEventListener("change", () => { V.missing[f.name] = missing.checked; input.disabled = missing.checked; out.textContent = missing.checked ? "faltante" : out.textContent; V.touched = true; predictSoon(); });
      return h("div", { class: "tl-case-row" }, h("label", { class: "tl-lab" }, h("span", {}, f.name), out),
        h("div", { style: { display: "flex", gap: "8px", alignItems: "center" } }, input,
          h("label", { class: "tl-check tl-small", title: "Probar qué pasa si falta este dato" }, missing, "falta")));
    });
    return h("div", { class: "tl-case" }, rows);
  }
  const caseForm = buildCaseForm();
  const caseResult = h("div", { class: "tl-predict-result" });
  caseHost.append(caseForm, caseResult);

  function verdict(name, cls, model, names) {
    return h("div", {}, h("div", { class: "tl-verdict" }, h("span", { class: cls }, h("i", { class: "tl-dot", style: { background: "currentColor" } }), name),
      h("b", {}, model.label)), stackedBar(model.proba, { height: 12, gap: 2 }),
      h("p", { class: "tl-muted tl-small", style: { marginTop: "3px" } }, names.map((n, i) => `${shorten(n, 14)} ${pct(model.proba[i], 0)}`).join(" · ")));
  }
  function drawPrediction() {
    clear(caseResult);
    const p = V.prediction;
    clearCase.hidden = !p;
    if (!p) { caseResult.append(h("p", { class: "tl-empty tl-small" }, "Mové cualquier valor de la izquierda para ver qué clase predice cada modelo.")); return; }
    caseResult.append(h("div", {}, verdict("Árbol propio", "tl-dtc", p.dt, p.classes), verdict("Random Forest", "tl-rfc", p.rf, p.classes),
      p.dt.rules.length ? h("ol", { class: "tl-rules" }, p.dt.rules.map((r) => h("li", {}, r))) : null,
      p.missing.length ? h("p", { class: "tl-muted tl-small", style: { marginTop: "6px" } }, `Falta el dato de ${p.missing.join(", ")}. Como no puede elegir una sola rama, el árbol reparte el caso entre todas, en proporción a los casos de entrenamiento de cada una. Los números ×0,xx del dibujo muestran ese reparto.`) : null,
      info.disclaimer ? h("p", { class: "tl-warn tl-small", style: { marginTop: "8px" } }, info.disclaimer) : null));
  }
  async function predict() {
    const row = {};
    for (const f of info.features) row[f.name] = V.missing[f.name] ? null : V.values[f.name];
    const mine = ++V.seq;
    await run("Prediciendo", async () => {
      const prediction = await rpc("predict", { row });
      if (mine !== V.seq || !V.touched) return;   // llegó tarde o se quitó el caso mientras tanto
      V.prediction = prediction;
      drawPrediction();
      drawNow();
    }).catch(() => {});
  }

  async function train(resetCase = false) {
    const mine = ++V.seq;
    await run("Entrenando el árbol", async () => {
      const result = await rpc("train", { params: P });
      if (mine !== V.seq) return;
      V.result = result; V.item = null; V.pinned = null;
      if (resetCase) V.prediction = null;
      drawStats(); drawMetrics(); drawPrediction(); drawNow();
    }).catch(() => {});
    if (V.touched && V.result) predict();
  }

  drawPrediction();
  return { el, onShow() { if (!V.result) train(); else drawNow(); } };
}
/* ============================ gráficos de líneas y de pares ============================ */
function lineChart({ width = 470, height = 260, series, labels, xs = null, yMin = null, yMax = null, markIndex = null, ariaLabel = "Gráfico" }) {
  const m = { l: 50, r: 32, t: 14, b: 40 };   // margen derecho amplio para que la última etiqueta no se corte
  const w = width - m.l - m.r, ht = height - m.t - m.b, n = labels.length;
  const all = series.flatMap((se) => se.points.flatMap((p) => [p.y, p.lo, p.hi].filter((v) => !isNil(v))));
  let lo = yMin !== null ? yMin : Math.min(...all), hi = yMax !== null ? yMax : Math.max(...all);
  if (yMin === null) lo = Math.max(0, lo - ((hi - lo) * 0.15 || 0.05));
  if (yMax === null) hi = Math.min(1, hi + ((hi - lo) * 0.15 || 0.05));
  const x = (i) => (xs ? m.l + ((xs[i] - xs[0]) / ((xs[n - 1] - xs[0]) || 1)) * w : m.l + (n === 1 ? w / 2 : (i / (n - 1)) * w));
  const y = (v) => m.t + (1 - (v - lo) / ((hi - lo) || 1)) * ht;
  const svg = s("svg", { class: "tl-chart", viewBox: `0 0 ${width} ${height}`, role: "img", "aria-label": ariaLabel, style: { width: "100%", height: "auto", maxWidth: `${width}px` } });
  for (let k = 0; k < 5; k++) {
    const v = lo + ((hi - lo) * k) / 4;
    svg.append(s("line", { class: "tl-grid", x1: m.l, x2: width - m.r, y1: y(v), y2: y(v) }),
      s("text", { x: m.l - 8, y: y(v), "text-anchor": "end", "dominant-baseline": "middle" }, fix(v, 2)));
  }
  labels.forEach((text, i) => svg.append(s("text", { x: x(i), y: height - 14, "text-anchor": "middle" }, shorten(text, 12))));
  svg.append(s("line", { class: "tl-axis", x1: m.l, x2: width - m.r, y1: m.t + ht, y2: m.t + ht }));
  for (const se of series) {
    const pts = se.points.map((p, i) => ({ ...p, px: x(i) }));
    if (pts.every((p) => !isNil(p.lo) && !isNil(p.hi)) && pts.length > 1) {
      const top = pts.map((p) => `${p.px},${y(p.hi)}`).join(" L "), bottom = [...pts].reverse().map((p) => `${p.px},${y(p.lo)}`).join(" L ");
      svg.append(s("path", { d: `M ${top} L ${bottom} Z`, fill: se.color, opacity: 0.16 }));
    }
    svg.append(s("path", { d: `M ${pts.map((p) => `${p.px},${y(p.y)}`).join(" L ")}`, fill: "none", stroke: se.color, "stroke-width": 2.4,
      "stroke-dasharray": se.dash || null, "stroke-linejoin": "round", "stroke-linecap": "round" }));
    pts.forEach((p, i) => svg.append(s("circle", { cx: p.px, cy: y(p.y), r: 3.6, fill: se.color }, s("title", {}, `${se.name} · ${labels[i]}: ${fix(p.y, 3)}`))));
  }
  if (markIndex !== null && markIndex >= 0) {
    const first = series[0].points[markIndex];
    svg.append(s("circle", { cx: x(markIndex), cy: y(first.y), r: 9, fill: "none", stroke: "var(--ink)", "stroke-width": 1.6 }));
  }
  return svg;
}

/* Un trazo por partición entre el valor del árbol y el del bosque: se ve quién gana en cada una. */
function pairedPlot({ a, b, nameA, nameB, colorA, colorB, width = 320, height = 250 }) {
  const m = { l: 46, r: 16, t: 14, b: 38 };
  const w = width - m.l - m.r, ht = height - m.t - m.b;
  const all = [...a, ...b];
  const pad = (Math.max(...all) - Math.min(...all)) * 0.2 || 0.04;
  const lo = Math.max(0, Math.min(...all) - pad), hi = Math.min(1, Math.max(...all) + pad);
  const y = (v) => m.t + (1 - (v - lo) / ((hi - lo) || 1)) * ht;
  const xa = m.l + w * 0.25, xb = m.l + w * 0.75;
  const svg = s("svg", { class: "tl-chart", viewBox: `0 0 ${width} ${height}`, role: "img", style: { width: "100%", height: "auto", maxWidth: `${width}px` },
    "aria-label": `Exactitud por partición: ${nameA} contra ${nameB}` });
  for (let k = 0; k < 5; k++) {
    const v = lo + ((hi - lo) * k) / 4;
    svg.append(s("line", { class: "tl-grid", x1: m.l, x2: width - m.r, y1: y(v), y2: y(v) }),
      s("text", { x: m.l - 8, y: y(v), "text-anchor": "end", "dominant-baseline": "middle" }, fix(v, 2)));
  }
  a.forEach((va, i) => {
    const vb = b[i], color = Math.abs(vb - va) < 1e-9 ? "var(--muted)" : vb > va ? colorB : colorA;
    svg.append(s("line", { x1: xa, y1: y(va), x2: xb, y2: y(vb), stroke: color, "stroke-width": 1.6, opacity: 0.6 }),
      s("circle", { cx: xa, cy: y(va), r: 3, fill: colorA, opacity: 0.8 }), s("circle", { cx: xb, cy: y(vb), r: 3, fill: colorB, opacity: 0.8 }));
  });
  const meanA = sum(a) / a.length, meanB = sum(b) / b.length;
  svg.append(s("line", { x1: xa - 16, x2: xa + 16, y1: y(meanA), y2: y(meanA), stroke: colorA, "stroke-width": 4, "stroke-linecap": "round" }),
    s("line", { x1: xb - 16, x2: xb + 16, y1: y(meanB), y2: y(meanB), stroke: colorB, "stroke-width": 4, "stroke-linecap": "round" }),
    s("text", { x: xa, y: height - 14, "text-anchor": "middle" }, nameA), s("text", { x: xb, y: height - 14, "text-anchor": "middle" }, nameB));
  return svg;
}

const METRIC_LABELS = { macro_f1: "F1 macro", accuracy: "Exactitud", balanced_accuracy: "Exactitud balanceada" };

/* ============================ vista: Comparar ============================ */
function buildComparar(ctx) {
  const { S, rpc, run } = ctx;
  const V = { folds: 5, repeats: 3, trees: 100, depth: 10, unlimited: false };
  const folds = slider({ label: "Pliegues", min: 2, max: 10, step: 1, value: V.folds, format: (v) => num(v, 0), onInput: (v) => { V.folds = v; },
    help: "Los datos se dividen en esta cantidad de partes. Cada parte se usa una vez para probar los modelos, y el resto se usa para entrenarlos." });
  const repeats = slider({ label: "Repeticiones", min: 1, max: 10, step: 1, value: V.repeats, format: (v) => num(v, 0), onInput: (v) => { V.repeats = v; },
    help: "Cantidad de veces que se repite la división al azar. Con más repeticiones, los resultados son más estables, aunque el cálculo tarda más." });
  const trees = slider({ label: "Cantidad de árboles", min: 10, max: 300, step: 10, value: V.trees, format: (v) => num(v, 0), onInput: (v) => { V.trees = v; } });
  const depth = slider({ label: "Profundidad máxima", min: 1, max: 20, step: 1, value: V.depth, format: (v) => num(v, 0), onInput: (v) => { V.depth = v; },
    help: "La cátedra propone usar 100 árboles con una profundidad máxima de 10." });
  const noLimit = checkbox("Sin límite de profundidad", false, (on) => { V.unlimited = on; depth.setDisabled(on); });
  const result = h("div", { style: { marginTop: "22px" } }, h("p", { class: "tl-empty" }, "Elegí los valores y tocá «Comparar» para entrenar y evaluar los dos modelos."));

  function dtSummary() {
    const p = S.params;
    return `${p.prune ? `Con poda (confianza ${num(p.confidence, 2)})` : "Sin poda"}, profundidad ${p.max_depth === null ? "sin límite" : p.max_depth}, mínimo ${num(p.min_samples_leaf, 0)} casos por hoja.`;
  }
  const go = button("Comparar", () => run("Comparando los modelos; puede tardar unos segundos", async () => {
    go.disabled = true;
    try {
      const r = await rpc("compare", { dt_params: S.params, rf_params: { n_estimators: V.trees, max_depth: V.unlimited ? null : V.depth },
        n_splits: V.folds, n_repeats: V.repeats });
      draw(r);
    } finally { go.disabled = false; }
  }).catch(() => {}), "tl-btn-primary");

  function row(label, key, formatter, r, note) {
    const cell = (m) => { const v = r.models[m].stats[key]; return v ? `${formatter(v.mean)} ± ${formatter(v.std)}` : "—"; };
    return h("tr", {}, h("td", { title: note || "" }, label), h("td", {}, cell("DT")), h("td", {}, cell("RF")));
  }

  function draw(r) {
    const d = r.paired.accuracy, dt = r.models.DT, rf = r.models.RF;
    const lead = Math.abs(d.mean_diff) < 0.0005 ? "En exactitud, el árbol y el bosque obtienen el mismo resultado promedio."
      : `En exactitud, el ${d.mean_diff > 0 ? "bosque supera al árbol" : "árbol supera al bosque"} por ${fix(Math.abs(d.mean_diff) * 100, 1)} puntos en promedio.`;
    clear(result).append(
      h("div", { class: "tl-note" },
        h("p", {}, h("b", {}, lead), ` De las ${d.n} particiones, el bosque gana en ${d.wins_a}, el árbol gana en ${d.wins_b} y empatan en ${d.ties}.`),
        h("p", { style: { marginTop: "4px" } }, `Según la prueba de Wilcoxon (p = ${fix(d.p_wilcoxon, 3)}), ${d.p_wilcoxon < 0.05 ? "la diferencia se mantiene de forma consistente entre las particiones" : "la diferencia no es lo bastante consistente, por lo que podría deberse al azar"}. Este resultado es solo orientativo, porque las particiones comparten datos entre sí.`)),
      h("div", { class: "tl-two tl-section" },
        h("div", {}, h("h3", { class: "tl-h3" }, `Resultados promedio en ${r.n_partitions} particiones (media ± desvío estándar)`),
          h("div", { class: "tl-table-wrap" }, h("table", { class: "tl-table" },
            h("thead", {}, h("tr", {}, h("th", {}, ""), h("th", { class: "tl-dtc" }, "Árbol propio"), h("th", { class: "tl-rfc" }, "Random Forest"))),
            h("tbody", {}, row("Exactitud", "accuracy", (v) => fix(v, 3), r), row("Exactitud balanceada", "balanced_accuracy", (v) => fix(v, 3), r),
              row("F1 macro", "macro_f1", (v) => fix(v, 3), r), row("Área bajo la curva ROC", "roc_auc", (v) => fix(v, 3), r),
              row("Exactitud en entrenamiento", "train_accuracy", (v) => fix(v, 3), r),
              row("Brecha entrenamiento − prueba", "gap", (v) => fix(v, 3), r, "Diferencia entre la exactitud con los datos de entrenamiento y con los de prueba. Cuanto mayor es, más sobreajuste hay."),
              row("Tiempo de entrenamiento", "fit_time", secs, r), row("Tiempo de predicción", "predict_time", secs, r)))),
          h("p", { class: "tl-muted tl-small", style: { marginTop: "8px" } }, "Los tiempos no se pueden comparar de forma directa, porque el árbol propio está escrito en Python puro y el bosque de scikit-learn usa código compilado, que es mucho más rápido.")),
        h("div", {}, h("h3", { class: "tl-h3" }, "Exactitud en cada partición"),
          pairedPlot({ a: dt.per_split, b: rf.per_split, nameA: "Árbol propio", nameB: "Random Forest", colorA: "var(--dt)", colorB: "var(--rf)" }),
          h("p", { class: "tl-muted tl-small" }, "Cada línea une la exactitud del árbol y la del bosque en una misma partición. Es esperable que los resultados del árbol varíen más que los del bosque, porque el bosque promedia muchos árboles."))),
      h("div", { class: "tl-two tl-section" },
        ["DT", "RF"].map((m) => h("div", {}, h("h3", { class: `tl-h3 ${m === "DT" ? "tl-dtc" : "tl-rfc"}` }, `Matriz de confusión: ${m === "DT" ? "árbol propio" : "Random Forest"}`),
          h("div", { class: "tl-table-wrap" }, confusionTable(r.models[m].confusion, r.classes)),
          h("p", { class: "tl-muted tl-small" }, "Suma de los resultados de todas las particiones de prueba.")))),
      h("section", { class: "tl-section" }, h("h3", { class: "tl-h3" }, "En qué se diferencian"),
        h("ul", { class: "tl-rules" },
          h("li", {}, "El árbol propio elige las divisiones con la razón de ganancia de C4.5 y puede crear una rama por cada valor de un atributo nominal. En cambio, los árboles del bosque siempre dividen en dos ramas, como en el algoritmo CART."),
          h("li", {}, "Cuando falta un valor, el árbol propio reparte el caso entre las ramas. El bosque, en cambio, necesita completar antes el dato faltante con la mediana o el valor más frecuente del entrenamiento."),
          h("li", {}, "Un único árbol es fácil de leer y de explicar. El bosque combina muchos árboles, lo que hace sus resultados más estables, pero el modelo ya no se puede interpretar a simple vista."))),
      r.decision ? h("details", { class: "tl-section tl-decisions" },
        h("summary", {}, "Decisión sobre la evaluación: ", h("span", { class: "tl-muted" }, r.decision.chosen), " ", h("span", { class: "tl-tag" }, r.decision.id)),
        decisionBody(r.decision)) : null);
  }

  const el = h("div", {},
    h("h2", { class: "tl-h2" }, "Comparar el árbol propio con el Random Forest"),
    h("p", { class: "tl-muted", style: { maxWidth: "70ch" } }, "Los dos modelos se entrenan y se evalúan con exactamente las mismas particiones de los datos. De esta forma, las diferencias se deben a los modelos y no a la suerte en la división de los datos."),
    h("div", { class: "tl-card", style: { marginTop: "14px" } },
      h("div", { class: "tl-groups" },
        h("fieldset", { class: "tl-group" }, h("legend", {}, "Cómo se evalúa"), folds.el, repeats.el),
        h("fieldset", { class: "tl-group tl-group-rf" }, h("legend", {}, "Random Forest"), trees.el, depth.el, noLimit.el),
        h("fieldset", { class: "tl-group tl-group-dt" }, h("legend", {}, "Árbol propio"),
          h("p", { class: "tl-small", "data-role": "dt-summary" }, dtSummary()),
          h("p", { class: "tl-muted tl-small" }, "Se usan los mismos parámetros elegidos en la pestaña Árbol."),
          button("Cambiar en la pestaña Árbol", () => ctx.goto("arbol"), "tl-btn-link"))),
      h("div", { class: "tl-actions" }, go)),
    result);
  return { el, onShow() { const p = el.querySelector("[data-role=dt-summary]"); if (p) p.textContent = dtSummary(); } };
}

/* ============================ vista: Experimentos ============================ */
/* Cada parámetro tiene su nombre, los valores sugeridos (null = sin límite) y qué se espera ver al probarlos. */
const SWEEPS = {
  dt: {
    max_depth: { label: "Profundidad máxima", values: [1, 2, 3, 4, 5, 6, 8, null],
      expect: "Con poca profundidad, el árbol es demasiado simple y se equivoca tanto con los datos de entrenamiento como con los de prueba. Al aumentar la profundidad, el resultado mejora hasta cierto punto. Desde ahí, el árbol sigue acertando más en el entrenamiento, pero ya no mejora en la prueba: esa separación entre las dos líneas es el sobreajuste." },
    min_samples_leaf: { label: "Mínimo de casos por hoja", values: [1, 2, 3, 5, 8, 12, 20],
      expect: "Con un mínimo bajo, el árbol puede crear hojas con muy pocos casos y adaptarse demasiado a los datos de entrenamiento. Al subir el mínimo, el árbol se vuelve más simple. Si el mínimo es demasiado alto, el árbol pierde detalle y empieza a equivocarse más." },
    min_samples_split: { label: "Mínimo de casos para dividir", values: [2, 4, 8, 16, 32],
      expect: "Este parámetro indica cuántos casos necesita un nodo para poder dividirse. Su efecto se parece al del mínimo de casos por hoja: con valores altos, el árbol queda más simple." },
    min_gain: { label: "Ganancia mínima", values: [0, 0.01, 0.02, 0.05, 0.1, 0.2],
      expect: "Con una ganancia mínima de 0, el árbol hace cualquier división que mejore algo. Al subirla, solo se hacen las divisiones más útiles y el árbol queda más chico. Si es demasiado alta, el árbol puede quedar con muy pocas divisiones y perder exactitud." },
    confidence: { label: "Confianza de la poda", values: [0.01, 0.05, 0.1, 0.25, 0.4, 0.5],
      expect: "Con una confianza baja, la poda es más agresiva y el árbol queda más simple. Con una confianza alta, se poda menos. Suele haber un valor intermedio que logra un buen equilibrio entre simplicidad y exactitud." },
  },
  rf: {
    n_estimators: { label: "Cantidad de árboles", values: [1, 3, 10, 30, 100, 300],
      expect: "Con pocos árboles, el bosque se parece a un árbol solo y sus resultados varían mucho. Al agregar árboles, el resultado mejora y se estabiliza. A partir de cierta cantidad, agregar más árboles casi no cambia el resultado y solo hace más lento el cálculo." },
    max_depth: { label: "Profundidad máxima", values: [1, 2, 3, 5, 8, 12, null],
      expect: "Con árboles poco profundos, el bosque es demasiado simple. Al aumentar la profundidad, el resultado mejora. A diferencia de un árbol solo, el bosque suele sobreajustar menos, porque promedia muchos árboles distintos." },
    min_samples_leaf: { label: "Mínimo de casos por hoja", values: [1, 2, 5, 10, 20],
      expect: "Al subir el mínimo de casos por hoja, cada árbol del bosque queda más simple. Con valores muy altos, los árboles pierden detalle y el resultado empeora." },
    min_samples_split: { label: "Mínimo de casos para dividir", values: [2, 4, 8, 16],
      expect: "Este parámetro indica cuántos casos necesita un nodo para poder dividirse. Con valores altos, los árboles del bosque quedan más simples." },
  },
};
const METRIC_HELP = {
  macro_f1: "Promedia el F1 de cada clase, así que todas las clases pesan lo mismo, aunque algunas tengan pocos casos.",
  accuracy: "Es la proporción de casos que el modelo clasifica correctamente.",
  balanced_accuracy: "Promedia el porcentaje de aciertos de cada clase. Es útil cuando algunas clases tienen muchos más casos que otras.",
};
const FRACTIONS = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8];

function parseValue(text) {
  const t = text.trim().replace(",", ".");
  if (/^(none|null|sin l[ií]mite|infinito|∞)$/i.test(t)) return null;
  const value = Number(t);
  if (!t || Number.isNaN(value)) throw new Error(`No se pudo interpretar el valor «${text.trim()}». Escribí un número, por ejemplo 0.25.`);
  return value;
}
const valueLabel = (v) => (v === null ? "sin límite" : num(v, 3));
const byValue = (a, b) => (a === null ? 1 : b === null ? -1 : a - b);   // "sin límite" va al final

/* Grupo de valores que se activan o desactivan con un clic, con la opción de agregar uno propio. */
function chipPicker({ values, selected = values, format = valueLabel, allowCustom = true, label }) {
  let options = [...values];
  let chosen = new Set(selected);
  const list = h("div", { class: "tl-chips", role: "group", "aria-label": label });
  function render() {
    clear(list).append(...options.map((v) => {
      const chip = h("button", { type: "button", class: "tl-chip", "aria-pressed": String(chosen.has(v)) }, format(v));
      chip.addEventListener("click", () => { if (chosen.has(v)) chosen.delete(v); else chosen.add(v); render(); });
      return chip;
    }));
  }
  const custom = h("input", { class: "tl-input tl-chip-input", type: "text", placeholder: "Otro valor", "aria-label": `Agregar otro valor a ${label}` });
  const add = () => {
    if (!custom.value.trim()) return;
    const value = parseValue(custom.value);
    if (!options.includes(value)) options = [...options, value].sort(byValue);
    chosen.add(value);
    custom.value = "";
    render();
  };
  custom.addEventListener("keydown", (e) => { if (e.key === "Enter") { e.preventDefault(); try { add(); } catch (err) { custom.setCustomValidity(err.message); custom.reportValidity(); } } });
  custom.addEventListener("input", () => custom.setCustomValidity(""));
  render();
  return {
    el: h("div", {}, list, allowCustom ? h("div", { class: "tl-chip-add" }, custom, button("Agregar", () => { try { add(); } catch (err) { custom.setCustomValidity(err.message); custom.reportValidity(); } })) : null),
    get: () => options.filter((v) => chosen.has(v)),
    reset(newValues) { options = [...newValues]; chosen = new Set(newValues); render(); },
  };
}

/* Recuadro que se muestra antes de correr un experimento: cuenta qué se espera ver. */
function expectBox(text) {
  return h("div", { class: "tl-expect" }, h("p", { class: "tl-expect-title" }, "Qué se espera ver"), h("p", {}, text),
    h("p", { class: "tl-muted tl-small", style: { marginTop: "8px" } }, "Cuando el experimento termine, el gráfico y la tabla van a aparecer acá."));
}

function buildLab(ctx) {
  const { S, rpc, run } = ctx;
  const V = { kind: "dt", param: "max_depth", metric: "macro_f1", seeds: 2, sweepDone: false, robustDone: false };
  const sweep = () => SWEEPS[V.kind][V.param];

  /* ---- un parámetro a la vez ---- */
  const sweepOut = h("div", { class: "tl-lab-out" });
  const showExpect = () => { if (!V.sweepDone) clear(sweepOut).append(expectBox(sweep().expect)); };
  const chips = chipPicker({ values: sweep().values, label: "Valores a probar" });
  const paramHost = h("div", {});
  function fillParams() {
    const sel = selectBox(Object.entries(SWEEPS[V.kind]).map(([k, p]) => [k, p.label]), V.param,
      (v) => { V.param = v; chips.reset(sweep().values); V.sweepDone = false; showExpect(); }, { "aria-label": "Parámetro" });
    clear(paramHost).append(sel);
    chips.reset(sweep().values);
  }
  const kindSeg = segmented([["dt", "Árbol propio"], ["rf", "Random Forest"]], V.kind,
    (k) => { V.kind = k; V.param = Object.keys(SWEEPS[k])[0]; fillParams(); V.sweepDone = false; showExpect(); });
  fillParams();
  const metricHelp = h("p", { class: "tl-help" }, METRIC_HELP[V.metric]);
  const metricSel = selectBox(Object.entries(METRIC_LABELS), V.metric, (v) => { V.metric = v; metricHelp.textContent = METRIC_HELP[v]; }, { "aria-label": "Métrica" });

  const runSweep = button("Probar valores", () => run("Probando los valores; puede tardar unos segundos", async () => {
    runSweep.disabled = true;
    try {
      const list = chips.get();
      if (list.length < 2) throw new Error("Elegí al menos dos valores para poder compararlos.");
      const r = await rpc("sweep", { kind: V.kind, param: V.param, values: list, base_params: V.kind === "dt" ? S.params : {}, metric: V.metric, n_splits: 5, n_repeats: 2 });
      V.sweepDone = true;
      drawSweep(r);
    } finally { runSweep.disabled = false; }
  }).catch(() => {}), "tl-btn-primary");

  function drawSweep(r) {
    const labels = r.rows.map((row) => valueLabel(row.value)), color = r.kind === "dt" ? "var(--dt)" : "var(--rf)", name = METRIC_LABELS[r.metric];
    const best = r.rows.findIndex((row) => row.value === r.best), title = SWEEPS[r.kind][r.param].label;
    clear(sweepOut).append(
      h("div", { class: "tl-two" },
        h("div", {}, h("h3", { class: "tl-h3" }, `${name} según ${title.toLowerCase()}`),
          lineChart({ labels, markIndex: best, ariaLabel: `${name} según ${title}`, series: [
            { name: `${name} en prueba`, color, points: r.rows.map((row) => ({ y: row.test_mean, lo: row.test_mean - row.test_std, hi: row.test_mean + row.test_std })) },
            { name: `${name} en entrenamiento`, color: "var(--muted)", dash: "6 5", points: r.rows.map((row) => ({ y: row.train_mean })) }] }),
          h("div", { class: "tl-legend" }, h("span", {}, h("i", { style: { background: color } }), `${name} con datos de prueba (la banda muestra ±1 desvío estándar)`),
            h("span", {}, h("i", { style: { background: "var(--muted)" } }), "con datos de entrenamiento"))),
        h("div", {}, h("p", { class: "tl-note" }, `El mejor resultado con los datos de prueba se obtiene con el valor ${valueLabel(r.best)}, marcado con un círculo en el gráfico. Si las bandas de dos valores se superponen, la diferencia entre ellos no es clara; en ese caso, conviene elegir el modelo más simple.`),
          h("p", { style: { marginTop: "10px" } }, h("b", {}, "Qué se esperaba ver. "), SWEEPS[r.kind][r.param].expect),
          h("p", { class: "tl-muted", style: { marginTop: "8px" } }, "Cuando la línea de entrenamiento queda muy por encima de la de prueba, el modelo está memorizando los datos. La distancia entre las dos líneas muestra el sobreajuste."))),
      h("div", { class: "tl-table-wrap", style: { marginTop: "12px" } }, h("table", { class: "tl-table" },
        h("thead", {}, h("tr", {}, h("th", {}, title), h("th", {}, `${name} en prueba`), h("th", {}, "Desvío"), h("th", {}, "En entrenamiento"), h("th", {}, "Brecha"), h("th", {}, "Tiempo"))),
        h("tbody", {}, r.rows.map((row) => h("tr", {}, h("td", {}, valueLabel(row.value)), h("td", {}, fix(row.test_mean)), h("td", {}, fix(row.test_std)),
          h("td", {}, fix(row.train_mean)), h("td", {}, fix(row.gap)), h("td", {}, secs(row.fit_time))))))),
      h("p", { class: "tl-muted tl-small", style: { marginTop: "6px" } }, `Cada valor se evaluó en ${r.n_partitions} particiones. El resto de los parámetros del árbol son los elegidos en la pestaña Árbol${r.param === "confidence" ? ", con la poda activada" : ""}.`));
  }

  /* ---- robustez a faltantes ---- */
  const fractions = chipPicker({ values: FRACTIONS, selected: FRACTIONS.slice(0, 5), format: (v) => pct(v, 0), allowCustom: false, label: "Proporción de datos a borrar" });
  const seeds = slider({ label: "Repeticiones del borrado", min: 1, max: 5, step: 1, value: V.seeds, format: (v) => num(v, 0), onInput: (v) => { V.seeds = v; },
    help: "Cantidad de veces que se repite el borrado, eligiendo otros datos al azar cada vez. Los resultados se promedian." });
  const ROBUST_EXPECT = "Lo esperable es que los dos modelos empeoren a medida que se borran más datos. La diferencia está en cómo manejan los datos que faltan: el árbol propio reparte cada caso incompleto entre las ramas, como hace C4.5, mientras que el bosque completa los datos faltantes antes de clasificar.";
  const robustOut = h("div", { class: "tl-lab-out" }, expectBox(ROBUST_EXPECT));
  const runRobust = button("Medir robustez", () => run("Borrando datos y evaluando; puede tardar unos segundos", async () => {
    runRobust.disabled = true;
    try {
      const list = fractions.get();
      if (list.length < 2) throw new Error("Elegí al menos dos proporciones para poder compararlas.");
      const r = await rpc("robustness", { fractions: list, seeds: Array.from({ length: V.seeds }, (_, i) => i), n_splits: 5, n_repeats: 1, dt_params: S.params, rf_params: {}, metric: V.metric });
      drawRobust(r);
    } finally { runRobust.disabled = false; }
  }).catch(() => {}), "tl-btn-primary");

  function drawRobust(r) {
    const name = METRIC_LABELS[r.metric], first = r.rows[0], last = r.rows[r.rows.length - 1];
    const labels = r.rows.map((row) => pct(row.fraction, 0));
    clear(robustOut).append(
      h("div", { class: "tl-two" },
        h("div", {}, h("h3", { class: "tl-h3" }, `${name} según la proporción de datos borrados`),
          lineChart({ labels, xs: r.rows.map((row) => row.fraction), ariaLabel: `${name} según la proporción de datos borrados`, series: [
            { name: "Árbol propio", color: "var(--dt)", points: r.rows.map((row) => ({ y: row.DT, lo: row.DT - row.DT_std, hi: row.DT + row.DT_std })) },
            { name: "Random Forest", color: "var(--rf)", points: r.rows.map((row) => ({ y: row.RF, lo: row.RF - row.RF_std, hi: row.RF + row.RF_std })) }] }),
          h("div", { class: "tl-legend" }, h("span", {}, h("i", { style: { background: "var(--dt)" } }), "Árbol propio"), h("span", {}, h("i", { style: { background: "var(--rf)" } }), "Random Forest"))),
        h("div", {}, h("p", { class: "tl-note" }, `Al pasar de borrar el ${pct(first.fraction, 0)} a borrar el ${pct(last.fraction, 0)} de los datos, el ${name} del árbol pasa de ${fix(first.DT, 3)} a ${fix(last.DT, 3)}, y el del bosque, de ${fix(first.RF, 3)} a ${fix(last.RF, 3)}.`),
          h("p", { class: "tl-muted", style: { marginTop: "8px" } }, "Cuando falta un dato, el árbol propio reparte el caso entre las ramas, como hace C4.5. El bosque, en cambio, completa antes el dato con la mediana o el valor más frecuente del entrenamiento. Los valores se borran al azar, tanto en los datos de entrenamiento como en los de prueba."))),
      h("p", { class: "tl-muted tl-small", style: { marginTop: "6px" } }, `Cada proporción se evaluó en ${r.n_partitions} particiones, con ${r.seeds.length} ${r.seeds.length === 1 ? "borrado distinto" : "borrados distintos"}. Las bandas muestran ±1 desvío estándar.`));
  }

  showExpect();
  return { el: h("div", {},
    h("h2", { class: "tl-h2" }, "Experimentos"),
    h("p", { class: "tl-muted", style: { maxWidth: "75ch" } }, "En esta pestaña se puede estudiar cómo influye cada parámetro en el resultado y qué tan bien tolera cada modelo los datos faltantes. Cada punto de los gráficos es el promedio de varias particiones de prueba."),

    h("section", { class: "tl-section" },
      h("h3", { class: "tl-h3" }, "¿Cómo influye un parámetro?"),
      h("p", { class: "tl-muted tl-small", style: { marginBottom: "10px" } }, "Se entrena el modelo con cada uno de los valores elegidos y se compara el resultado con los datos de entrenamiento y con los de prueba."),
      h("div", { class: "tl-card" },
        h("div", { class: "tl-groups" },
          h("fieldset", { class: "tl-group" }, h("legend", {}, "Qué se prueba"), field("Modelo", kindSeg.el), field("Parámetro", paramHost)),
          h("fieldset", { class: "tl-group" }, h("legend", {}, "Valores a probar"), chips.el,
            h("p", { class: "tl-help" }, "Tocá un valor para incluirlo o quitarlo. También podés agregar otro valor; usá punto para los decimales.")),
          h("fieldset", { class: "tl-group" }, h("legend", {}, "Cómo se mide"), field("Métrica", metricSel), metricHelp)),
        h("div", { class: "tl-actions tl-actions-split" },
          h("p", { class: "tl-muted tl-small" }, "Cada valor se evalúa con validación cruzada de 5 pliegues, repetida 2 veces."), runSweep)),
      sweepOut),

    h("section", { class: "tl-section" },
      h("h3", { class: "tl-h3" }, "¿Qué pasa cuando faltan datos?"),
      h("p", { class: "tl-muted tl-small", style: { marginBottom: "10px" } }, "Se borra al azar una parte de los datos y se mide cómo cambia el resultado de cada modelo. Así se puede ver cuál de los dos tolera mejor los datos faltantes."),
      h("div", { class: "tl-card" },
        h("div", { class: "tl-groups" },
          h("fieldset", { class: "tl-group" }, h("legend", {}, "Datos a borrar"), fractions.el,
            h("p", { class: "tl-help" }, "Cada porcentaje indica qué parte de los valores se borra. Tocá un porcentaje para incluirlo o quitarlo.")),
          h("fieldset", { class: "tl-group" }, h("legend", {}, "Repeticiones"), seeds.el)),
        h("div", { class: "tl-actions tl-actions-split" },
          h("p", { class: "tl-muted tl-small" }, "Se usa la métrica elegida en el experimento anterior y los parámetros del árbol de la pestaña Árbol."), runRobust)),
      robustOut)) };
}

/* ============================ armazón de la aplicación ============================ */
const TABS = [["datos", "Datos"], ["arbol", "Árbol"], ["comparar", "Comparar"], ["lab", "Experimentos"]];
const BUILDERS = { datos: buildDatos, arbol: buildArbol, comparar: buildComparar, lab: buildLab };

function initialTheme() {
  try { const saved = localStorage.getItem("tl-theme"); if (saved) return saved; } catch (e) { /* sin almacenamiento */ }
  return window.matchMedia && window.matchMedia("(prefers-color-scheme: dark)").matches ? "dark" : "light";
}

function createApp(root, rpc) {
  const S = { datasets: [], info: null, tab: "datos", views: {}, params: { ...DEFAULT_PARAMS }, busy: 0 };
  const status = h("span", { class: "tl-muted tl-small", role: "status", "aria-live": "polite" });
  const errorBox = h("div", { role: "alert", hidden: true });
  const datasetSelect = h("select", { class: "tl-select", "aria-label": "Dataset" });
  const tabBar = h("div", { class: "tl-tabs", role: "tablist" });
  const panel = h("div", { class: "tl-panel", role: "tabpanel" });

  const themeButton = button("", () => {
    const next = root.dataset.theme === "dark" ? "light" : "dark";
    root.dataset.theme = next;
    try { localStorage.setItem("tl-theme", next); } catch (e) { /* sin almacenamiento */ }
    themeButton.textContent = next === "dark" ? "Modo claro" : "Modo oscuro";
  }, "tl-btn-quiet");
  themeButton.textContent = root.dataset.theme === "dark" ? "Modo claro" : "Modo oscuro";

  root.append(
    h("header", { class: "tl-head" },
      h("h1", { class: "tl-title" }, "C4.5 ", h("span", { class: "tl-muted" }, "vs."), " Random Forest"),
      h("div", { class: "tl-head-tools" }, status,
        h("label", { class: "tl-dataset-pick" }, h("span", {}, "Dataset"), datasetSelect), themeButton)),
    tabBar, errorBox, panel);

  function setStatus(text, spinning) { clear(status); if (spinning) status.append(h("span", { class: "tl-spin" })); if (text) status.append(text); }
  function showError(message) {
    errorBox.hidden = !message;
    clear(errorBox);
    if (message) { errorBox.className = "tl-error"; errorBox.append(h("b", {}, "No se pudo completar: "), message); }
  }
  async function run(label, action) {
    S.busy += 1;
    setStatus(label, true);
    showError("");
    try { return await action(); }
    catch (error) { showError(error.message); throw error; }
    finally { S.busy -= 1; if (!S.busy) setStatus(""); }
  }

  const ctx = {
    S, rpc, run, showError,
    goto: (tab) => showTab(tab),
    selectDataset: (key) => run("Cargando datos", async () => {
      S.info = await rpc("select", { key });
      S.views = {};
      S.params = { ...DEFAULT_PARAMS };
      datasetSelect.value = key;
      showTab(S.tab);
    }).catch(() => {}),
    loadCsv: (text, target, name) => run("Leyendo el archivo", async () => {
      S.info = await rpc("load_csv", { text, target, name });
      S.datasets = (await rpc("datasets")).datasets;
      S.views = {};
      refreshDatasetSelect();
      showTab("datos");
    }).catch(() => {}),
  };

  function refreshDatasetSelect() {
    clear(datasetSelect).append(...S.datasets.map((d) => h("option", { value: d.key, selected: S.info && d.key === S.info.key }, d.title)));
  }
  datasetSelect.addEventListener("change", () => ctx.selectDataset(datasetSelect.value));

  function showTab(tab) {
    S.tab = tab;
    for (const b of tabBar.children) b.setAttribute("aria-selected", String(b.dataset.tab === tab));
    if (!S.info) { clear(panel).append(h("div", { class: "tl-empty" }, "Elegí un dataset para empezar.")); return; }
    if (!S.views[tab]) S.views[tab] = BUILDERS[tab](ctx);
    clear(panel).append(S.views[tab].el);
    if (S.views[tab].onShow) S.views[tab].onShow();
  }

  TABS.forEach(([key, text], i) => {
    tabBar.append(h("button", { type: "button", class: "tl-tab", role: "tab", "data-tab": key, "aria-selected": "false", onclick: () => showTab(key) },
      h("span", { class: "tl-step", "aria-hidden": "true" }, String(i + 1)), text));
  });

  return {
    async start() {
      await run("Iniciando", async () => {
        S.datasets = (await rpc("datasets")).datasets;
        refreshDatasetSelect();
        const first = S.datasets[0];   // se abre con el primero de la lista (default_entries en service.py)
        S.info = await rpc("select", { key: first.key });
        datasetSelect.value = first.key;
        showTab("datos");
      }).catch(() => {});
    },
  };
}

/* ============================ punto de entrada de anywidget ============================ */
function render({ model, el }) {
  const rpc = createRpc(model);
  const root = h("div", { class: "tl", "data-theme": initialTheme() });
  el.append(root);
  createApp(root, rpc).start();
  return () => { rpc.dispose(); root.remove(); };
}

export default { render };


In [ ]:
%%writefile src/tp_ia/web/static/app.css
@import url("https://fonts.googleapis.com/css2?family=Bricolage+Grotesque:opsz,wdth,wght@12..96,75..100,200..800&display=swap");

/* Laboratorio de árboles de decisión: estilos. Todo vive bajo .tl para no tocar nada del notebook que lo aloja.
   Paleta: gris azulado claro de fondo, tinta azul noche, árbol propio en azul acero y bosque en ocre.
   Las clases usan la paleta Okabe-Ito (apta para daltónicos). Una sola familia tipográfica, con peso y ancho variables. */

.tl {
  --bg: #EDF1F3; --surface: #FFFFFF; --surface-2: #F5F8F9; --ink: #13222D; --muted: #5B6B77; --line: #D5DDE2;
  --dt: #1B4965; --rf: #B7791F; --accent: #1B4965; --accent-soft: #DCE9F1; --warn: #B4442C; --ok: #1C7C54; --caution: #8A5A00;
  --c0: #0072B2; --c1: #E69F00; --c2: #009E73; --c3: #CC79A7; --c4: #56B4E9; --c5: #D55E00; --c6: #7A7A7A;
  --radius: 10px; --shadow: 0 1px 2px rgba(19, 34, 45, .06), 0 10px 28px -14px rgba(19, 34, 45, .22);
  box-sizing: border-box; background: var(--bg); color: var(--ink); border-radius: 16px; padding: 20px;
  font-family: "Bricolage Grotesque", system-ui, -apple-system, "Segoe UI", Roboto, sans-serif;
  font-size: 14.5px; line-height: 1.45; font-variant-numeric: tabular-nums; min-width: 320px; position: relative;
}
.tl[data-theme="dark"] {
  --bg: #0E171D; --surface: #15222B; --surface-2: #1A2A35; --ink: #E7EEF2; --muted: #95A6B1; --line: #293A46;
  --dt: #7FB6D6; --rf: #E2B057; --accent: #7FB6D6; --accent-soft: #1F3646; --warn: #E8806A; --ok: #5CC592; --caution: #E3B341;
  --shadow: 0 1px 2px rgba(0, 0, 0, .4), 0 10px 28px -14px rgba(0, 0, 0, .6);
}
.tl *, .tl *::before, .tl *::after { box-sizing: border-box; }
.tl h1, .tl h2, .tl h3, .tl p { margin: 0; }
.tl button, .tl select, .tl input { font: inherit; color: inherit; }
.tl :focus-visible { outline: 2px solid var(--accent); outline-offset: 2px; border-radius: 6px; }
.tl [hidden] { display: none !important; }
.tl .tl-muted { color: var(--muted); }
.tl .tl-small { font-size: 12.5px; }

/* ---------- encabezado y pestañas ---------- */
.tl .tl-head { display: flex; flex-wrap: wrap; gap: 10px 24px; align-items: center; justify-content: space-between; }
.tl .tl-title { font-size: 21px; line-height: 1.1; font-weight: 700; font-stretch: 85%; letter-spacing: -0.01em; }
.tl .tl-head-tools { display: flex; flex-wrap: wrap; gap: 10px; align-items: center; }
.tl .tl-dataset-pick { display: inline-flex; gap: 8px; align-items: center; font-weight: 580; font-size: 13px; color: var(--muted); }
.tl .tl-dataset-pick .tl-select { color: var(--ink); font-weight: 560; font-size: 14px; }
.tl .tl-tabs { display: flex; gap: 4px; margin: 14px 0 20px; border-bottom: 1px solid var(--line); overflow-x: auto; overflow-y: hidden; }
.tl .tl-tab { display: inline-flex; align-items: center; gap: 8px; background: none; border: 0; padding: 10px 14px; cursor: pointer;
  color: var(--muted); font-weight: 560; border-bottom: 3px solid transparent; margin-bottom: -1px; white-space: nowrap; }
.tl .tl-tab:hover { color: var(--ink); }
.tl .tl-tab[aria-selected="true"] { color: var(--ink); border-bottom-color: var(--accent); }
.tl .tl-step { display: inline-grid; place-items: center; width: 20px; height: 20px; border-radius: 50%; font-size: 11.5px; font-weight: 680;
  border: 1.5px solid currentColor; line-height: 1; }
.tl .tl-tab[aria-selected="true"] .tl-step { background: var(--accent); border-color: var(--accent); color: var(--surface); }
.tl .tl-panel { min-height: 320px; }

/* ---------- controles ---------- */
.tl .tl-btn { background: var(--surface); border: 1px solid var(--line); border-radius: 8px; padding: 7px 13px; cursor: pointer;
  font-weight: 560; transition: background .15s, border-color .15s; }
.tl .tl-btn:hover { border-color: var(--accent); }
.tl .tl-btn:disabled { opacity: .5; cursor: progress; }
.tl .tl-btn-primary { background: var(--accent); border-color: var(--accent); color: var(--surface); }
.tl[data-theme="dark"] .tl-btn-primary { color: #0E171D; }
.tl .tl-btn-quiet { background: none; border-color: transparent; color: var(--muted); }
.tl .tl-btn-link { background: none; border: 0; padding: 0; color: var(--accent); font-size: 13px; text-decoration: underline;
  text-underline-offset: 3px; align-self: flex-start; }
.tl label.tl-btn { display: inline-flex; align-items: center; position: relative; }
.tl .tl-file-input { position: absolute; width: 1px; height: 1px; opacity: 0; overflow: hidden; }
.tl label.tl-btn:focus-within { outline: 2px solid var(--accent); outline-offset: 2px; }
.tl .tl-select, .tl .tl-input { background: var(--surface); border: 1px solid var(--line); border-radius: 8px; padding: 6px 10px; min-width: 0; }
.tl .tl-input { width: 100%; }
.tl .tl-seg { display: inline-flex; border: 1px solid var(--line); border-radius: 8px; overflow: hidden; background: var(--surface); }
.tl .tl-seg button { background: none; border: 0; padding: 6px 11px; cursor: pointer; color: var(--muted); font-size: 13px; }
.tl .tl-seg button[aria-pressed="true"] { background: var(--accent); color: var(--surface); }
.tl[data-theme="dark"] .tl-seg button[aria-pressed="true"] { color: #0E171D; }
.tl input[type="range"] { -webkit-appearance: none; appearance: none; width: 100%; height: 22px; background: none; margin: 0; cursor: pointer; }
.tl input[type="range"]::-webkit-slider-runnable-track { height: 4px; background: var(--line); border-radius: 4px; }
.tl input[type="range"]::-moz-range-track { height: 4px; background: var(--line); border-radius: 4px; }
.tl input[type="range"]::-webkit-slider-thumb { -webkit-appearance: none; width: 16px; height: 16px; margin-top: -6px; border-radius: 50%;
  background: var(--accent); border: 2px solid var(--surface); box-shadow: 0 1px 4px rgba(0, 0, 0, .3); }
.tl input[type="range"]::-moz-range-thumb { width: 14px; height: 14px; border-radius: 50%; background: var(--accent); border: 2px solid var(--surface); }
.tl input[type="range"]:disabled { opacity: .4; cursor: not-allowed; }
.tl .tl-check { display: inline-flex; gap: 8px; align-items: center; cursor: pointer; user-select: none; }
.tl .tl-check input { accent-color: var(--accent); width: 16px; height: 16px; margin: 0; }

/* ---------- bloques ---------- */
.tl .tl-card { background: var(--surface); border-radius: var(--radius); padding: 16px; box-shadow: var(--shadow); }
.tl .tl-h2 { font-size: 19px; font-weight: 680; letter-spacing: -0.01em; margin-bottom: 4px; }
.tl .tl-h3 { font-size: 14px; font-weight: 640; margin-bottom: 8px; }
.tl .tl-section { margin-top: 22px; }
.tl .tl-row { display: flex; flex-wrap: wrap; gap: 10px 18px; align-items: center; }
.tl .tl-note { background: var(--accent-soft); border-radius: 8px; padding: 10px 12px; color: var(--ink); }
/* Aviso informativo (ámbar): no es un error, es algo a tener en cuenta. Los errores usan .tl-error (rojo). */
.tl .tl-warn { background: color-mix(in srgb, var(--caution) 11%, var(--surface)); border-left: 3px solid var(--caution); padding: 9px 12px; border-radius: 6px; }
.tl .tl-warn::before { content: "Aviso · "; font-weight: 640; color: var(--caution); }
.tl .tl-error { background: color-mix(in srgb, var(--warn) 16%, var(--surface)); border-left: 3px solid var(--warn); padding: 10px 14px;
  border-radius: 6px; margin-bottom: 12px; }
.tl .tl-empty { color: var(--muted); padding: 36px 8px; text-align: center; }
.tl .tl-busy::after { content: ""; position: absolute; inset: 0; background: color-mix(in srgb, var(--bg) 55%, transparent);
  border-radius: 16px; pointer-events: none; animation: tl-pulse 1.1s ease-in-out infinite; }
@keyframes tl-pulse { 50% { opacity: .45; } }
.tl .tl-spin { display: inline-block; width: 13px; height: 13px; border: 2px solid var(--line); border-top-color: var(--accent);
  border-radius: 50%; animation: tl-rot .8s linear infinite; vertical-align: -2px; margin-right: 7px; }
@keyframes tl-rot { to { transform: rotate(360deg); } }

/* ---------- datos ---------- */
.tl .tl-facts { display: flex; flex-wrap: wrap; gap: 8px 30px; margin: 4px 0 14px; }
.tl .tl-fact b { display: block; font-size: 24px; font-weight: 660; line-height: 1.1; }
.tl .tl-fact span { color: var(--muted); font-size: 12.5px; }
.tl .tl-classbar { display: flex; height: 22px; border-radius: 6px; overflow: hidden; gap: 2px; }
.tl .tl-classbar div { min-width: 3px; }
.tl .tl-legend { display: flex; flex-wrap: wrap; gap: 6px 16px; margin-top: 8px; font-size: 13px; }
.tl .tl-legend i { display: inline-block; width: 10px; height: 10px; border-radius: 3px; margin-right: 6px; }
.tl .tl-features { display: grid; grid-template-columns: repeat(auto-fill, minmax(280px, 1fr)); gap: 10px 22px; }
.tl .tl-feature { padding: 8px 0; border-top: 1px solid var(--line); }
/* Si el nombre es largo, las etiquetas bajan a la línea siguiente en lugar de aplastar el nombre. */
.tl .tl-feature header { display: flex; flex-wrap: wrap; justify-content: space-between; gap: 4px 8px; align-items: baseline; }
.tl .tl-feature b { font-weight: 620; overflow-wrap: anywhere; flex: 1 0 auto; max-width: 100%; }
.tl .tl-feature-desc { color: var(--muted); font-size: 12.5px; margin: 2px 0 8px; }
.tl .tl-fill-split { display: flex; gap: 1px; background: none; }
.tl .tl-fill-split > div { min-width: 2px; }
.tl .tl-tag { font-size: 11.5px; padding: 1px 7px; border-radius: 99px; background: var(--accent-soft); color: var(--accent); white-space: nowrap; }
.tl .tl-tag-warn { background: color-mix(in srgb, var(--caution) 15%, var(--surface)); color: var(--caution); }
.tl .tl-decisions details { border-top: 1px solid var(--line); padding: 8px 0; }
.tl .tl-decisions summary { cursor: pointer; font-weight: 580; }
.tl .tl-decisions details > div, .tl details.tl-decisions > div { margin: 8px 0 4px 16px; color: var(--muted); display: grid; gap: 6px; max-width: 95ch; }
.tl .tl-decisions b { color: var(--ink); font-weight: 620; }
.tl .tl-evidence { margin: 0; padding-left: 18px; font-size: 13px; }
.tl details.tl-decisions summary { cursor: pointer; font-weight: 580; }
.tl .tl-table-wrap { overflow-x: auto; }
.tl .tl-table { border-collapse: collapse; width: 100%; font-size: 13.5px; }
.tl .tl-table th, .tl .tl-table td { padding: 6px 10px; text-align: right; border-bottom: 1px solid var(--line); white-space: nowrap; }
.tl .tl-table th:first-child, .tl .tl-table td:first-child { text-align: left; }
.tl .tl-table th { color: var(--muted); font-weight: 560; }

/* ---------- árbol ---------- */
.tl .tl-arbol { display: grid; grid-template-columns: 260px minmax(0, 1fr); gap: 20px; align-items: start; }
.tl .tl-rail { position: sticky; top: 8px; display: flex; flex-direction: column; gap: 14px; }
.tl .tl-rail-head { display: flex; justify-content: space-between; align-items: baseline; gap: 8px; }
.tl .tl-rail-group { display: flex; flex-direction: column; gap: 12px; padding-top: 14px; border-top: 1px solid var(--line); }
.tl .tl-advanced { border-top: 1px solid var(--line); padding-top: 12px; }
.tl .tl-advanced summary { cursor: pointer; font-weight: 600; display: flex; align-items: center; gap: 8px; list-style-position: inside; }
.tl .tl-advanced summary::-webkit-details-marker { display: none; }
.tl .tl-advanced summary::before { content: "▸"; color: var(--muted); transition: transform .15s; }
.tl .tl-advanced[open] summary::before { transform: rotate(90deg); }
.tl .tl-advanced-body { display: flex; flex-direction: column; gap: 14px; margin-top: 14px; }
.tl .tl-param label.tl-lab { display: flex; justify-content: space-between; gap: 8px; font-weight: 580; margin-bottom: 2px; }
.tl .tl-param output { color: var(--accent); font-weight: 640; }
.tl .tl-help { color: var(--muted); font-size: 12.5px; margin-top: 2px; }
.tl .tl-stats { display: flex; flex-wrap: wrap; gap: 10px 26px; margin-bottom: 12px; }
.tl .tl-stat b { display: block; font-size: 22px; font-weight: 660; line-height: 1.1; }
.tl .tl-stat span { color: var(--muted); font-size: 12.5px; }
.tl .tl-treewrap { background: var(--surface); border-radius: var(--radius); box-shadow: var(--shadow); overflow: auto; max-height: 640px; position: relative; }
.tl .tl-treetools { position: absolute; right: 10px; top: 8px; display: flex; gap: 6px; z-index: 2; }
.tl .tl-tree text { font-family: inherit; fill: var(--ink); }
.tl .tl-tree .tl-node { cursor: pointer; }
.tl .tl-tree .tl-node rect.tl-box { fill: var(--surface-2); stroke: var(--line); stroke-width: 1.2; transition: stroke .15s; }
.tl .tl-tree .tl-node:hover rect.tl-box, .tl .tl-tree .tl-node.is-pinned rect.tl-box { stroke: var(--accent); stroke-width: 2; }
.tl .tl-tree .tl-node.is-path rect.tl-box { stroke: var(--ink); stroke-width: 2.5; }
.tl .tl-tree .tl-leaf-label { font-weight: 640; font-size: 12.5px; }
.tl .tl-tree .tl-feat { font-weight: 640; font-size: 12.5px; }
.tl .tl-tree .tl-n { font-size: 11px; fill: var(--muted); }
.tl .tl-tree .tl-edge { font-size: 11px; fill: var(--muted); paint-order: stroke; stroke: var(--surface); stroke-width: 4px; }
.tl .tl-tree .tl-link { fill: none; stroke-linecap: butt; }
.tl .tl-tree.has-path .tl-link { opacity: .25; }
.tl .tl-tree.has-path .tl-link.is-path { opacity: 1; }
.tl .tl-tree.has-path .tl-node:not(.is-path) { opacity: .45; }
/* Debajo del árbol: fila 1 = nodo (angosto) + predecir (ancho, destacado); fila 2 = importancias + matriz. */
.tl .tl-lower { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 2fr); gap: 16px; margin-top: 16px; align-items: start; }
.tl .tl-lower-even { grid-template-columns: minmax(0, 1fr) minmax(0, 1fr); }
.tl .tl-card-accent { box-shadow: var(--shadow), inset 0 3px 0 var(--accent); }
.tl .tl-card-head { display: flex; justify-content: space-between; align-items: baseline; gap: 8px; }
.tl .tl-predict { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 1fr); gap: 22px; align-items: start; }
.tl .tl-predict-result .tl-empty { padding: 24px 8px; }
.tl .tl-source { margin-top: 16px; }
.tl .tl-source summary { cursor: pointer; font-size: 13px; color: var(--muted); }
.tl .tl-inspector dl { display: grid; grid-template-columns: auto 1fr; gap: 3px 14px; margin: 0; }
.tl .tl-inspector dt { color: var(--muted); }
.tl .tl-inspector dd { margin: 0; font-weight: 580; text-align: right; }
.tl .tl-bars { display: grid; gap: 6px; }
.tl .tl-bar { display: grid; grid-template-columns: minmax(70px, 150px) 1fr 46px; gap: 8px; align-items: center; font-size: 13px; }
.tl .tl-bar .tl-track { height: 9px; background: var(--surface-2); border-radius: 5px; overflow: hidden; }
.tl .tl-bar .tl-fill { height: 100%; border-radius: 5px; background: var(--accent); }
.tl .tl-bar span:last-child { text-align: right; color: var(--muted); }
.tl .tl-case { display: grid; gap: 10px; max-height: 360px; overflow: auto; padding-right: 6px; }
.tl .tl-case-row { display: grid; grid-template-columns: 1fr; gap: 2px; }
.tl .tl-case-row .tl-lab { display: flex; justify-content: space-between; gap: 8px; font-weight: 560; font-size: 13px; }
.tl .tl-case-row .tl-lab output { color: var(--accent); font-weight: 640; }
.tl .tl-verdict { display: flex; justify-content: space-between; align-items: baseline; gap: 8px; margin: 10px 0 4px; }
.tl .tl-verdict b { font-size: 18px; font-weight: 680; }
.tl .tl-pbar { display: flex; height: 12px; border-radius: 6px; overflow: hidden; gap: 2px; }
.tl .tl-rules { margin: 8px 0 0; padding-left: 18px; color: var(--muted); font-size: 13px; }
.tl .tl-matrix { border-collapse: separate; border-spacing: 3px; font-size: 12.5px; }
.tl .tl-matrix td { text-align: center; padding: 6px 9px; border-radius: 5px; min-width: 38px; }
.tl .tl-matrix th { color: var(--muted); font-weight: 520; padding: 2px 6px; }
.tl pre.tl-code { background: var(--surface-2); border-radius: 8px; padding: 12px; overflow: auto; max-height: 300px; font-size: 12px; margin: 0;
  font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; }

/* ---------- comparar y laboratorio ---------- */
.tl .tl-controls { display: grid; grid-template-columns: repeat(auto-fit, minmax(190px, 1fr)); gap: 14px 22px; align-items: end; }
/* Controles agrupados por tema (pestaña Comparar): cada grupo es una columna con su título. */
.tl .tl-groups { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 18px 28px; align-items: start; }
.tl .tl-group { border: 0; margin: 0; padding: 0; min-width: 0; display: flex; flex-direction: column; gap: 12px; }
.tl .tl-group legend { padding: 0; margin-bottom: 10px; font-size: 12px; font-weight: 680; letter-spacing: .06em; text-transform: uppercase; color: var(--muted); }
.tl .tl-group-rf legend { color: var(--rf); }
.tl .tl-group-dt legend { color: var(--dt); }
.tl .tl-actions { display: flex; justify-content: flex-end; margin-top: 16px; padding-top: 14px; border-top: 1px solid var(--line); }
.tl .tl-actions-split { justify-content: space-between; align-items: center; gap: 12px; flex-wrap: wrap; }

/* Experimentos: valores que se activan con un clic, y el recuadro previo que cuenta qué se espera ver. */
.tl .tl-chips { display: flex; flex-wrap: wrap; gap: 6px; }
.tl .tl-chip { background: var(--surface); border: 1px solid var(--line); border-radius: 99px; padding: 4px 12px; cursor: pointer;
  font-size: 13px; color: var(--muted); transition: background .15s, border-color .15s, color .15s; }
.tl .tl-chip:hover { border-color: var(--accent); }
.tl .tl-chip[aria-pressed="true"] { background: var(--accent-soft); border-color: var(--accent); color: var(--accent); font-weight: 620; }
.tl .tl-chip-add { display: flex; gap: 6px; margin-top: 10px; }
.tl .tl-chip-input { width: 120px; flex: 0 0 auto; }
.tl .tl-lab-out { margin-top: 16px; }
.tl .tl-expect { border: 1.5px dashed var(--line); border-radius: var(--radius); padding: 16px 18px; max-width: 80ch; }
.tl .tl-expect-title { font-weight: 660; margin-bottom: 4px; }
.tl .tl-chart text { font-family: inherit; fill: var(--muted); font-size: 11.5px; }
.tl .tl-chart .tl-axis { stroke: var(--line); }
.tl .tl-chart .tl-grid { stroke: var(--line); stroke-dasharray: 2 4; }
.tl .tl-two { display: grid; grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); gap: 22px; }
.tl .tl-dtc { color: var(--dt); }
.tl .tl-rfc { color: var(--rf); }
.tl .tl-dot { display: inline-block; width: 10px; height: 10px; border-radius: 50%; margin-right: 6px; }

.tl .tl-arbol > * { min-width: 0; }
@media (max-width: 900px) {
  .tl { padding: 14px; }
  .tl .tl-arbol { grid-template-columns: minmax(0, 1fr); }
  .tl .tl-lower, .tl .tl-lower-even, .tl .tl-predict { grid-template-columns: minmax(0, 1fr); }
  .tl .tl-rail { position: static; }
  .tl .tl-title { font-size: 23px; }
}
@media (prefers-reduced-motion: reduce) {
  .tl *, .tl *::before, .tl *::after { animation: none !important; transition: none !important; }
}


In [ ]:
%%writefile src/tp_ia/web/devserver.py
"""
Servidor de desarrollo: muestra la interfaz en un navegador común, sin Jupyter ni Colab.

GUÍA DE LECTURA
---------------
La interfaz (static/app.js) habla con Python por mensajes {id, method, params}. En Colab los transporta el widget de
anywidget; acá los transporta un POST a /rpc. En los dos casos contesta LabService.handle, así que lo que se ve en
este servidor es lo que se ve en el notebook. Sirve para desarrollar la interfaz y para probarla con un navegador.

  python src/tp_ia/web/devserver.py --port 8765       (abrir http://127.0.0.1:8765)

Los archivos estáticos se leen en cada pedido: alcanza con recargar la página para ver un cambio.
Un solo usuario a la vez (un candado serializa los pedidos, igual que el kernel de un notebook).
"""
from __future__ import annotations

import argparse
import json
import sys
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

if __package__ in (None, ""):                         # ejecutado como script, no como módulo
    sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
    from tp_ia.web.service import LabService
else:
    from .service import LabService

STATIC = Path(__file__).resolve().parent / "static"
FILES = {"/": ("dev.html", "text/html; charset=utf-8"),
         "/app.js": ("app.js", "text/javascript; charset=utf-8"),
         "/app.css": ("app.css", "text/css; charset=utf-8")}


def make_handler(service: LabService):
    lock = threading.Lock()

    class Handler(BaseHTTPRequestHandler):
        def _send(self, status: int, body: bytes, content_type: str) -> None:
            self.send_response(status)
            self.send_header("Content-Type", content_type)
            self.send_header("Content-Length", str(len(body)))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            self.wfile.write(body)

        def do_GET(self):
            path = self.path.split("?")[0]
            if path not in FILES:
                self._send(404, b"No encontrado", "text/plain; charset=utf-8")
                return
            name, content_type = FILES[path]
            self._send(200, (STATIC / name).read_bytes(), content_type)

        def do_POST(self):
            if self.path != "/rpc":
                self._send(404, b"No encontrado", "text/plain; charset=utf-8")
                return
            size = int(self.headers.get("Content-Length") or 0)
            try:
                message = json.loads(self.rfile.read(size) or b"{}")
            except json.JSONDecodeError:
                message = None
            with lock:
                reply = service.handle(message)
            self._send(200, json.dumps(reply, allow_nan=False).encode("utf-8"), "application/json")

        def log_message(self, format, *args):          # sin ruido en la consola
            pass

    return Handler


def serve(port: int = 8765, host: str = "127.0.0.1", service: LabService | None = None) -> None:
    server = ThreadingHTTPServer((host, port), make_handler(service or LabService()))
    print(f"Laboratorio en http://{host}:{port}  (Ctrl+C para cortar)", flush=True)
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        pass
    finally:
        server.server_close()


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__.split("\n")[1])
    parser.add_argument("--port", type=int, default=8765)
    parser.add_argument("--host", default="127.0.0.1")
    args = parser.parse_args()
    serve(args.port, args.host)


In [ ]:
%%writefile src/tp_ia/web/static/dev.html
<!doctype html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Laboratorio de árboles de decisión (desarrollo)</title>
  <link rel="stylesheet" href="/app.css">
  <style>body { margin: 0; background: #EDF1F3; } #root { max-width: 1240px; margin: 0 auto; padding: 16px; }</style>
</head>
<body>
  <div id="root"></div>
  <script type="module">
    // Hace de "model" de anywidget: manda los pedidos al servidor de desarrollo y entrega las respuestas.
    import widget from "/app.js";
    const listeners = {};
    const model = {
      get: () => undefined, set() {}, save_changes() {},
      on(name, callback) { (listeners[name] ||= []).push(callback); },
      off(name, callback) { listeners[name] = (listeners[name] || []).filter((f) => f !== callback); },
      send(content) {
        fetch("/rpc", { method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify(content) })
          .then((response) => response.json())
          .then((reply) => (listeners["msg:custom"] || []).forEach((callback) => callback(reply, [])));
      },
    };
    widget.render({ model, el: document.getElementById("root") });
  </script>
</body>
</html>


### 1.5 Los datos del trabajo

Los dos archivos están dentro del cuaderno, así que no se descarga nada.

- `processed.cleveland.data`: los 303 pacientes de Cleveland (UCI Heart Disease).
- `iris.csv`: las 150 flores de Iris.

- **Enfermedad cardíaca** (`processed.cleveland.data`): es el archivo de los pacientes de la Cleveland Clinic del dataset
  *Heart Disease* del UCI Machine Learning Repository, con licencia CC BY 4.0. Cita: Janosi, A., Steinbrunn, W.,
  Pfisterer, M., & Detrano, R. (1989). *Heart Disease* [Dataset]. UCI Machine Learning Repository.
  https://doi.org/10.24432/C52P4X
- **Iris** (`iris.csv`): Fisher, R. A. (1936). The use of multiple measurements in taxonomic problems. *Annals of
  Eugenics, 7*(2), 179-188.
- **Play Tennis** y el ejemplo de Liu están escritos en el código (`fixtures.py`). **Wine** y **Breast Cancer
  Wisconsin** vienen incluidos en la librería scikit-learn.

In [ ]:
%%writefile data/uci_heart/processed.cleveland.data
63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0
56.0,1.0,2.0,120.0,236.0,0.0,0.0,178.0,0.0,0.8,1.0,0.0,3.0,0
62.0,0.0,4.0,140.0,268.0,0.0,2.0,160.0,0.0,3.6,3.0,2.0,3.0,3
57.0,0.0,4.0,120.0,354.0,0.0,0.0,163.0,1.0,0.6,1.0,0.0,3.0,0
63.0,1.0,4.0,130.0,254.0,0.0,2.0,147.0,0.0,1.4,2.0,1.0,7.0,2
53.0,1.0,4.0,140.0,203.0,1.0,2.0,155.0,1.0,3.1,3.0,0.0,7.0,1
57.0,1.0,4.0,140.0,192.0,0.0,0.0,148.0,0.0,0.4,2.0,0.0,6.0,0
56.0,0.0,2.0,140.0,294.0,0.0,2.0,153.0,0.0,1.3,2.0,0.0,3.0,0
56.0,1.0,3.0,130.0,256.0,1.0,2.0,142.0,1.0,0.6,2.0,1.0,6.0,2
44.0,1.0,2.0,120.0,263.0,0.0,0.0,173.0,0.0,0.0,1.0,0.0,7.0,0
52.0,1.0,3.0,172.0,199.0,1.0,0.0,162.0,0.0,0.5,1.0,0.0,7.0,0
57.0,1.0,3.0,150.0,168.0,0.0,0.0,174.0,0.0,1.6,1.0,0.0,3.0,0
48.0,1.0,2.0,110.0,229.0,0.0,0.0,168.0,0.0,1.0,3.0,0.0,7.0,1
54.0,1.0,4.0,140.0,239.0,0.0,0.0,160.0,0.0,1.2,1.0,0.0,3.0,0
48.0,0.0,3.0,130.0,275.0,0.0,0.0,139.0,0.0,0.2,1.0,0.0,3.0,0
49.0,1.0,2.0,130.0,266.0,0.0,0.0,171.0,0.0,0.6,1.0,0.0,3.0,0
64.0,1.0,1.0,110.0,211.0,0.0,2.0,144.0,1.0,1.8,2.0,0.0,3.0,0
58.0,0.0,1.0,150.0,283.0,1.0,2.0,162.0,0.0,1.0,1.0,0.0,3.0,0
58.0,1.0,2.0,120.0,284.0,0.0,2.0,160.0,0.0,1.8,2.0,0.0,3.0,1
58.0,1.0,3.0,132.0,224.0,0.0,2.0,173.0,0.0,3.2,1.0,2.0,7.0,3
60.0,1.0,4.0,130.0,206.0,0.0,2.0,132.0,1.0,2.4,2.0,2.0,7.0,4
50.0,0.0,3.0,120.0,219.0,0.0,0.0,158.0,0.0,1.6,2.0,0.0,3.0,0
58.0,0.0,3.0,120.0,340.0,0.0,0.0,172.0,0.0,0.0,1.0,0.0,3.0,0
66.0,0.0,1.0,150.0,226.0,0.0,0.0,114.0,0.0,2.6,3.0,0.0,3.0,0
43.0,1.0,4.0,150.0,247.0,0.0,0.0,171.0,0.0,1.5,1.0,0.0,3.0,0
40.0,1.0,4.0,110.0,167.0,0.0,2.0,114.0,1.0,2.0,2.0,0.0,7.0,3
69.0,0.0,1.0,140.0,239.0,0.0,0.0,151.0,0.0,1.8,1.0,2.0,3.0,0
60.0,1.0,4.0,117.0,230.0,1.0,0.0,160.0,1.0,1.4,1.0,2.0,7.0,2
64.0,1.0,3.0,140.0,335.0,0.0,0.0,158.0,0.0,0.0,1.0,0.0,3.0,1
59.0,1.0,4.0,135.0,234.0,0.0,0.0,161.0,0.0,0.5,2.0,0.0,7.0,0
44.0,1.0,3.0,130.0,233.0,0.0,0.0,179.0,1.0,0.4,1.0,0.0,3.0,0
42.0,1.0,4.0,140.0,226.0,0.0,0.0,178.0,0.0,0.0,1.0,0.0,3.0,0
43.0,1.0,4.0,120.0,177.0,0.0,2.0,120.0,1.0,2.5,2.0,0.0,7.0,3
57.0,1.0,4.0,150.0,276.0,0.0,2.0,112.0,1.0,0.6,2.0,1.0,6.0,1
55.0,1.0,4.0,132.0,353.0,0.0,0.0,132.0,1.0,1.2,2.0,1.0,7.0,3
61.0,1.0,3.0,150.0,243.0,1.0,0.0,137.0,1.0,1.0,2.0,0.0,3.0,0
65.0,0.0,4.0,150.0,225.0,0.0,2.0,114.0,0.0,1.0,2.0,3.0,7.0,4
40.0,1.0,1.0,140.0,199.0,0.0,0.0,178.0,1.0,1.4,1.0,0.0,7.0,0
71.0,0.0,2.0,160.0,302.0,0.0,0.0,162.0,0.0,0.4,1.0,2.0,3.0,0
59.0,1.0,3.0,150.0,212.0,1.0,0.0,157.0,0.0,1.6,1.0,0.0,3.0,0
61.0,0.0,4.0,130.0,330.0,0.0,2.0,169.0,0.0,0.0,1.0,0.0,3.0,1
58.0,1.0,3.0,112.0,230.0,0.0,2.0,165.0,0.0,2.5,2.0,1.0,7.0,4
51.0,1.0,3.0,110.0,175.0,0.0,0.0,123.0,0.0,0.6,1.0,0.0,3.0,0
50.0,1.0,4.0,150.0,243.0,0.0,2.0,128.0,0.0,2.6,2.0,0.0,7.0,4
65.0,0.0,3.0,140.0,417.0,1.0,2.0,157.0,0.0,0.8,1.0,1.0,3.0,0
53.0,1.0,3.0,130.0,197.0,1.0,2.0,152.0,0.0,1.2,3.0,0.0,3.0,0
41.0,0.0,2.0,105.0,198.0,0.0,0.0,168.0,0.0,0.0,1.0,1.0,3.0,0
65.0,1.0,4.0,120.0,177.0,0.0,0.0,140.0,0.0,0.4,1.0,0.0,7.0,0
44.0,1.0,4.0,112.0,290.0,0.0,2.0,153.0,0.0,0.0,1.0,1.0,3.0,2
44.0,1.0,2.0,130.0,219.0,0.0,2.0,188.0,0.0,0.0,1.0,0.0,3.0,0
60.0,1.0,4.0,130.0,253.0,0.0,0.0,144.0,1.0,1.4,1.0,1.0,7.0,1
54.0,1.0,4.0,124.0,266.0,0.0,2.0,109.0,1.0,2.2,2.0,1.0,7.0,1
50.0,1.0,3.0,140.0,233.0,0.0,0.0,163.0,0.0,0.6,2.0,1.0,7.0,1
41.0,1.0,4.0,110.0,172.0,0.0,2.0,158.0,0.0,0.0,1.0,0.0,7.0,1
54.0,1.0,3.0,125.0,273.0,0.0,2.0,152.0,0.0,0.5,3.0,1.0,3.0,0
51.0,1.0,1.0,125.0,213.0,0.0,2.0,125.0,1.0,1.4,1.0,1.0,3.0,0
51.0,0.0,4.0,130.0,305.0,0.0,0.0,142.0,1.0,1.2,2.0,0.0,7.0,2
46.0,0.0,3.0,142.0,177.0,0.0,2.0,160.0,1.0,1.4,3.0,0.0,3.0,0
58.0,1.0,4.0,128.0,216.0,0.0,2.0,131.0,1.0,2.2,2.0,3.0,7.0,1
54.0,0.0,3.0,135.0,304.0,1.0,0.0,170.0,0.0,0.0,1.0,0.0,3.0,0
54.0,1.0,4.0,120.0,188.0,0.0,0.0,113.0,0.0,1.4,2.0,1.0,7.0,2
60.0,1.0,4.0,145.0,282.0,0.0,2.0,142.0,1.0,2.8,2.0,2.0,7.0,2
60.0,1.0,3.0,140.0,185.0,0.0,2.0,155.0,0.0,3.0,2.0,0.0,3.0,1
54.0,1.0,3.0,150.0,232.0,0.0,2.0,165.0,0.0,1.6,1.0,0.0,7.0,0
59.0,1.0,4.0,170.0,326.0,0.0,2.0,140.0,1.0,3.4,3.0,0.0,7.0,2
46.0,1.0,3.0,150.0,231.0,0.0,0.0,147.0,0.0,3.6,2.0,0.0,3.0,1
65.0,0.0,3.0,155.0,269.0,0.0,0.0,148.0,0.0,0.8,1.0,0.0,3.0,0
67.0,1.0,4.0,125.0,254.0,1.0,0.0,163.0,0.0,0.2,2.0,2.0,7.0,3
62.0,1.0,4.0,120.0,267.0,0.0,0.0,99.0,1.0,1.8,2.0,2.0,7.0,1
65.0,1.0,4.0,110.0,248.0,0.0,2.0,158.0,0.0,0.6,1.0,2.0,6.0,1
44.0,1.0,4.0,110.0,197.0,0.0,2.0,177.0,0.0,0.0,1.0,1.0,3.0,1
65.0,0.0,3.0,160.0,360.0,0.0,2.0,151.0,0.0,0.8,1.0,0.0,3.0,0
60.0,1.0,4.0,125.0,258.0,0.0,2.0,141.0,1.0,2.8,2.0,1.0,7.0,1
51.0,0.0,3.0,140.0,308.0,0.0,2.0,142.0,0.0,1.5,1.0,1.0,3.0,0
48.0,1.0,2.0,130.0,245.0,0.0,2.0,180.0,0.0,0.2,2.0,0.0,3.0,0
58.0,1.0,4.0,150.0,270.0,0.0,2.0,111.0,1.0,0.8,1.0,0.0,7.0,3
45.0,1.0,4.0,104.0,208.0,0.0,2.0,148.0,1.0,3.0,2.0,0.0,3.0,0
53.0,0.0,4.0,130.0,264.0,0.0,2.0,143.0,0.0,0.4,2.0,0.0,3.0,0
39.0,1.0,3.0,140.0,321.0,0.0,2.0,182.0,0.0,0.0,1.0,0.0,3.0,0
68.0,1.0,3.0,180.0,274.0,1.0,2.0,150.0,1.0,1.6,2.0,0.0,7.0,3
52.0,1.0,2.0,120.0,325.0,0.0,0.0,172.0,0.0,0.2,1.0,0.0,3.0,0
44.0,1.0,3.0,140.0,235.0,0.0,2.0,180.0,0.0,0.0,1.0,0.0,3.0,0
47.0,1.0,3.0,138.0,257.0,0.0,2.0,156.0,0.0,0.0,1.0,0.0,3.0,0
53.0,0.0,3.0,128.0,216.0,0.0,2.0,115.0,0.0,0.0,1.0,0.0,?,0
53.0,0.0,4.0,138.0,234.0,0.0,2.0,160.0,0.0,0.0,1.0,0.0,3.0,0
51.0,0.0,3.0,130.0,256.0,0.0,2.0,149.0,0.0,0.5,1.0,0.0,3.0,0
66.0,1.0,4.0,120.0,302.0,0.0,2.0,151.0,0.0,0.4,2.0,0.0,3.0,0
62.0,0.0,4.0,160.0,164.0,0.0,2.0,145.0,0.0,6.2,3.0,3.0,7.0,3
62.0,1.0,3.0,130.0,231.0,0.0,0.0,146.0,0.0,1.8,2.0,3.0,7.0,0
44.0,0.0,3.0,108.0,141.0,0.0,0.0,175.0,0.0,0.6,2.0,0.0,3.0,0
63.0,0.0,3.0,135.0,252.0,0.0,2.0,172.0,0.0,0.0,1.0,0.0,3.0,0
52.0,1.0,4.0,128.0,255.0,0.0,0.0,161.0,1.0,0.0,1.0,1.0,7.0,1
59.0,1.0,4.0,110.0,239.0,0.0,2.0,142.0,1.0,1.2,2.0,1.0,7.0,2
60.0,0.0,4.0,150.0,258.0,0.0,2.0,157.0,0.0,2.6,2.0,2.0,7.0,3
52.0,1.0,2.0,134.0,201.0,0.0,0.0,158.0,0.0,0.8,1.0,1.0,3.0,0
48.0,1.0,4.0,122.0,222.0,0.0,2.0,186.0,0.0,0.0,1.0,0.0,3.0,0
45.0,1.0,4.0,115.0,260.0,0.0,2.0,185.0,0.0,0.0,1.0,0.0,3.0,0
34.0,1.0,1.0,118.0,182.0,0.0,2.0,174.0,0.0,0.0,1.0,0.0,3.0,0
57.0,0.0,4.0,128.0,303.0,0.0,2.0,159.0,0.0,0.0,1.0,1.0,3.0,0
71.0,0.0,3.0,110.0,265.0,1.0,2.0,130.0,0.0,0.0,1.0,1.0,3.0,0
49.0,1.0,3.0,120.0,188.0,0.0,0.0,139.0,0.0,2.0,2.0,3.0,7.0,3
54.0,1.0,2.0,108.0,309.0,0.0,0.0,156.0,0.0,0.0,1.0,0.0,7.0,0
59.0,1.0,4.0,140.0,177.0,0.0,0.0,162.0,1.0,0.0,1.0,1.0,7.0,2
57.0,1.0,3.0,128.0,229.0,0.0,2.0,150.0,0.0,0.4,2.0,1.0,7.0,1
61.0,1.0,4.0,120.0,260.0,0.0,0.0,140.0,1.0,3.6,2.0,1.0,7.0,2
39.0,1.0,4.0,118.0,219.0,0.0,0.0,140.0,0.0,1.2,2.0,0.0,7.0,3
61.0,0.0,4.0,145.0,307.0,0.0,2.0,146.0,1.0,1.0,2.0,0.0,7.0,1
56.0,1.0,4.0,125.0,249.0,1.0,2.0,144.0,1.0,1.2,2.0,1.0,3.0,1
52.0,1.0,1.0,118.0,186.0,0.0,2.0,190.0,0.0,0.0,2.0,0.0,6.0,0
43.0,0.0,4.0,132.0,341.0,1.0,2.0,136.0,1.0,3.0,2.0,0.0,7.0,2
62.0,0.0,3.0,130.0,263.0,0.0,0.0,97.0,0.0,1.2,2.0,1.0,7.0,2
41.0,1.0,2.0,135.0,203.0,0.0,0.0,132.0,0.0,0.0,2.0,0.0,6.0,0
58.0,1.0,3.0,140.0,211.0,1.0,2.0,165.0,0.0,0.0,1.0,0.0,3.0,0
35.0,0.0,4.0,138.0,183.0,0.0,0.0,182.0,0.0,1.4,1.0,0.0,3.0,0
63.0,1.0,4.0,130.0,330.0,1.0,2.0,132.0,1.0,1.8,1.0,3.0,7.0,3
65.0,1.0,4.0,135.0,254.0,0.0,2.0,127.0,0.0,2.8,2.0,1.0,7.0,2
48.0,1.0,4.0,130.0,256.0,1.0,2.0,150.0,1.0,0.0,1.0,2.0,7.0,3
63.0,0.0,4.0,150.0,407.0,0.0,2.0,154.0,0.0,4.0,2.0,3.0,7.0,4
51.0,1.0,3.0,100.0,222.0,0.0,0.0,143.0,1.0,1.2,2.0,0.0,3.0,0
55.0,1.0,4.0,140.0,217.0,0.0,0.0,111.0,1.0,5.6,3.0,0.0,7.0,3
65.0,1.0,1.0,138.0,282.0,1.0,2.0,174.0,0.0,1.4,2.0,1.0,3.0,1
45.0,0.0,2.0,130.0,234.0,0.0,2.0,175.0,0.0,0.6,2.0,0.0,3.0,0
56.0,0.0,4.0,200.0,288.0,1.0,2.0,133.0,1.0,4.0,3.0,2.0,7.0,3
54.0,1.0,4.0,110.0,239.0,0.0,0.0,126.0,1.0,2.8,2.0,1.0,7.0,3
44.0,1.0,2.0,120.0,220.0,0.0,0.0,170.0,0.0,0.0,1.0,0.0,3.0,0
62.0,0.0,4.0,124.0,209.0,0.0,0.0,163.0,0.0,0.0,1.0,0.0,3.0,0
54.0,1.0,3.0,120.0,258.0,0.0,2.0,147.0,0.0,0.4,2.0,0.0,7.0,0
51.0,1.0,3.0,94.0,227.0,0.0,0.0,154.0,1.0,0.0,1.0,1.0,7.0,0
29.0,1.0,2.0,130.0,204.0,0.0,2.0,202.0,0.0,0.0,1.0,0.0,3.0,0
51.0,1.0,4.0,140.0,261.0,0.0,2.0,186.0,1.0,0.0,1.0,0.0,3.0,0
43.0,0.0,3.0,122.0,213.0,0.0,0.0,165.0,0.0,0.2,2.0,0.0,3.0,0
55.0,0.0,2.0,135.0,250.0,0.0,2.0,161.0,0.0,1.4,2.0,0.0,3.0,0
70.0,1.0,4.0,145.0,174.0,0.0,0.0,125.0,1.0,2.6,3.0,0.0,7.0,4
62.0,1.0,2.0,120.0,281.0,0.0,2.0,103.0,0.0,1.4,2.0,1.0,7.0,3
35.0,1.0,4.0,120.0,198.0,0.0,0.0,130.0,1.0,1.6,2.0,0.0,7.0,1
51.0,1.0,3.0,125.0,245.0,1.0,2.0,166.0,0.0,2.4,2.0,0.0,3.0,0
59.0,1.0,2.0,140.0,221.0,0.0,0.0,164.0,1.0,0.0,1.0,0.0,3.0,0
59.0,1.0,1.0,170.0,288.0,0.0,2.0,159.0,0.0,0.2,2.0,0.0,7.0,1
52.0,1.0,2.0,128.0,205.0,1.0,0.0,184.0,0.0,0.0,1.0,0.0,3.0,0
64.0,1.0,3.0,125.0,309.0,0.0,0.0,131.0,1.0,1.8,2.0,0.0,7.0,1
58.0,1.0,3.0,105.0,240.0,0.0,2.0,154.0,1.0,0.6,2.0,0.0,7.0,0
47.0,1.0,3.0,108.0,243.0,0.0,0.0,152.0,0.0,0.0,1.0,0.0,3.0,1
57.0,1.0,4.0,165.0,289.0,1.0,2.0,124.0,0.0,1.0,2.0,3.0,7.0,4
41.0,1.0,3.0,112.0,250.0,0.0,0.0,179.0,0.0,0.0,1.0,0.0,3.0,0
45.0,1.0,2.0,128.0,308.0,0.0,2.0,170.0,0.0,0.0,1.0,0.0,3.0,0
60.0,0.0,3.0,102.0,318.0,0.0,0.0,160.0,0.0,0.0,1.0,1.0,3.0,0
52.0,1.0,1.0,152.0,298.0,1.0,0.0,178.0,0.0,1.2,2.0,0.0,7.0,0
42.0,0.0,4.0,102.0,265.0,0.0,2.0,122.0,0.0,0.6,2.0,0.0,3.0,0
67.0,0.0,3.0,115.0,564.0,0.0,2.0,160.0,0.0,1.6,2.0,0.0,7.0,0
55.0,1.0,4.0,160.0,289.0,0.0,2.0,145.0,1.0,0.8,2.0,1.0,7.0,4
64.0,1.0,4.0,120.0,246.0,0.0,2.0,96.0,1.0,2.2,3.0,1.0,3.0,3
70.0,1.0,4.0,130.0,322.0,0.0,2.0,109.0,0.0,2.4,2.0,3.0,3.0,1
51.0,1.0,4.0,140.0,299.0,0.0,0.0,173.0,1.0,1.6,1.0,0.0,7.0,1
58.0,1.0,4.0,125.0,300.0,0.0,2.0,171.0,0.0,0.0,1.0,2.0,7.0,1
60.0,1.0,4.0,140.0,293.0,0.0,2.0,170.0,0.0,1.2,2.0,2.0,7.0,2
68.0,1.0,3.0,118.0,277.0,0.0,0.0,151.0,0.0,1.0,1.0,1.0,7.0,0
46.0,1.0,2.0,101.0,197.0,1.0,0.0,156.0,0.0,0.0,1.0,0.0,7.0,0
77.0,1.0,4.0,125.0,304.0,0.0,2.0,162.0,1.0,0.0,1.0,3.0,3.0,4
54.0,0.0,3.0,110.0,214.0,0.0,0.0,158.0,0.0,1.6,2.0,0.0,3.0,0
58.0,0.0,4.0,100.0,248.0,0.0,2.0,122.0,0.0,1.0,2.0,0.0,3.0,0
48.0,1.0,3.0,124.0,255.0,1.0,0.0,175.0,0.0,0.0,1.0,2.0,3.0,0
57.0,1.0,4.0,132.0,207.0,0.0,0.0,168.0,1.0,0.0,1.0,0.0,7.0,0
52.0,1.0,3.0,138.0,223.0,0.0,0.0,169.0,0.0,0.0,1.0,?,3.0,0
54.0,0.0,2.0,132.0,288.0,1.0,2.0,159.0,1.0,0.0,1.0,1.0,3.0,0
35.0,1.0,4.0,126.0,282.0,0.0,2.0,156.0,1.0,0.0,1.0,0.0,7.0,1
45.0,0.0,2.0,112.0,160.0,0.0,0.0,138.0,0.0,0.0,2.0,0.0,3.0,0
70.0,1.0,3.0,160.0,269.0,0.0,0.0,112.0,1.0,2.9,2.0,1.0,7.0,3
53.0,1.0,4.0,142.0,226.0,0.0,2.0,111.0,1.0,0.0,1.0,0.0,7.0,0
59.0,0.0,4.0,174.0,249.0,0.0,0.0,143.0,1.0,0.0,2.0,0.0,3.0,1
62.0,0.0,4.0,140.0,394.0,0.0,2.0,157.0,0.0,1.2,2.0,0.0,3.0,0
64.0,1.0,4.0,145.0,212.0,0.0,2.0,132.0,0.0,2.0,2.0,2.0,6.0,4
57.0,1.0,4.0,152.0,274.0,0.0,0.0,88.0,1.0,1.2,2.0,1.0,7.0,1
52.0,1.0,4.0,108.0,233.0,1.0,0.0,147.0,0.0,0.1,1.0,3.0,7.0,0
56.0,1.0,4.0,132.0,184.0,0.0,2.0,105.0,1.0,2.1,2.0,1.0,6.0,1
43.0,1.0,3.0,130.0,315.0,0.0,0.0,162.0,0.0,1.9,1.0,1.0,3.0,0
53.0,1.0,3.0,130.0,246.0,1.0,2.0,173.0,0.0,0.0,1.0,3.0,3.0,0
48.0,1.0,4.0,124.0,274.0,0.0,2.0,166.0,0.0,0.5,2.0,0.0,7.0,3
56.0,0.0,4.0,134.0,409.0,0.0,2.0,150.0,1.0,1.9,2.0,2.0,7.0,2
42.0,1.0,1.0,148.0,244.0,0.0,2.0,178.0,0.0,0.8,1.0,2.0,3.0,0
59.0,1.0,1.0,178.0,270.0,0.0,2.0,145.0,0.0,4.2,3.0,0.0,7.0,0
60.0,0.0,4.0,158.0,305.0,0.0,2.0,161.0,0.0,0.0,1.0,0.0,3.0,1
63.0,0.0,2.0,140.0,195.0,0.0,0.0,179.0,0.0,0.0,1.0,2.0,3.0,0
42.0,1.0,3.0,120.0,240.0,1.0,0.0,194.0,0.0,0.8,3.0,0.0,7.0,0
66.0,1.0,2.0,160.0,246.0,0.0,0.0,120.0,1.0,0.0,2.0,3.0,6.0,2
54.0,1.0,2.0,192.0,283.0,0.0,2.0,195.0,0.0,0.0,1.0,1.0,7.0,1
69.0,1.0,3.0,140.0,254.0,0.0,2.0,146.0,0.0,2.0,2.0,3.0,7.0,2
50.0,1.0,3.0,129.0,196.0,0.0,0.0,163.0,0.0,0.0,1.0,0.0,3.0,0
51.0,1.0,4.0,140.0,298.0,0.0,0.0,122.0,1.0,4.2,2.0,3.0,7.0,3
43.0,1.0,4.0,132.0,247.0,1.0,2.0,143.0,1.0,0.1,2.0,?,7.0,1
62.0,0.0,4.0,138.0,294.0,1.0,0.0,106.0,0.0,1.9,2.0,3.0,3.0,2
68.0,0.0,3.0,120.0,211.0,0.0,2.0,115.0,0.0,1.5,2.0,0.0,3.0,0
67.0,1.0,4.0,100.0,299.0,0.0,2.0,125.0,1.0,0.9,2.0,2.0,3.0,3
69.0,1.0,1.0,160.0,234.0,1.0,2.0,131.0,0.0,0.1,2.0,1.0,3.0,0
45.0,0.0,4.0,138.0,236.0,0.0,2.0,152.0,1.0,0.2,2.0,0.0,3.0,0
50.0,0.0,2.0,120.0,244.0,0.0,0.0,162.0,0.0,1.1,1.0,0.0,3.0,0
59.0,1.0,1.0,160.0,273.0,0.0,2.0,125.0,0.0,0.0,1.0,0.0,3.0,1
50.0,0.0,4.0,110.0,254.0,0.0,2.0,159.0,0.0,0.0,1.0,0.0,3.0,0
64.0,0.0,4.0,180.0,325.0,0.0,0.0,154.0,1.0,0.0,1.0,0.0,3.0,0
57.0,1.0,3.0,150.0,126.0,1.0,0.0,173.0,0.0,0.2,1.0,1.0,7.0,0
64.0,0.0,3.0,140.0,313.0,0.0,0.0,133.0,0.0,0.2,1.0,0.0,7.0,0
43.0,1.0,4.0,110.0,211.0,0.0,0.0,161.0,0.0,0.0,1.0,0.0,7.0,0
45.0,1.0,4.0,142.0,309.0,0.0,2.0,147.0,1.0,0.0,2.0,3.0,7.0,3
58.0,1.0,4.0,128.0,259.0,0.0,2.0,130.0,1.0,3.0,2.0,2.0,7.0,3
50.0,1.0,4.0,144.0,200.0,0.0,2.0,126.0,1.0,0.9,2.0,0.0,7.0,3
55.0,1.0,2.0,130.0,262.0,0.0,0.0,155.0,0.0,0.0,1.0,0.0,3.0,0
62.0,0.0,4.0,150.0,244.0,0.0,0.0,154.0,1.0,1.4,2.0,0.0,3.0,1
37.0,0.0,3.0,120.0,215.0,0.0,0.0,170.0,0.0,0.0,1.0,0.0,3.0,0
38.0,1.0,1.0,120.0,231.0,0.0,0.0,182.0,1.0,3.8,2.0,0.0,7.0,4
41.0,1.0,3.0,130.0,214.0,0.0,2.0,168.0,0.0,2.0,2.0,0.0,3.0,0
66.0,0.0,4.0,178.0,228.0,1.0,0.0,165.0,1.0,1.0,2.0,2.0,7.0,3
52.0,1.0,4.0,112.0,230.0,0.0,0.0,160.0,0.0,0.0,1.0,1.0,3.0,1
56.0,1.0,1.0,120.0,193.0,0.0,2.0,162.0,0.0,1.9,2.0,0.0,7.0,0
46.0,0.0,2.0,105.0,204.0,0.0,0.0,172.0,0.0,0.0,1.0,0.0,3.0,0
46.0,0.0,4.0,138.0,243.0,0.0,2.0,152.0,1.0,0.0,2.0,0.0,3.0,0
64.0,0.0,4.0,130.0,303.0,0.0,0.0,122.0,0.0,2.0,2.0,2.0,3.0,0
59.0,1.0,4.0,138.0,271.0,0.0,2.0,182.0,0.0,0.0,1.0,0.0,3.0,0
41.0,0.0,3.0,112.0,268.0,0.0,2.0,172.0,1.0,0.0,1.0,0.0,3.0,0
54.0,0.0,3.0,108.0,267.0,0.0,2.0,167.0,0.0,0.0,1.0,0.0,3.0,0
39.0,0.0,3.0,94.0,199.0,0.0,0.0,179.0,0.0,0.0,1.0,0.0,3.0,0
53.0,1.0,4.0,123.0,282.0,0.0,0.0,95.0,1.0,2.0,2.0,2.0,7.0,3
63.0,0.0,4.0,108.0,269.0,0.0,0.0,169.0,1.0,1.8,2.0,2.0,3.0,1
34.0,0.0,2.0,118.0,210.0,0.0,0.0,192.0,0.0,0.7,1.0,0.0,3.0,0
47.0,1.0,4.0,112.0,204.0,0.0,0.0,143.0,0.0,0.1,1.0,0.0,3.0,0
67.0,0.0,3.0,152.0,277.0,0.0,0.0,172.0,0.0,0.0,1.0,1.0,3.0,0
54.0,1.0,4.0,110.0,206.0,0.0,2.0,108.0,1.0,0.0,2.0,1.0,3.0,3
66.0,1.0,4.0,112.0,212.0,0.0,2.0,132.0,1.0,0.1,1.0,1.0,3.0,2
52.0,0.0,3.0,136.0,196.0,0.0,2.0,169.0,0.0,0.1,2.0,0.0,3.0,0
55.0,0.0,4.0,180.0,327.0,0.0,1.0,117.0,1.0,3.4,2.0,0.0,3.0,2
49.0,1.0,3.0,118.0,149.0,0.0,2.0,126.0,0.0,0.8,1.0,3.0,3.0,1
74.0,0.0,2.0,120.0,269.0,0.0,2.0,121.0,1.0,0.2,1.0,1.0,3.0,0
54.0,0.0,3.0,160.0,201.0,0.0,0.0,163.0,0.0,0.0,1.0,1.0,3.0,0
54.0,1.0,4.0,122.0,286.0,0.0,2.0,116.0,1.0,3.2,2.0,2.0,3.0,3
56.0,1.0,4.0,130.0,283.0,1.0,2.0,103.0,1.0,1.6,3.0,0.0,7.0,2
46.0,1.0,4.0,120.0,249.0,0.0,2.0,144.0,0.0,0.8,1.0,0.0,7.0,1
49.0,0.0,2.0,134.0,271.0,0.0,0.0,162.0,0.0,0.0,2.0,0.0,3.0,0
42.0,1.0,2.0,120.0,295.0,0.0,0.0,162.0,0.0,0.0,1.0,0.0,3.0,0
41.0,1.0,2.0,110.0,235.0,0.0,0.0,153.0,0.0,0.0,1.0,0.0,3.0,0
41.0,0.0,2.0,126.0,306.0,0.0,0.0,163.0,0.0,0.0,1.0,0.0,3.0,0
49.0,0.0,4.0,130.0,269.0,0.0,0.0,163.0,0.0,0.0,1.0,0.0,3.0,0
61.0,1.0,1.0,134.0,234.0,0.0,0.0,145.0,0.0,2.6,2.0,2.0,3.0,2
60.0,0.0,3.0,120.0,178.0,1.0,0.0,96.0,0.0,0.0,1.0,0.0,3.0,0
67.0,1.0,4.0,120.0,237.0,0.0,0.0,71.0,0.0,1.0,2.0,0.0,3.0,2
58.0,1.0,4.0,100.0,234.0,0.0,0.0,156.0,0.0,0.1,1.0,1.0,7.0,2
47.0,1.0,4.0,110.0,275.0,0.0,2.0,118.0,1.0,1.0,2.0,1.0,3.0,1
52.0,1.0,4.0,125.0,212.0,0.0,0.0,168.0,0.0,1.0,1.0,2.0,7.0,3
62.0,1.0,2.0,128.0,208.0,1.0,2.0,140.0,0.0,0.0,1.0,0.0,3.0,0
57.0,1.0,4.0,110.0,201.0,0.0,0.0,126.0,1.0,1.5,2.0,0.0,6.0,0
58.0,1.0,4.0,146.0,218.0,0.0,0.0,105.0,0.0,2.0,2.0,1.0,7.0,1
64.0,1.0,4.0,128.0,263.0,0.0,0.0,105.0,1.0,0.2,2.0,1.0,7.0,0
51.0,0.0,3.0,120.0,295.0,0.0,2.0,157.0,0.0,0.6,1.0,0.0,3.0,0
43.0,1.0,4.0,115.0,303.0,0.0,0.0,181.0,0.0,1.2,2.0,0.0,3.0,0
42.0,0.0,3.0,120.0,209.0,0.0,0.0,173.0,0.0,0.0,2.0,0.0,3.0,0
67.0,0.0,4.0,106.0,223.0,0.0,0.0,142.0,0.0,0.3,1.0,2.0,3.0,0
76.0,0.0,3.0,140.0,197.0,0.0,1.0,116.0,0.0,1.1,2.0,0.0,3.0,0
70.0,1.0,2.0,156.0,245.0,0.0,2.0,143.0,0.0,0.0,1.0,0.0,3.0,0
57.0,1.0,2.0,124.0,261.0,0.0,0.0,141.0,0.0,0.3,1.0,0.0,7.0,1
44.0,0.0,3.0,118.0,242.0,0.0,0.0,149.0,0.0,0.3,2.0,1.0,3.0,0
58.0,0.0,2.0,136.0,319.0,1.0,2.0,152.0,0.0,0.0,1.0,2.0,3.0,3
60.0,0.0,1.0,150.0,240.0,0.0,0.0,171.0,0.0,0.9,1.0,0.0,3.0,0
44.0,1.0,3.0,120.0,226.0,0.0,0.0,169.0,0.0,0.0,1.0,0.0,3.0,0
61.0,1.0,4.0,138.0,166.0,0.0,2.0,125.0,1.0,3.6,2.0,1.0,3.0,4
42.0,1.0,4.0,136.0,315.0,0.0,0.0,125.0,1.0,1.8,2.0,0.0,6.0,2
52.0,1.0,4.0,128.0,204.0,1.0,0.0,156.0,1.0,1.0,2.0,0.0,?,2
59.0,1.0,3.0,126.0,218.0,1.0,0.0,134.0,0.0,2.2,2.0,1.0,6.0,2
40.0,1.0,4.0,152.0,223.0,0.0,0.0,181.0,0.0,0.0,1.0,0.0,7.0,1
42.0,1.0,3.0,130.0,180.0,0.0,0.0,150.0,0.0,0.0,1.0,0.0,3.0,0
61.0,1.0,4.0,140.0,207.0,0.0,2.0,138.0,1.0,1.9,1.0,1.0,7.0,1
66.0,1.0,4.0,160.0,228.0,0.0,2.0,138.0,0.0,2.3,1.0,0.0,6.0,0
46.0,1.0,4.0,140.0,311.0,0.0,0.0,120.0,1.0,1.8,2.0,2.0,7.0,2
71.0,0.0,4.0,112.0,149.0,0.0,0.0,125.0,0.0,1.6,2.0,0.0,3.0,0
59.0,1.0,1.0,134.0,204.0,0.0,0.0,162.0,0.0,0.8,1.0,2.0,3.0,1
64.0,1.0,1.0,170.0,227.0,0.0,2.0,155.0,0.0,0.6,2.0,0.0,7.0,0
66.0,0.0,3.0,146.0,278.0,0.0,2.0,152.0,0.0,0.0,2.0,1.0,3.0,0
39.0,0.0,3.0,138.0,220.0,0.0,0.0,152.0,0.0,0.0,2.0,0.0,3.0,0
57.0,1.0,2.0,154.0,232.0,0.0,2.0,164.0,0.0,0.0,1.0,1.0,3.0,1
58.0,0.0,4.0,130.0,197.0,0.0,0.0,131.0,0.0,0.6,2.0,0.0,3.0,0
57.0,1.0,4.0,110.0,335.0,0.0,0.0,143.0,1.0,3.0,2.0,1.0,7.0,2
47.0,1.0,3.0,130.0,253.0,0.0,0.0,179.0,0.0,0.0,1.0,0.0,3.0,0
55.0,0.0,4.0,128.0,205.0,0.0,1.0,130.0,1.0,2.0,2.0,1.0,7.0,3
35.0,1.0,2.0,122.0,192.0,0.0,0.0,174.0,0.0,0.0,1.0,0.0,3.0,0
61.0,1.0,4.0,148.0,203.0,0.0,0.0,161.0,0.0,0.0,1.0,1.0,7.0,2
58.0,1.0,4.0,114.0,318.0,0.0,1.0,140.0,0.0,4.4,3.0,3.0,6.0,4
58.0,0.0,4.0,170.0,225.0,1.0,2.0,146.0,1.0,2.8,2.0,2.0,6.0,2
58.0,1.0,2.0,125.0,220.0,0.0,0.0,144.0,0.0,0.4,2.0,?,7.0,0
56.0,1.0,2.0,130.0,221.0,0.0,2.0,163.0,0.0,0.0,1.0,0.0,7.0,0
56.0,1.0,2.0,120.0,240.0,0.0,0.0,169.0,0.0,0.0,3.0,0.0,3.0,0
67.0,1.0,3.0,152.0,212.0,0.0,2.0,150.0,0.0,0.8,2.0,0.0,7.0,1
55.0,0.0,2.0,132.0,342.0,0.0,0.0,166.0,0.0,1.2,1.0,0.0,3.0,0
44.0,1.0,4.0,120.0,169.0,0.0,0.0,144.0,1.0,2.8,3.0,0.0,6.0,2
63.0,1.0,4.0,140.0,187.0,0.0,2.0,144.0,1.0,4.0,1.0,2.0,7.0,2
63.0,0.0,4.0,124.0,197.0,0.0,0.0,136.0,1.0,0.0,2.0,0.0,3.0,1
41.0,1.0,2.0,120.0,157.0,0.0,0.0,182.0,0.0,0.0,1.0,0.0,3.0,0
59.0,1.0,4.0,164.0,176.0,1.0,2.0,90.0,0.0,1.0,2.0,2.0,6.0,3
57.0,0.0,4.0,140.0,241.0,0.0,0.0,123.0,1.0,0.2,2.0,0.0,7.0,1
45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0,1
68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0,2
57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0,3
57.0,0.0,2.0,130.0,236.0,0.0,2.0,174.0,0.0,0.0,2.0,1.0,3.0,1
38.0,1.0,3.0,138.0,175.0,0.0,0.0,173.0,0.0,0.0,1.0,?,3.0,0


In [ ]:
%%writefile data/iris/iris.csv
sepal_length,sepal_width,petal_length,petal_width,species
5.1,3.5,1.4,0.2,setosa
4.9,3.0,1.4,0.2,setosa
4.7,3.2,1.3,0.2,setosa
4.6,3.1,1.5,0.2,setosa
5.0,3.6,1.4,0.2,setosa
5.4,3.9,1.7,0.4,setosa
4.6,3.4,1.4,0.3,setosa
5.0,3.4,1.5,0.2,setosa
4.4,2.9,1.4,0.2,setosa
4.9,3.1,1.5,0.1,setosa
5.4,3.7,1.5,0.2,setosa
4.8,3.4,1.6,0.2,setosa
4.8,3.0,1.4,0.1,setosa
4.3,3.0,1.1,0.1,setosa
5.8,4.0,1.2,0.2,setosa
5.7,4.4,1.5,0.4,setosa
5.4,3.9,1.3,0.4,setosa
5.1,3.5,1.4,0.3,setosa
5.7,3.8,1.7,0.3,setosa
5.1,3.8,1.5,0.3,setosa
5.4,3.4,1.7,0.2,setosa
5.1,3.7,1.5,0.4,setosa
4.6,3.6,1.0,0.2,setosa
5.1,3.3,1.7,0.5,setosa
4.8,3.4,1.9,0.2,setosa
5.0,3.0,1.6,0.2,setosa
5.0,3.4,1.6,0.4,setosa
5.2,3.5,1.5,0.2,setosa
5.2,3.4,1.4,0.2,setosa
4.7,3.2,1.6,0.2,setosa
4.8,3.1,1.6,0.2,setosa
5.4,3.4,1.5,0.4,setosa
5.2,4.1,1.5,0.1,setosa
5.5,4.2,1.4,0.2,setosa
4.9,3.1,1.5,0.2,setosa
5.0,3.2,1.2,0.2,setosa
5.5,3.5,1.3,0.2,setosa
4.9,3.6,1.4,0.1,setosa
4.4,3.0,1.3,0.2,setosa
5.1,3.4,1.5,0.2,setosa
5.0,3.5,1.3,0.3,setosa
4.5,2.3,1.3,0.3,setosa
4.4,3.2,1.3,0.2,setosa
5.0,3.5,1.6,0.6,setosa
5.1,3.8,1.9,0.4,setosa
4.8,3.0,1.4,0.3,setosa
5.1,3.8,1.6,0.2,setosa
4.6,3.2,1.4,0.2,setosa
5.3,3.7,1.5,0.2,setosa
5.0,3.3,1.4,0.2,setosa
7.0,3.2,4.7,1.4,versicolor
6.4,3.2,4.5,1.5,versicolor
6.9,3.1,4.9,1.5,versicolor
5.5,2.3,4.0,1.3,versicolor
6.5,2.8,4.6,1.5,versicolor
5.7,2.8,4.5,1.3,versicolor
6.3,3.3,4.7,1.6,versicolor
4.9,2.4,3.3,1.0,versicolor
6.6,2.9,4.6,1.3,versicolor
5.2,2.7,3.9,1.4,versicolor
5.0,2.0,3.5,1.0,versicolor
5.9,3.0,4.2,1.5,versicolor
6.0,2.2,4.0,1.0,versicolor
6.1,2.9,4.7,1.4,versicolor
5.6,2.9,3.6,1.3,versicolor
6.7,3.1,4.4,1.4,versicolor
5.6,3.0,4.5,1.5,versicolor
5.8,2.7,4.1,1.0,versicolor
6.2,2.2,4.5,1.5,versicolor
5.6,2.5,3.9,1.1,versicolor
5.9,3.2,4.8,1.8,versicolor
6.1,2.8,4.0,1.3,versicolor
6.3,2.5,4.9,1.5,versicolor
6.1,2.8,4.7,1.2,versicolor
6.4,2.9,4.3,1.3,versicolor
6.6,3.0,4.4,1.4,versicolor
6.8,2.8,4.8,1.4,versicolor
6.7,3.0,5.0,1.7,versicolor
6.0,2.9,4.5,1.5,versicolor
5.7,2.6,3.5,1.0,versicolor
5.5,2.4,3.8,1.1,versicolor
5.5,2.4,3.7,1.0,versicolor
5.8,2.7,3.9,1.2,versicolor
6.0,2.7,5.1,1.6,versicolor
5.4,3.0,4.5,1.5,versicolor
6.0,3.4,4.5,1.6,versicolor
6.7,3.1,4.7,1.5,versicolor
6.3,2.3,4.4,1.3,versicolor
5.6,3.0,4.1,1.3,versicolor
5.5,2.5,4.0,1.3,versicolor
5.5,2.6,4.4,1.2,versicolor
6.1,3.0,4.6,1.4,versicolor
5.8,2.6,4.0,1.2,versicolor
5.0,2.3,3.3,1.0,versicolor
5.6,2.7,4.2,1.3,versicolor
5.7,3.0,4.2,1.2,versicolor
5.7,2.9,4.2,1.3,versicolor
6.2,2.9,4.3,1.3,versicolor
5.1,2.5,3.0,1.1,versicolor
5.7,2.8,4.1,1.3,versicolor
6.3,3.3,6.0,2.5,virginica
5.8,2.7,5.1,1.9,virginica
7.1,3.0,5.9,2.1,virginica
6.3,2.9,5.6,1.8,virginica
6.5,3.0,5.8,2.2,virginica
7.6,3.0,6.6,2.1,virginica
4.9,2.5,4.5,1.7,virginica
7.3,2.9,6.3,1.8,virginica
6.7,2.5,5.8,1.8,virginica
7.2,3.6,6.1,2.5,virginica
6.5,3.2,5.1,2.0,virginica
6.4,2.7,5.3,1.9,virginica
6.8,3.0,5.5,2.1,virginica
5.7,2.5,5.0,2.0,virginica
5.8,2.8,5.1,2.4,virginica
6.4,3.2,5.3,2.3,virginica
6.5,3.0,5.5,1.8,virginica
7.7,3.8,6.7,2.2,virginica
7.7,2.6,6.9,2.3,virginica
6.0,2.2,5.0,1.5,virginica
6.9,3.2,5.7,2.3,virginica
5.6,2.8,4.9,2.0,virginica
7.7,2.8,6.7,2.0,virginica
6.3,2.7,4.9,1.8,virginica
6.7,3.3,5.7,2.1,virginica
7.2,3.2,6.0,1.8,virginica
6.2,2.8,4.8,1.8,virginica
6.1,3.0,4.9,1.8,virginica
6.4,2.8,5.6,2.1,virginica
7.2,3.0,5.8,1.6,virginica
7.4,2.8,6.1,1.9,virginica
7.9,3.8,6.4,2.0,virginica
6.4,2.8,5.6,2.2,virginica
6.3,2.8,5.1,1.5,virginica
6.1,2.6,5.6,1.4,virginica
7.7,3.0,6.1,2.3,virginica
6.3,3.4,5.6,2.4,virginica
6.4,3.1,5.5,1.8,virginica
6.0,3.0,4.8,1.8,virginica
6.9,3.1,5.4,2.1,virginica
6.7,3.1,5.6,2.4,virginica
6.9,3.1,5.1,2.3,virginica
5.8,2.7,5.1,1.9,virginica
6.8,3.2,5.9,2.3,virginica
6.7,3.3,5.7,2.5,virginica
6.7,3.0,5.2,2.3,virginica
6.3,2.5,5.0,1.9,virginica
6.5,3.0,5.2,2.0,virginica
6.2,3.4,5.4,2.3,virginica
5.9,3.0,5.1,1.8,virginica


## 2. Interfaz interactiva

La interfaz permite explorar los datos, entrenar el árbol, compararlo con el Random Forest y hacer experimentos,
todo sin escribir código. Los cálculos se hacen en Python y los resultados se dibujan en el navegador.

Al iniciar, la interfaz muestra el dataset de enfermedad cardíaca. Para usar otro, se elige en la lista de arriba a la derecha. También se puede cargar un archivo CSV propio desde la pestaña **Datos**.

### Conjuntos de datos disponibles

| Conjunto de datos | Qué contiene | Para qué sirve |
|---|---|---|
| **Enfermedad cardíaca** | 303 pacientes de la Cleveland Clinic, descritos con 13 atributos clínicos, como la edad, el colesterol o el tipo de dolor de pecho. | Es el dataset principal del trabajo: un problema real que mezcla atributos numéricos y nominales, y que tiene algunos valores faltantes. Se predice si el paciente tiene o no una enfermedad cardíaca. |
| **Play Tennis** | 14 días descritos por el clima. | Es el ejemplo clásico del libro de Quinlan, así que permite comparar el árbol obtenido con el del libro. |
| **Iris** | 150 flores de 3 especies, descritas con 4 medidas. | Es el dataset con el que se desarrolló y se probó el algoritmo. |
| **Wine** | 178 vinos de 3 variedades, descritos con 13 atributos químicos. | Muestra cómo se comportan los modelos cuando hay más atributos. |
| **Breast Cancer Wisconsin** | 569 casos con 30 medidas, clasificados en 2 diagnósticos. | Es otro caso real, más exigente por la cantidad de atributos. |

> En el dataset de enfermedad cardíaca, los atributos tienen nombres en español y sus códigos se reemplazaron por su significado (por ejemplo, *sin síntomas* en lugar de *4*). Así el árbol se puede leer sin consultar un diccionario. En la pestaña **Datos**, cada atributo muestra una breve explicación de lo que mide.

| Pestaña | Qué se puede hacer |
|---|---|
| **1. Datos** | Ver el conjunto de datos elegido: sus atributos, la calidad de los datos y las decisiones que se tomaron sobre ellos. |
| **2. Árbol** | Ver el árbol C4.5 dibujado. Al cambiar los parámetros, el árbol se vuelve a entrenar al instante. También se puede describir un caso y ver qué clase predice cada modelo. |
| **3. Comparar** | Comparar el árbol con el Random Forest sobre las mismas particiones de los datos, con el promedio y el desvío de cada métrica. |
| **4. Experimentos** | Probar varios valores de un parámetro para ver cuándo aparece el sobreajuste, y medir qué pasa cuando faltan datos. |

### Recorrido sugerido

1. **Datos:** con *Enfermedad cardíaca*, que es el dataset que se muestra al iniciar, leé qué mide cada atributo y fijate, por los colores de las barras, qué valores aparecen más en los pacientes con enfermedad.
2. **Árbol:** activá *Podar por error pesimista* y compará la cantidad de nodos y la exactitud antes y después de podar.
3. **Árbol:** en *Predecir un caso*, mové los valores. Después marcá un atributo como *falta* y fijate cómo el árbol reparte el caso entre las ramas.
4. **Comparar:** tocá **Comparar** con los valores que propone la cátedra (100 árboles y profundidad 10).
5. **Experimentos:** probá distintos valores de *Profundidad máxima*, del 1 al 8, y buscá a partir de cuál el árbol empieza a sobreajustar.

In [ ]:
# Abre la interfaz interactiva.
from tp_ia.web.widget import show

lab = show()

## 3. El árbol desde código

Esta sección entrena el árbol C4.5 directamente desde Python, sin pasar por la interfaz. Muestra que la implementación
es una clase que se puede usar en cualquier programa: se crea el árbol, se lo entrena con `fit` y queda listo para
predecir.

Se usa el dataset de enfermedad cardíaca con la poda activada y la misma partición que usa la interfaz al iniciar:
el 70 % de los pacientes para entrenar y el 30 % restante para medir la exactitud con casos que el árbol no vio.
Por eso el árbol que aparece acá es idéntico al de la pestaña **Árbol** con la poda activada. Para probar otros
parámetros o consultar las decisiones sobre los datos conviene usar la interfaz, porque las pestañas **Árbol** y
**Datos** muestran esa información de forma interactiva.

In [ ]:
# Entrena el árbol propio con poda sobre el dataset de enfermedad cardíaca y lo dibuja con Graphviz,
# con el formato de las láminas de la cátedra.
from IPython.display import SVG, display
from tp_ia.c45 import C45DecisionTree
from tp_ia.data import load_heart_dataset
from tp_ia.eval import make_splits

ds = load_heart_dataset()

# La misma partición que la interfaz: 70 % para entrenar y 30 % para evaluar, estratificada y con la semilla 0.
split = make_splits(ds.y, scheme="holdout", n_repeats=1, test_size=0.3, seed=0)[0]
y = ds.y.to_numpy()
X_train, y_train = ds.X.iloc[split.train], y[split.train]
X_test, y_test = ds.X.iloc[split.test], y[split.test]

tree = C45DecisionTree(prune=True).fit(X_train, y_train, feature_types=ds.feature_types)
print(f"Entrenado con {len(y_train)} pacientes y evaluado con {len(y_test)}.")
print(f"{tree.n_nodes_} nodos (se podaron {tree.n_pruned_}), profundidad {tree.depth_}")
print(f"Exactitud con los datos de prueba: {tree.score(X_test, y_test):.1%}")

try:    # se dibuja acá mismo para que un error de Graphviz se pueda atrapar y explicar
    from graphviz import Source
    display(SVG(Source(tree.to_dot()).pipe(format="svg").decode("utf-8")))
except Exception as error:
    print("\nNo se pudo dibujar con Graphviz:", error)
    print("Hace falta tener instalado Graphviz. Mientras tanto, el árbol se puede ver como texto con print(tree.export_text()).")

## 4. Verificación (opcional)

Primero se escriben los archivos de prueba (carpeta `tests/`) y el archivo de configuración de pytest. Después, la
última celda ejecuta las pruebas automáticas del proyecto. Las pruebas comprueban que funcionen correctamente el
algoritmo C4.5, la carga de los datos, la evaluación de los modelos y la interfaz.

In [ ]:
%%writefile pytest.ini
[pytest]
pythonpath = src
testpaths = tests
addopts = -q


In [ ]:
%%writefile tests/test_criteria.py
"""Tests de los criterios de C4.5 contra valores calculados a mano y contra un oráculo externo.

El oráculo (scipy.stats.entropy) se usa SOLO acá, para comprobar nuestras fórmulas; el árbol no
depende de scipy para nada.
"""
import numpy as np
import pytest
from scipy.stats import entropy as scipy_entropy

from tp_ia.c45.criteria import entropy, entropy_rows, gain_ratio, information_gain, split_info
from tp_ia.data.fixtures import liu_loans, play_tennis


def class_counts(y, classes):
    return np.array([(y == c).sum() for c in classes], dtype=float)


def gain_of(X, y, column):
    """Ganancia de dividir por `column` (una rama por valor) usando nuestras funciones."""
    classes = sorted(y.unique())
    parent = class_counts(y, classes)
    children = np.array([class_counts(y[X[column] == v], classes) for v in sorted(X[column].unique())])
    return information_gain(parent, children)


def test_entropy_known_values():
    assert entropy([5, 0]) == 0.0                              # conjunto puro
    assert entropy([0, 0]) == 0.0                              # vacío
    assert entropy([3, 3]) == pytest.approx(1.0)               # equilibrado, 2 clases = 1 bit
    assert entropy([1, 1, 1, 1]) == pytest.approx(2.0)         # 4 clases uniformes = 2 bits
    assert entropy([9, 5]) == pytest.approx(0.9402859586706309, abs=1e-9)  # Play Tennis


def test_entropy_matches_scipy_oracle():
    rng = np.random.default_rng(0)
    for _ in range(100):
        counts = rng.integers(0, 20, size=rng.integers(2, 7)).astype(float)
        if counts.sum() == 0:
            continue
        assert entropy(counts) == pytest.approx(scipy_entropy(counts, base=2), abs=1e-9)


def test_entropy_bounds_and_scale_invariance():
    rng = np.random.default_rng(1)
    for _ in range(50):
        counts = rng.random(rng.integers(2, 8)) * 10
        h = entropy(counts)
        assert -1e-12 <= h <= np.log2(len(counts)) + 1e-12
        assert entropy(counts * 3.7) == pytest.approx(h)       # pesos fraccionarios: no importa la escala


def test_entropy_rows_matches_scalar_version():
    m = np.array([[5, 0, 0], [3, 3, 0], [1, 1, 1], [0, 0, 0]], dtype=float)
    expected = [entropy(row) for row in m]
    np.testing.assert_allclose(entropy_rows(m), expected, atol=1e-12)


def test_play_tennis_gains_match_textbook():
    X, y = play_tennis()
    expected = {"Outlook": 0.246, "Humidity": 0.151, "Wind": 0.048, "Temperature": 0.029}
    for column, value in expected.items():
        assert gain_of(X, y, column) == pytest.approx(value, abs=1e-3)


def test_liu_gains_match_book():
    X, y = liu_loans()
    expected = {"Age": 0.083, "Has_job": 0.324, "Own_house": 0.420, "Credit_rating": 0.363}
    for column, value in expected.items():
        assert gain_of(X, y, column) == pytest.approx(value, abs=1e-3)


def test_gain_ratio_of_outlook():
    X, y = play_tennis()
    gain = gain_of(X, y, "Outlook")
    si = split_info([5, 4, 5])                                  # Outlook: 5 Sunny, 4 Overcast, 5 Rain
    assert si == pytest.approx(scipy_entropy([5, 4, 5], base=2))
    assert gain_ratio(gain, si) == pytest.approx(0.156, abs=1e-3)


def test_degenerate_split_has_zero_gain_ratio():
    assert split_info([14]) == 0.0                              # una sola rama
    assert gain_ratio(0.3, 0.0) == 0.0                          # sin división por cero


def test_gain_is_never_negative_nor_above_parent_entropy():
    rng = np.random.default_rng(2)
    for _ in range(100):
        parent = rng.integers(1, 30, size=3).astype(float)
        branches = rng.integers(2, 5)
        # cada clase se reparte al azar entre las ramas -> los hijos suman exactamente al padre
        children = np.array([rng.multinomial(int(c), np.ones(branches) / branches) for c in parent]).T
        g = information_gain(parent, children)
        assert -1e-12 <= g <= entropy(parent) + 1e-12


In [ ]:
%%writefile tests/test_data.py
"""Tests de la capa de datos: Iris (CSV real del repo) y archivos 'sucios' sintéticos."""
import re

import numpy as np
import pandas as pd
import pytest

from tp_ia.data.decisions import DecisionLog
from tp_ia.data.iris import IRIS_SPEC, default_data_dir, load_iris_dataset
from tp_ia.data.loader import build_dataset, dataset_from_frame, read_table
from tp_ia.data.quality import audit
from tp_ia.data.schema import DatasetSpec, Scenario


# --------------------------------------------------------------------- lectura de archivos sucios
def test_read_table_semicolons_decimal_comma_latin1_and_missing_tokens(tmp_path):
    path = tmp_path / "raro.csv"
    path.write_bytes("edad;peso;ciudad;clase\n30;70,5;Córdoba;a\n41;?;Rosario;b\n 52 ;80,25;  ;a\n".encode("latin-1"))
    df = read_table(path)
    assert list(df.columns) == ["edad", "peso", "ciudad", "clase"]
    assert df["edad"].tolist() == [30, 41, 52]
    assert df["peso"].iloc[0] == 70.5 and np.isnan(df["peso"].iloc[1]) and df["peso"].iloc[2] == 80.25
    assert df["ciudad"].iloc[0] == "Córdoba" and pd.isna(df["ciudad"].iloc[2])


def test_read_table_tab_separated_without_header(tmp_path):
    path = tmp_path / "t.tsv"
    path.write_text("1\t2.5\tx\n2\t3.5\ty\n3\t?\tx\n", encoding="utf-8")
    df = read_table(path, header=False, columns=["a", "b", "c"])
    assert list(df.columns) == ["a", "b", "c"] and len(df) == 3
    assert np.isnan(df["b"].iloc[2]) and df["c"].tolist() == ["x", "y", "x"]


def test_a_column_with_one_text_value_stays_text(tmp_path):
    path = tmp_path / "m.csv"
    path.write_text("a,b\n1,x\n2,y\nfoo,z\n", encoding="utf-8")
    assert read_table(path)["a"].tolist() == ["1", "2", "foo"]


# --------------------------------------------------------------------- Iris
def test_iris_loads_from_csv_with_expected_shape_and_types():
    ds = load_iris_dataset()
    assert ds.X.shape == (150, 4) and len(ds.y) == 150
    assert ds.classes == ["setosa", "versicolor", "virginica"]
    assert ds.report["class_counts"] == {"setosa": 50, "versicolor": 50, "virginica": 50}
    assert set(ds.feature_types.values()) == {"numeric"}
    assert ds.report["missing"] == {} and ds.report["rows_with_missing"] == 0
    assert ds.report["duplicates"] == 1
    assert any(w.startswith("Hay 1 fila duplicada.") for w in ds.report["warnings"])


def test_iris_records_every_decision_with_evidence():
    ds = load_iris_dataset()
    assert {"D-01", "D-02", "D-03", "D-04", "D-05", "D-06"} <= set(ds.log.ids())
    markdown = ds.log.to_markdown()
    assert "(código D-04)" in markdown and "- Filas duplicadas: 1" in markdown
    assert "## Primera decisión: usar el dataset «Iris» (archivo iris.csv)" in markdown
    assert "**Por qué:** Es un conjunto de datos clásico" in markdown and "{" not in markdown   # sin dicts crudos
    assert len(ds.log.to_records()) == len(ds.log)
    assert ds.summary()["n_rows"] == 150


def test_duplicates_policy_drop_removes_the_repeated_row():
    ds = load_iris_dataset(duplicates="drop")
    assert len(ds.X) == 149 and ds.report["duplicates"] == 0


def test_missing_policy_keep_vs_drop_rows():
    df = read_table(default_data_dir() / "iris.csv")
    df.loc[[0, 10, 20], "sepal_width"] = np.nan
    kept = build_dataset(df, IRIS_SPEC)
    assert len(kept.X) == 150 and kept.report["missing"]["sepal_width"]["count"] == 3
    dropped = build_dataset(df, IRIS_SPEC, missing_policy="drop_rows")
    assert len(dropped.X) == 147 and len(dropped.y) == 147 and dropped.report["rows_with_missing"] == 0
    with pytest.raises(ValueError):
        build_dataset(df, IRIS_SPEC, missing_policy="otra")
    with pytest.raises(ValueError):
        build_dataset(df, IRIS_SPEC, duplicates="otra")


# --------------------------------------------------------------------- objetivo, tipos y ceros
def test_scenarios_remap_the_target_and_reject_unmapped_values():
    spec = DatasetSpec(name="demo", target="clase", scenarios={
        "binaria": Scenario("binaria", {"a": 0, "b": 1, "c": 1}, {0: "no", 1: "sí"}, "a contra el resto")})
    df = pd.DataFrame({"x": range(12), "clase": list("abc") * 4})
    ds = build_dataset(df, spec, scenario="binaria")
    assert ds.classes == [0, 1] and ds.y.value_counts().to_dict() == {1: 8, 0: 4}
    assert ds.class_names == {0: "no", 1: "sí"}
    with pytest.raises(ValueError):
        build_dataset(df, spec, scenario="inexistente")
    incomplete = DatasetSpec(name="demo", target="clase", scenarios={"x": Scenario("x", {"a": 0})})
    with pytest.raises(ValueError):
        build_dataset(df, incomplete, scenario="x")      # 'b' y 'c' no tienen mapeo


def test_declared_nominal_overrides_numeric_dtype_and_bad_names_are_rejected():
    df = pd.DataFrame({"codigo": [1, 2, 3, 1, 2, 3, 1, 2, 3, 1], "edad": range(10), "clase": list("ab") * 5})
    ds = dataset_from_frame(df, target="clase", nominal=["codigo"])
    assert ds.feature_types == {"codigo": "nominal", "edad": "numeric"}
    with pytest.raises(ValueError):
        dataset_from_frame(df, target="clase", nominal=["no_existe"])
    with pytest.raises(ValueError):
        dataset_from_frame(df, target="no_existe")


def test_zero_is_missing_turns_impossible_zeros_into_missing_values():
    df = pd.DataFrame({"chol": [200, 0, 250, 0, 230, 210, 190, 220, 240, 205], "clase": list("ab") * 5})
    ds = build_dataset(df, DatasetSpec(name="demo", target="clase", zero_is_missing=["chol"]))
    assert int(ds.X["chol"].isna().sum()) == 2 and "D-03" in ds.log.ids()


def test_rows_without_a_target_are_dropped_and_logged():
    df = pd.DataFrame({"x": range(10), "clase": ["a", "b", None, "a", "b", "a", "b", None, "a", "b"]})
    ds = dataset_from_frame(df, target="clase")
    assert len(ds.X) == 8 and any(d.title == "Filas sin objetivo" for d in ds.log)


# --------------------------------------------------------------------- auditoría y registro
def test_audit_flags_small_classes_imbalance_duplicates_constant_and_id_columns():
    n = 40
    base = pd.DataFrame({"id": range(n), "const": [1] * n, "x": np.arange(n) % 7})
    y = pd.Series(["a"] * 35 + ["b"] * 5)
    report = audit(base, y, {"id": "numeric", "const": "numeric", "x": "numeric"})
    assert report["constant_columns"] == ["const"] and report["id_like_columns"] == ["id"]
    assert report["duplicates"] == 0
    text = " ".join(report["warnings"])
    assert "La clase «b»" in text and "desbalanceadas" in text

    repeated = pd.concat([base, base.iloc[[3]]], ignore_index=True)
    y_repeated = pd.concat([y, y.iloc[[3]]], ignore_index=True)
    assert audit(repeated, y_repeated, {"id": "numeric", "const": "numeric", "x": "numeric"})["duplicates"] == 1


def test_decision_log_markdown_and_records():
    log = DecisionLog()
    log.add("D-09", "Prueba", "elegir A", rationale="Porque sí.", alternatives=["elegir B", "elegir C", "elegir D"],
            evidence={"casos": 3, "por clase": {"x": 1, "y": 2}, "columnas": []})
    markdown = log.to_markdown()
    assert "## Primera decisión: elegir A" in markdown and "*Tema: Prueba (código D-09)*" in markdown
    assert "**Otras opciones posibles:** elegir B, elegir C o elegir D." in markdown
    assert "- Casos: 3" in markdown and "- Por clase: x: 1, y: 2" in markdown and "- Columnas: ninguno" in markdown
    assert log.to_records()[0]["id"] == "D-09" and len(log) == 1


def test_decisions_are_numbered_in_words():
    from tp_ia.data.decisions import decision_label
    assert [decision_label(n) for n in (1, 2, 7, 10)] == ["Primera decisión", "Segunda decisión", "Séptima decisión", "Décima decisión"]
    assert decision_label(13) == "Decisión n.º 13"


def test_decision_log_is_listed_in_numeric_order_and_replace_does_not_duplicate():
    log = DecisionLog()
    for decision_id in ("D-05", "D-01", "D-03", "D-02"):
        log.add(decision_id, "t", "c")
    log.add("D-03", "segunda de las faltantes", "c")          # un mismo id puede registrarse dos veces
    assert re.findall(r"\(código (D-\d\d)\)", log.to_markdown()) == ["D-01", "D-02", "D-03", "D-03", "D-05"]
    assert [r["title"] for r in log.to_records() if r["id"] == "D-03"] == ["t", "segunda de las faltantes"]
    log.replace("D-03", "única", "c")
    assert log.ids().count("D-03") == 1 and len(log) == 4


def test_the_iris_decision_report_reads_in_order():
    headings = re.findall(r"\(código (D-\d\d)\)", load_iris_dataset().log.to_markdown())
    assert headings == sorted(headings) and headings[0] == "D-01" and "D-06" in headings


# --------------------------------------------------------------------- D-06: valores extremos
def test_outliers_follow_tukey_rule_and_skip_columns_without_spread():
    X = pd.DataFrame({"a": [1, 2, 3, 4, 5, 6, 7, 8, 9, 100],       # Q1=3.25, Q3=7.75 -> límites -3.5 y 14.5
                      "sin_dispersion": [0] * 9 + [5],             # Q1 == Q3: la regla no se puede aplicar
                      "k": range(1, 11), "texto": list("abcdefghij")})
    y = pd.Series(list("xy") * 5)
    types = {"a": "numeric", "sin_dispersion": "numeric", "k": "numeric", "texto": "nominal"}
    assert audit(X, y, types)["outliers"] == {"a": {"count": 1, "low": -3.5, "high": 14.5}}
    assert audit(X, y, types, outlier_factor=100)["outliers"] == {}


def test_iris_outliers_match_an_independent_numpy_computation_and_are_kept():
    ds = load_iris_dataset()
    expected = {}
    for column in ds.X.columns:
        values = ds.X[column].to_numpy()
        q1, q3 = np.percentile(values, [25, 75])
        count = int(((values < q1 - 1.5 * (q3 - q1)) | (values > q3 + 1.5 * (q3 - q1))).sum())
        if count:
            expected[column] = count
    assert expected == {"sepal_width": 4}                          # 2.0, 4.1, 4.2 y 4.4
    assert {c: i["count"] for c, i in ds.report["outliers"].items()} == expected
    d06 = next(d for d in ds.log if d.id == "D-06")
    assert d06.chosen == "conservar los valores extremos" and d06.evidence["valores extremos"] == 4
    assert len(ds.X) == 150 and ds.X["sepal_width"].max() == 4.4   # no se borró ni se recortó nada
    assert not any("extremo" in w for w in ds.report["warnings"])  # informa, no alarma


In [ ]:
%%writefile tests/test_eval.py
"""Tests de la evaluación: particiones, métricas, Random Forest y corredor. Todo sobre Iris."""
import numpy as np
import pandas as pd
import pytest
from sklearn.metrics import roc_auc_score

from tp_ia.c45 import C45DecisionTree
from tp_ia.data.fixtures import play_tennis
from tp_ia.data.iris import load_iris_dataset
from tp_ia.eval import (RandomForestModel, confusion, environment_info, evaluate, make_model, make_splits,
                        max_stratified_splits, paired_comparison, register_evaluation, run_experiment,
                        save_results, summarize)
from tp_ia.eval.runner import _align_proba


@pytest.fixture(scope="module")
def iris():
    return load_iris_dataset()


# --------------------------------------------------------------------- particiones
def test_splits_cover_every_sample_once_per_repeat_and_are_stratified(iris):
    y = iris.y.to_numpy()
    splits = make_splits(y, n_splits=5, n_repeats=2, seed=1)
    assert len(splits) == 10
    for repeat in (0, 1):
        tests = np.concatenate([s.test for s in splits if s.repeat == repeat])
        assert sorted(tests.tolist()) == list(range(150))
    for s in splits:
        assert not set(s.train) & set(s.test)
        counts = pd.Series(y[s.test]).value_counts()
        assert counts.min() == counts.max() == 10          # 150 / 5 pliegues = 30 por pliegue, 10 por clase


def test_splits_are_deterministic_and_depend_on_the_seed(iris):
    y = iris.y.to_numpy()
    a, b, c = make_splits(y, seed=3), make_splits(y, seed=3), make_splits(y, seed=4)
    assert all(np.array_equal(x.test, z.test) for x, z in zip(a, b))
    assert not all(np.array_equal(x.test, z.test) for x, z in zip(a, c))


def test_splits_reject_more_folds_than_the_smallest_class():
    y = np.array([0] * 20 + [1] * 3)
    assert max_stratified_splits(y) == 3
    with pytest.raises(ValueError, match="clase más chica"):
        make_splits(y, n_splits=5)
    with pytest.raises(ValueError):
        make_splits(y, n_splits=1)


def test_group_splits_never_separate_a_group():
    y = np.array([0] * 50 + [1] * 30 + [2] * 20)
    groups = np.arange(100) // 2                       # pares de filas "repetidas"
    for s in make_splits(y, n_splits=5, n_repeats=2, groups=groups):
        assert not set(groups[s.train]) & set(groups[s.test])


def test_holdout_scheme_and_invalid_scheme(iris):
    y = iris.y.to_numpy()
    splits = make_splits(y, scheme="holdout", n_repeats=4, test_size=0.3)
    assert len(splits) == 4 and all(len(s.test) == 45 for s in splits)
    with pytest.raises(ValueError):
        make_splits(y, scheme="otro")
    with pytest.raises(ValueError):
        make_splits(y, scheme="holdout", groups=np.arange(150))


# --------------------------------------------------------------------- métricas
def test_evaluate_matches_hand_computed_multiclass_values():
    y_true = ["a", "a", "b", "b", "c", "c"]
    y_pred = ["a", "b", "b", "b", "c", "a"]
    m = evaluate(y_true, y_pred, classes=["a", "b", "c"])
    assert m["accuracy"] == pytest.approx(4 / 6)
    assert m["balanced_accuracy"] == pytest.approx(2 / 3)
    assert (m["precision_a"], m["precision_b"], m["precision_c"]) == pytest.approx((0.5, 2 / 3, 1.0))
    assert (m["recall_a"], m["recall_b"], m["recall_c"]) == pytest.approx((0.5, 1.0, 0.5))
    assert m["macro_f1"] == pytest.approx((0.5 + 0.8 + 2 / 3) / 3)


def test_binary_sensitivity_specificity_and_auc_match_sklearn():
    m = evaluate(np.array([0, 0, 0, 1, 1]), np.array([0, 0, 1, 1, 0]), classes=[0, 1])
    assert m["sensitivity"] == pytest.approx(0.5) and m["specificity"] == pytest.approx(2 / 3)
    rng = np.random.default_rng(0)
    y = rng.integers(0, 2, 60)
    p1 = rng.random(60)
    scored = evaluate(y, (p1 > 0.5).astype(int), np.c_[1 - p1, p1], [0, 1])
    assert scored["roc_auc"] == pytest.approx(roc_auc_score(y, p1))


def test_evaluate_survives_a_class_missing_from_the_fold():
    proba = np.array([[0.9, 0.1, 0.0], [0.4, 0.6, 0.0], [0.2, 0.8, 0.0], [0.1, 0.9, 0.0]])
    m = evaluate([0, 0, 1, 1], [0, 1, 1, 1], proba, [0, 1, 2])
    assert np.isfinite(m["roc_auc"]) and m["support_2"] == 0 and m["recall_2"] == 0


def test_confusion_rows_are_the_true_class():
    assert confusion(["a", "b", "b"], ["a", "a", "b"], ["a", "b"]).tolist() == [[1, 0], [1, 1]]


# --------------------------------------------------------------------- modelos
def test_random_forest_on_iris_with_missing_values(iris):
    X = iris.X.copy()
    X.iloc[::9, 1] = np.nan
    rf = RandomForestModel(n_estimators=30, random_state=0).fit(X, iris.y, feature_types=iris.feature_types)
    np.testing.assert_allclose(rf.predict_proba(X).sum(axis=1), 1.0)
    assert list(rf.classes_) == iris.classes
    assert (rf.predict(X) == iris.y.to_numpy()).mean() > 0.9
    assert set(rf.feature_importances_) == set(X.columns)
    assert sum(rf.feature_importances_.values()) == pytest.approx(1.0)


def test_random_forest_handles_nominal_attributes_and_unseen_values():
    X, y = play_tennis()
    X = X.copy()
    X.loc[[0, 3], "Humidity"] = np.nan
    rf = RandomForestModel(n_estimators=20, random_state=0).fit(
        X, y, feature_types={c: "nominal" for c in X.columns})
    new = X.head(3).copy()
    new["Outlook"] = "Foggy"                           # valor que no vio al entrenar
    np.testing.assert_allclose(rf.predict_proba(new).sum(axis=1), 1.0)
    assert set(rf.feature_importances_) == set(X.columns)


def test_make_model_and_params():
    dt = make_model("dt", max_depth=2)
    assert isinstance(dt, C45DecisionTree) and dt.max_depth == 2
    rf = make_model("rf", n_estimators=7)
    assert rf.get_params()["n_estimators"] == 7 and rf.set_params(max_depth=3) is rf
    with pytest.raises(ValueError):
        make_model("otro")
    with pytest.raises(ValueError):
        rf.set_params(no_existe=1)


# --------------------------------------------------------------------- corredor
MODELS = {"DT": ("dt", {}), "DT-prof2": ("dt", {"max_depth": 2}), "RF": ("rf", {"n_estimators": 25, "random_state": 0})}


@pytest.fixture(scope="module")
def results(iris):
    return run_experiment(iris, MODELS, make_splits(iris.y, n_splits=3, n_repeats=2, seed=0))


def test_run_experiment_returns_one_row_per_split_and_model(results):
    assert len(results) == 6 * 3
    for column in ("model", "split", "fit_time", "predict_time", "test_accuracy", "train_accuracy",
                   "test_macro_f1", "test_roc_auc", "confusion", "params"):
        assert column in results.columns
    assert results["test_accuracy"].between(0, 1).all()
    assert results.attrs["env"]["sklearn"]


def test_summarize_orders_models_and_gives_sensible_scores(results):
    table = summarize(results)
    assert list(table.index) == ["DT", "DT-prof2", "RF"]
    assert table.loc["DT", "accuracy_mean"] > 0.9 and table.loc["RF", "accuracy_mean"] > 0.9
    assert "accuracy_std" in table.columns and "fit_time_mean" in table.columns


def test_a_deeper_tree_never_fits_the_training_data_worse(results):
    train = results.groupby("model")["train_accuracy"].mean()
    assert train["DT"] >= train["DT-prof2"]


def test_runs_are_reproducible(iris):
    splits = make_splits(iris.y, n_splits=3, n_repeats=1, seed=5)
    a = run_experiment(iris, MODELS, splits)
    b = run_experiment(iris, MODELS, splits)
    assert a["test_accuracy"].tolist() == b["test_accuracy"].tolist()


def test_paired_comparison_counts_and_p_value(results):
    comparison = paired_comparison(results, "DT", "DT-prof2")
    assert comparison["n"] == 6
    assert comparison["wins_a"] + comparison["wins_b"] + comparison["ties"] == 6
    assert 0.0 <= comparison["p_wilcoxon"] <= 1.0
    same = paired_comparison(results, "DT", "DT")
    assert same["mean_diff"] == 0 and same["p_wilcoxon"] == 1.0


def test_the_evaluation_decision_is_registered_once_and_reflects_the_splits():
    ds = load_iris_dataset()                       # propio: el fixture compartido no se toca
    splits = make_splits(ds.y, n_splits=3, n_repeats=2, seed=0)
    run_experiment(ds, {"DT": ("dt", {})}, splits)
    run_experiment(ds, {"DT": ("dt", {}), "RF": ("rf", {"n_estimators": 5})}, splits)   # se vuelve a correr
    d07 = [d for d in ds.log if d.id == "D-07"]
    assert len(d07) == 1                           # reemplaza, no duplica
    assert "3 pliegues" in d07[0].chosen and "repetida 2 veces" in d07[0].chosen
    sizes = lambda attr: sorted({len(getattr(s, attr)) for s in splits})
    span = lambda xs: f"{xs[0]}" if xs[0] == xs[-1] else f"entre {xs[0]} y {xs[-1]}"
    assert d07[0].evidence["particiones"] == 6 and d07[0].evidence["repeticiones"] == 2
    assert d07[0].evidence["casos de prueba por partición"] == span(sizes("test"))
    assert d07[0].evidence["casos de entrenamiento por partición"] == span(sizes("train"))
    assert set(d07[0].evidence["modelos"]) == {"DT", "RF"}          # queda la última evaluación


def test_the_evaluation_decision_describes_a_holdout_and_can_be_skipped():
    ds = load_iris_dataset()
    holdout = make_splits(ds.y, scheme="holdout", n_repeats=4, test_size=0.3)
    run_experiment(ds, {"DT": ("dt", {"max_depth": 2})}, holdout)
    d07 = next(d for d in ds.log if d.id == "D-07")
    assert d07.chosen.startswith("evaluar con una división estratificada en entrenamiento y prueba") and d07.evidence["pliegues por repetición"] == 1
    assert "profundidad máxima: 2" in d07.evidence["modelos"]["DT"]
    other = load_iris_dataset()
    run_experiment(other, {"DT": ("dt", {})}, holdout, register_decision=False)
    assert "D-07" not in other.log.ids()
    assert register_evaluation(other, [], {}) is None


def test_align_proba_fills_absent_classes_with_zero():
    out = _align_proba(np.array([[0.7, 0.3], [0.2, 0.8]]), [0, 2], [0, 1, 2])
    assert out.tolist() == [[0.7, 0.0, 0.3], [0.2, 0.0, 0.8]]


def test_results_roundtrip_through_csv_and_environment_info(results, tmp_path):
    path = tmp_path / "resultados.csv"
    save_results(results, path)
    back = pd.read_csv(path)
    assert len(back) == len(results) and "test_accuracy" in back.columns
    assert {"python", "numpy", "pandas", "scipy", "sklearn"} <= set(environment_info())


In [ ]:
%%writefile tests/test_experiments.py
"""Tests de los experimentos: barrido de parámetros, validación cruzada anidada y robustez a faltantes (sobre Iris)."""
import dataclasses
import json

import pytest

from tp_ia.data import load_iris_dataset
from tp_ia.eval import make_splits, run_experiment
from tp_ia.eval.experiments import (inject_missing, missing_robustness, nested_cv, parameter_sweep,
                                    robustness_table, selection_frequency)

DT = {"DT": ("dt", {})}
RF_SMALL = {"n_estimators": 15, "random_state": 0}
GRID = {"max_depth": [2, None], "min_samples_leaf": [2, 5]}


@pytest.fixture(scope="module")
def iris():
    return load_iris_dataset()


@pytest.fixture(scope="module")
def splits(iris):
    return make_splits(iris.y, n_splits=3, n_repeats=1, seed=0)


# --------------------------------------------------------------------- barrido de un parámetro
def test_sweep_shows_a_stump_cannot_separate_three_classes_and_reports_the_gap(iris, splits):
    sweep = parameter_sweep(iris, "dt", "max_depth", [1, 2, None], splits)
    assert list(sweep["value"]) == [1, 2, None]                   # se conservan los valores originales
    assert list(sweep.columns[:3]) == ["param", "value", "n_partitions"]
    assert (sweep["n_partitions"] == len(splits)).all()
    one, two, _ = sweep["test_macro_f1_mean"]
    assert one < 0.6 < 0.9 < two                                  # un solo corte separa a setosa y nada más
    assert (sweep["gap"] >= -1e-9).all()                          # en train se rinde igual o mejor que en test
    assert sweep["gap"].iloc[-1] > sweep["gap"].iloc[0]           # el árbol sin límite es el que más sobreajusta


def test_sweep_passes_base_params_and_works_for_the_random_forest(iris, splits):
    sweep = parameter_sweep(iris, "rf", "n_estimators", [3, 25], splits, base_params={"random_state": 0},
                            metric="accuracy")
    assert list(sweep["value"]) == [3, 25] and (sweep["test_accuracy_mean"] > 0.85).all()
    assert "test_macro_f1_mean" not in sweep.columns             # solo la métrica pedida


def test_sweep_rejects_repeated_values_empty_values_and_unknown_metrics(iris, splits):
    with pytest.raises(ValueError, match="repetir"):
        parameter_sweep(iris, "dt", "max_depth", [2, 2], splits)
    with pytest.raises(ValueError):
        parameter_sweep(iris, "dt", "max_depth", [], splits)
    with pytest.raises(ValueError, match="metric"):
        parameter_sweep(iris, "dt", "max_depth", [2], splits, metric="roc_auc")


# --------------------------------------------------------------------- validación cruzada anidada
def test_nested_cv_returns_one_row_per_outer_split_with_a_choice_from_the_grid(iris):
    outer = make_splits(iris.y, n_splits=3, n_repeats=1, seed=1)
    nested = nested_cv(iris, "dt", GRID, outer, inner_splits=3)
    assert len(nested) == 3 and list(nested["outer_split"]) == [s.id for s in outer]
    for chosen in nested["chosen"]:
        params = json.loads(chosen)
        assert set(params) == set(GRID) and all(params[name] in GRID[name] for name in GRID)
    assert nested["inner_score"].between(0, 1).all() and nested["test_macro_f1"].between(0, 1).all()
    assert (nested["test_accuracy"] > 0.8).all()
    assert selection_frequency(nested).sum() == 3


def test_the_outer_test_fold_never_influences_the_choice_and_runs_are_reproducible(iris):
    outer = make_splits(iris.y, n_splits=3, n_repeats=1, seed=2)[:1]
    altered_labels = iris.y.copy()
    altered_labels.iloc[outer[0].test] = iris.y.iloc[outer[0].test].to_numpy()[::-1]   # se rompe SOLO el test externo
    assert not altered_labels.equals(iris.y)
    original = nested_cv(iris, "dt", GRID, outer)
    altered = nested_cv(dataclasses.replace(iris, y=altered_labels), "dt", GRID, outer)
    assert original["chosen"][0] == altered["chosen"][0]
    assert original["inner_score"][0] == altered["inner_score"][0]           # la elección no vio el test
    assert original["test_accuracy"][0] != altered["test_accuracy"][0]       # y el resultado sí cambió
    again = nested_cv(iris, "dt", GRID, outer)
    assert again["chosen"][0] == original["chosen"][0] and again["test_accuracy"][0] == original["test_accuracy"][0]


def test_a_tie_in_the_inner_cv_goes_to_the_first_combination_of_the_grid(iris):
    outer = make_splits(iris.y, n_splits=3, n_repeats=1, seed=3)[:2]
    # max_depth None y 50 producen exactamente el mismo árbol (Iris no llega a 50 niveles): empate perfecto
    first = nested_cv(iris, "dt", {"max_depth": [None, 50]}, outer)
    second = nested_cv(iris, "dt", {"max_depth": [50, None]}, outer)
    assert {json.loads(c)["max_depth"] for c in first["chosen"]} == {None}
    assert {json.loads(c)["max_depth"] for c in second["chosen"]} == {50}


def test_nested_cv_rejects_an_empty_grid_a_parameter_without_values_and_a_bad_metric(iris, splits):
    for grid in ({}, {"max_depth": []}):
        with pytest.raises(ValueError, match="grid"):
            nested_cv(iris, "dt", grid, splits)
    with pytest.raises(ValueError, match="metric"):
        nested_cv(iris, "dt", {"max_depth": [2]}, splits, metric="kappa")


# --------------------------------------------------------------------- faltantes inyectados
def test_inject_missing_deletes_about_the_requested_fraction_and_leaves_the_original_intact(iris):
    out = inject_missing(iris, 0.3, seed=1)
    assert abs(out.X.isna().to_numpy().mean() - 0.3) < 0.06
    assert int(iris.X.isna().sum().sum()) == 0 and out.y.equals(iris.y)    # el original y la clase no se tocan
    assert out.X.shape == iris.X.shape and "faltantes" in out.name
    kept = out.X.notna().to_numpy()
    assert (out.X.to_numpy()[kept] == iris.X.to_numpy()[kept]).all()        # lo que no se borró queda igual


def test_inject_missing_is_reproducible_nested_and_validated(iris):
    a, b, c = (inject_missing(iris, 0.2, seed=s) for s in (5, 5, 6))
    assert a.X.isna().equals(b.X.isna()) and not a.X.isna().equals(c.X.isna())
    low, high = inject_missing(iris, 0.1, seed=5), inject_missing(iris, 0.3, seed=5)
    assert (low.X.isna().to_numpy() <= high.X.isna().to_numpy()).all()      # lo borrado al 10 % sigue borrado al 30 %
    assert inject_missing(iris, 0).X.equals(iris.X)
    only = inject_missing(iris, 0.5, seed=1, columns=["sepal_width"])
    assert only.X["sepal_width"].isna().any()
    assert only.X.drop(columns="sepal_width").equals(iris.X.drop(columns="sepal_width"))
    for bad in (-0.1, 1.0):
        with pytest.raises(ValueError):
            inject_missing(iris, bad)
    with pytest.raises(ValueError, match="no existen"):
        inject_missing(iris, 0.1, columns=["inexistente"])


# --------------------------------------------------------------------- robustez a faltantes
def test_robustness_at_zero_matches_the_plain_experiment_and_degrades_with_more_missing(iris, splits):
    models = {**DT, "RF": ("rf", RF_SMALL)}
    robust = missing_robustness(iris, models, [0, 0.4], splits)
    plain = run_experiment(iris, models, splits, register_decision=False)
    zero = robust[robust["missing_fraction"] == 0]
    assert zero["test_accuracy"].tolist() == plain["test_accuracy"].tolist()
    table = robustness_table(robust)
    assert list(table.index) == [0, 0.4] and list(table.columns) == ["DT", "RF"]
    assert (table.loc[0.4] < table.loc[0]).all()                            # quitar información no puede ayudar
    assert robust["mask_seed"].unique().tolist() == [0]


def test_robustness_can_average_over_several_mask_seeds(iris, splits):
    robust = missing_robustness(iris, DT, [0.2], splits, seeds=(0, 1, 2))
    assert sorted(robust["mask_seed"].unique()) == [0, 1, 2] and len(robust) == 3 * len(splits)
    assert robust.groupby("mask_seed")["test_macro_f1"].mean().nunique() > 1   # máscaras distintas, resultados distintos
    assert robustness_table(robust).iloc[0, 0] == pytest.approx(robust["test_macro_f1"].mean())


In [ ]:
%%writefile tests/test_export.py
"""Tests de los exportadores del árbol: estructura anidada, DOT de Graphviz y traza de un caso."""
import json

import numpy as np
import pandas as pd
import pytest

from tp_ia.c45 import C45DecisionTree
from tp_ia.data import load_iris_dataset
from tp_ia.data.fixtures import play_tennis


def walk(node):
    yield node
    for child in node.get("children", []):
        yield from walk(child)


@pytest.fixture(scope="module")
def tennis():
    X, y = play_tennis()
    tree = C45DecisionTree().fit(X, y, feature_types={c: "nominal" for c in X.columns})
    return tree, X, y


@pytest.fixture(scope="module")
def iris():
    ds = load_iris_dataset()
    return C45DecisionTree().fit(ds.X, ds.y), ds


# --------------------------------------------------------------------- estructura anidada
def test_the_dict_reproduces_the_tree_in_the_catedra_slide_for_play_tennis(tennis):
    tree, _, _ = tennis
    root = tree.to_dict()["root"]
    assert (root["feature"], root["n"]) == ("Outlook", 14)
    assert root["entropy"] == pytest.approx(0.940, abs=5e-4)                      # lámina 9: entropía 0.940
    assert root["gain"] == pytest.approx(0.247, abs=5e-4)                         # ganancia 0.247
    assert root["gain_ratio"] == pytest.approx(0.156, abs=5e-4)                   # razón de ganancia 0.156
    assert root["edges"] == ["Overcast", "Rain", "Sunny"]
    overcast, rain, sunny = root["children"]
    assert overcast["leaf"] and (overcast["label"], overcast["n"]) == ("Yes", 4)
    assert (rain["feature"], rain["n"], rain["edges"]) == ("Wind", 5, ["Strong", "Weak"])
    assert (sunny["feature"], sunny["n"], sunny["edges"]) == ("Humidity", 5, ["High", "Normal"])
    for inner in (rain, sunny):                                                   # entropía 0.971, ganancia 0.971, razón 1.000
        assert inner["entropy"] == pytest.approx(0.971, abs=5e-4)
        assert inner["gain"] == pytest.approx(0.971, abs=5e-4)
        assert inner["gain_ratio"] == pytest.approx(1.0, abs=5e-4)
    assert [(c["label"], c["n"]) for c in rain["children"]] == [("No", 2), ("Yes", 3)]
    assert [(c["label"], c["n"]) for c in sunny["children"]] == [("No", 3), ("Yes", 2)]


def test_ids_are_preorder_and_everything_is_json_safe(tennis):
    tree, _, _ = tennis
    data = tree.to_dict()
    json.dumps(data)                                                              # sin tipos de numpy
    nodes = list(walk(data["root"]))
    assert [n["id"] for n in nodes] == list(range(len(nodes))) and data["n_nodes"] == len(nodes)
    for node in nodes:
        if not node["leaf"]:
            assert node["children"][0]["id"] == node["id"] + 1                    # el primer hijo va pegado al padre
    assert data["n_leaves"] == sum(n["leaf"] for n in nodes) and data["depth"] == tree.depth_


def test_counts_probabilities_and_weights_are_consistent(tennis):
    tree, _, _ = tennis
    for node in walk(tree.to_dict()["root"]):
        assert sum(node["counts"]) == pytest.approx(node["n"])
        assert sum(node["proba"]) == pytest.approx(1.0)
        if not node["leaf"]:
            assert sum(c["n"] for c in node["children"]) == pytest.approx(node["n"])
            assert sum(node["branch_probs"]) == pytest.approx(1.0)


def test_numeric_edges_class_names_and_pruning_totals_on_iris(iris):
    full, ds = iris
    named = full.to_dict(class_names={"setosa": "Setosa", "virginica": "Virginica"})
    assert [c["name"] for c in named["classes"]] == ["Setosa", "versicolor", "Virginica"]
    assert [c["value"] for c in named["classes"]] == ["setosa", "versicolor", "virginica"]
    assert named["root"]["kind"] == "numeric" and named["root"]["edges"] == ["≤ 1.9", "> 1.9"]
    assert named["root"]["children"][0]["label"] == "Setosa"
    pruned = C45DecisionTree(prune=True).fit(ds.X, ds.y).to_dict()
    assert (pruned["n_nodes"], pruned["n_nodes_unpruned"], pruned["n_pruned"]) == (9, 13, 2)


def test_exporting_an_unfitted_tree_fails_clearly():
    with pytest.raises(RuntimeError):
        C45DecisionTree().to_dict()


# --------------------------------------------------------------------- DOT de Graphviz
def test_dot_follows_the_slide_format(tennis):
    tree, _, _ = tennis
    dot = tree.to_dot()
    nodes = list(walk(tree.to_dict()["root"]))
    assert dot.startswith("digraph Arbol {") and dot.endswith("}")
    assert dot.count(" -> ") == len(nodes) - 1
    for text in ("Outlook\\nentropía = 0.940\\nganancia = 0.247\\nrazón ganancia = 0.156\\nn = 14",
                 'label="Overcast"', 'label="Sunny"', "Yes\\nn = 4"):
        assert text in dot
    assert dot.count('fillcolor="#FFF2AE"') == sum(not n["leaf"] for n in nodes)    # nodos internos amarillos
    yes_color = [line for line in dot.splitlines() if 'label="Yes\\nn = 4"' in line][0]
    no_color = [line for line in dot.splitlines() if 'label="No\\nn = 2"' in line][0]
    assert yes_color.split('fillcolor="')[1] != no_color.split('fillcolor="')[1]   # una clase, un color


def test_dot_uses_display_names_and_escapes_special_characters():
    X = pd.DataFrame({'a"b': [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0], "c": range(8)})
    y = ["no"] * 4 + ["si"] * 4
    dot = C45DecisionTree().fit(X, y).to_dot(class_names={"no": "Sin enfermedad", "si": "Con enfermedad"})
    assert 'a\\"b' in dot and "Sin enfermedad" in dot and "Con enfermedad" in dot


# --------------------------------------------------------------------- traza de un caso
def leaf_probabilities(tree):
    return {n["id"]: np.array(n["proba"]) for n in walk(tree.to_dict()["root"]) if n["leaf"]}


def test_a_complete_case_follows_a_single_path(tennis):
    tree, _, _ = tennis
    weights = tree.trace({"Outlook": "Sunny", "Temperature": "Hot", "Humidity": "High", "Wind": "Weak"})
    nodes = {n["id"]: n for n in walk(tree.to_dict()["root"])}
    visited = {i for i, w in weights.items() if w > 0}
    assert all(weights[i] == 1.0 for i in visited) and len(weights) == len(nodes)
    leaf = [nodes[i] for i in visited if nodes[i]["leaf"]]
    assert len(leaf) == 1 and leaf[0]["label"] == "No"                              # soleado + humedad alta -> no juega
    assert sum(weights[i] for i, n in nodes.items() if n["leaf"]) == pytest.approx(1.0)


def test_a_missing_or_unseen_value_splits_the_case_by_branch_probabilities(tennis):
    tree, _, _ = tennis
    root = tree.to_dict()["root"]
    expected = [c["id"] for c in root["children"]], root["branch_probs"]
    for outlook in (None, np.nan, "Foggy"):                                          # faltante y nominal no visto
        weights = tree.trace({"Outlook": outlook, "Temperature": "Hot", "Humidity": "High", "Wind": "Weak"})
        assert [weights[i] for i in expected[0]] == pytest.approx(expected[1])
        assert weights[0] == 1.0


def test_the_trace_is_consistent_with_predict_proba(tennis, iris):
    tree, X, _ = tennis
    leaves = leaf_probabilities(tree)
    rows = [dict(r) for _, r in X.iterrows()]
    rows += [{**rows[0], "Outlook": None}, {**rows[3], "Wind": None, "Humidity": None}, {**rows[5], "Outlook": "Foggy"}]
    for row in rows:
        weights = tree.trace(row)
        mixed = sum(weights[i] * proba for i, proba in leaves.items())
        np.testing.assert_allclose(mixed, tree.predict_proba(pd.DataFrame([row]))[0], atol=1e-12)
    full, ds = iris                                                                  # y lo mismo con atributos numéricos
    leaves = leaf_probabilities(full)
    for _, r in ds.X.sample(25, random_state=0).iterrows():
        for broken in ({}, {"petal_length": np.nan}, {"petal_width": np.nan, "sepal_width": np.nan}):
            row = {**r.to_dict(), **broken}
            mixed = sum(full.trace(row)[i] * proba for i, proba in leaves.items())
            np.testing.assert_allclose(mixed, full.predict_proba(pd.DataFrame([row]))[0], atol=1e-12)


In [ ]:
%%writefile tests/test_heart.py
"""Enfermedad cardíaca (UCI Heart Disease, Cleveland): carga, traducción de códigos y uso desde la interfaz."""
import pytest

from tp_ia.c45 import C45DecisionTree
from tp_ia.data import load_heart_dataset
from tp_ia.web import LabService


@pytest.fixture(scope="module")
def heart():
    return load_heart_dataset()


def test_cleveland_has_the_documented_shape_and_missing_values(heart):
    assert heart.X.shape == (303, 13)
    missing = heart.X.isna().sum()
    assert missing[missing > 0].to_dict() == {"vasos_coloreados": 4, "talio": 2}   # como dice la auditoría de DATOS.md


def test_the_binary_target_groups_values_1_to_4_as_disease(heart):
    assert heart.report["class_counts"] == {0: 164, 1: 139}
    assert heart.class_names == {0: "sin enfermedad", 1: "con enfermedad"}
    d02 = [d for d in heart.log if d.id == "D-02"][-1]
    assert d02.chosen == "predecir si el paciente tiene o no una enfermedad cardíaca"


def test_the_multiclass_scenario_keeps_the_original_five_values():
    multi = load_heart_dataset(scenario="multiclase")
    assert multi.report["class_counts"] == {0: 164, 1: 55, 2: 36, 3: 35, 4: 13}


def test_nominal_codes_are_replaced_by_their_meaning(heart):
    assert set(heart.X["tipo_dolor"]) == {"angina típica", "angina atípica", "dolor no anginoso", "sin síntomas"}
    assert set(heart.X["sexo"]) == {"mujer", "hombre"}
    assert set(heart.X["talio"].dropna()) == {"normal", "defecto fijo", "defecto reversible"}
    assert heart.feature_types["vasos_coloreados"] == "numeric"          # es un conteo de 0 a 3, no un código
    assert sum(t == "nominal" for t in heart.feature_types.values()) == 7
    d05 = next(d for d in heart.log if d.id == "D-05")
    assert "talio" in d05.evidence["códigos reemplazados por su significado"]


def test_every_attribute_has_a_plain_language_description(heart):
    assert set(heart.descriptions) == set(heart.X.columns)
    assert "«cp»" in heart.descriptions["tipo_dolor"]                    # se menciona el nombre original de UCI


def test_the_tree_reads_with_words_instead_of_codes(heart):
    text = C45DecisionTree().fit(heart.X, heart.y, feature_types=heart.feature_types).export_text()
    assert "talio = " in text and "4.0" not in text


def test_heart_is_first_in_the_lab_and_shows_descriptions_and_class_split():
    lab = LabService()
    assert lab.datasets()["datasets"][0]["key"] == "heart"
    info = lab.select("heart")
    assert info["n_rows"] == 303 and [c["name"] for c in info["classes"]] == ["sin enfermedad", "con enfermedad"]
    pain = next(f for f in info["features"] if f["name"] == "tipo_dolor")
    assert pain["description"] and sum(pain["by_class"]["sin síntomas"]) == pain["counts"]["sin síntomas"]
    assert lab.train()["tree"]["root"]["leaf"] is False


In [ ]:
%%writefile tests/test_pruning.py
"""Tests de la poda por error pesimista de C4.5: fórmulas, ejemplo de Quinlan, árboles armados a mano y datos reales."""
import numpy as np
import pandas as pd
import pytest

from tp_ia.c45 import (C45DecisionTree, Node, added_errors, confidence_coefficient, estimated_errors,
                       prune_tree)
from tp_ia.data import load_iris_dataset
from tp_ia.data.fixtures import liu_loans, play_tennis
from tp_ia.eval import make_model, make_splits, parameter_sweep, run_experiment


# --------------------------------------------------------------------- árboles armados a mano
def leaf(*counts):
    c = np.array(counts, dtype=float)
    return Node(n_weight=float(c.sum()), class_counts=c, depth=1)


def split(children, depth=0):
    counts = sum(child.class_counts for child in children)
    return Node(n_weight=float(counts.sum()), class_counts=counts, depth=depth, feature=0, kind="nominal",
                values=list(range(len(children))), children=list(children),
                branch_probs=np.full(len(children), 1.0 / len(children)))


def quinlan_example():
    """16 casos: tres hojas (6, 0 errores), (9, 0 errores) y (1 caso de la otra clase)."""
    return split([leaf(6, 0), leaf(9, 0), leaf(0, 1)])


# --------------------------------------------------------------------- fórmulas
def test_the_coefficient_interpolates_quinlans_table():
    assert confidence_coefficient(0.25) == pytest.approx(0.6925 ** 2)     # entre 0.20 (0.84) y 0.40 (0.25)
    assert confidence_coefficient(0.10) == pytest.approx(1.28 ** 2)       # puntos de la tabla
    assert confidence_coefficient(0.05) == pytest.approx(1.65 ** 2)
    assert confidence_coefficient(1.0) == 0.0
    for bad in (0, -0.1, 1.5):
        with pytest.raises(ValueError):
            confidence_coefficient(bad)


def test_pessimistic_error_rates_match_the_values_in_quinlans_book():
    # U_25%(E, N): 0.206 está confirmado en una cita del libro; los otros tres son de su mismo ejemplo (de memoria)
    for (n, e), book in {(6, 0): 0.206, (9, 0): 0.143, (1, 0): 0.750, (16, 1): 0.157}.items():
        assert estimated_errors(n, e) / n == pytest.approx(book, abs=5e-4)


def test_added_errors_branches():
    cf = 0.25
    for n in (1, 6, 40):
        assert added_errors(n, 0, cf) == pytest.approx(n * (1 - cf ** (1 / n)))        # sin errores: binomial exacto
    for n in (5, 30):                                                                   # errores fraccionarios: interpolación
        assert added_errors(n, 0.5, cf) == pytest.approx((added_errors(n, 0, cf) + added_errors(n, 1, cf)) / 2)
    assert added_errors(2, 1.6, cf) == pytest.approx(0.67 * 0.4)                        # caso de borde e + 0.5 >= N
    assert added_errors(0, 0, cf) == 0.0 and added_errors(-1, 0, cf) == 0.0
    assert added_errors(10, 0.99989, cf) == pytest.approx(added_errors(10, 1.0, cf), abs=1e-3)   # continuidad


def test_added_errors_behaves_like_an_upper_confidence_limit():
    assert all(added_errors(n, e) > 0 for n in (3, 10, 50) for e in (0, 0.5, 1, 2))
    # más confianza exigida (CF más chico) -> límite superior más alto
    assert added_errors(20, 2, 0.05) > added_errors(20, 2, 0.25) > added_errors(20, 2, 0.5)
    # el mismo porcentaje de error con más casos se estima más ajustado
    assert added_errors(10, 0) / 10 > added_errors(100, 0) / 100
    assert added_errors(20, 1) / 20 < added_errors(20, 4) / 20 < added_errors(20, 8) / 20


# --------------------------------------------------------------------- poda sobre nodos armados a mano
def test_quinlans_example_replaces_the_three_leaf_subtree_by_a_single_leaf():
    root = quinlan_example()
    subtree = sum(estimated_errors(c.n_weight, c.errors) for c in root.children)
    as_leaf = estimated_errors(root.n_weight, root.errors)
    assert subtree == pytest.approx(3.273, abs=2e-3)                      # 6*0.206 + 9*0.143 + 1*0.750
    assert as_leaf == pytest.approx(2.512, abs=1e-2)                      # 16 * 0.157
    assert prune_tree(root, 0.25) == 1
    assert root.is_leaf and root.feature is None and root.threshold is None and root.branch_probs is None
    assert root.proba == pytest.approx([15 / 16, 1 / 16])                 # conserva la distribución de clases


def test_a_clean_split_is_kept():
    root = split([leaf(50, 0), leaf(0, 50)])
    assert prune_tree(root) == 0 and not root.is_leaf and len(root.children) == 2


def test_pruning_is_bottom_up_and_uses_the_pruned_estimate_of_the_children():
    inner = split([leaf(0, 25), leaf(0, 25)], depth=1)        # dos hojas puras de la misma clase: sobra el corte
    root = split([leaf(50, 0), inner])
    assert prune_tree(root) == 1
    assert inner.is_leaf and not root.is_leaf and root.children[1] is inner   # cae el de abajo, la raíz se queda


def test_tolerance_and_confidence_arguments():
    never = quinlan_example()
    assert prune_tree(never, 0.25, tolerance=-1e9) == 0 and not never.is_leaf     # exigir una mejora enorme: nada se poda
    everything = split([leaf(50, 0), leaf(0, 50)])
    assert prune_tree(everything, 0.25, tolerance=1e9) == 1 and everything.is_leaf
    for bad in (0, 0.6, -0.2):
        with pytest.raises(ValueError):
            prune_tree(quinlan_example(), bad)


# --------------------------------------------------------------------- el árbol completo
def test_prune_is_off_by_default_and_the_parameters_are_validated():
    tree = C45DecisionTree()
    assert tree.prune is False and tree.confidence == 0.25
    assert {"prune", "confidence"} <= set(tree.get_params())
    assert tree.set_params(prune=True, confidence=0.1) is tree and tree.prune is True
    X, y = play_tennis()
    plain = C45DecisionTree().fit(X, y)
    assert plain.n_pruned_ == 0 and plain.n_nodes_unpruned_ == plain.n_nodes_
    for kwargs in ({"confidence": 0}, {"confidence": -0.1}, {"confidence": 0.6}, {"prune": "si"}):
        with pytest.raises(ValueError):
            C45DecisionTree(**kwargs).fit(X, y)


def test_iris_pruned_tree_has_the_leaves_of_the_pruned_tree_reported_by_j48():
    iris = load_iris_dataset()
    full = C45DecisionTree().fit(iris.X, iris.y)
    pruned = C45DecisionTree(prune=True).fit(iris.X, iris.y)
    assert full.n_nodes_ == 13 and pruned.n_nodes_ == 9 and pruned.n_leaves_ == 5
    assert pruned.n_nodes_unpruned_ == 13 and pruned.n_pruned_ == 2
    leaves = sorted((round(n.n_weight), round(n.errors)) for n in pruned._iter_nodes() if n.is_leaf)
    # J48 (Weka) reporta para Iris las hojas 50, 48/1, 3, 3/1 y 46/1; solo cambia qué atributo separa a setosa
    # (petal_width <= 0.6 allá, petal_length <= 1.9 acá: empate de gain ratio que se resuelve por menor índice)
    assert leaves == sorted([(50, 0), (48, 1), (3, 0), (3, 1), (46, 1)])
    assert pruned.score(iris.X, iris.y) == pytest.approx(full.score(iris.X, iris.y))   # no pierde nada en train


def test_clean_toy_trees_are_not_pruned():
    X, y = play_tennis()
    types = {c: "nominal" for c in X.columns}
    a, b = C45DecisionTree().fit(X, y, feature_types=types), C45DecisionTree(prune=True).fit(X, y, feature_types=types)
    assert b.n_pruned_ == 0 and a.export_text() == b.export_text()
    X, y = liu_loans()
    a, b = C45DecisionTree().fit(X, y), C45DecisionTree(prune=True).fit(X, y)
    assert b.n_pruned_ == 0 and a.export_text() == b.export_text()


def noisy_data(n, noise, seed):
    """y = (x0 > 0) con una fracción `noise` de etiquetas invertidas, más 4 atributos que no sirven."""
    rng = np.random.default_rng(seed)
    X = pd.DataFrame(rng.normal(size=(n, 5)), columns=[f"x{i}" for i in range(5)])
    clean = (X["x0"] > 0).astype(int).to_numpy()
    return X, np.where(rng.random(n) < noise, 1 - clean, clean), clean


def test_pruning_removes_the_noise_a_full_tree_memorizes():
    X, y, _ = noisy_data(300, 0.2, seed=0)
    X_test, _, y_test = noisy_data(2000, 0.0, seed=99)                      # prueba sin ruido: la regla verdadera
    full = C45DecisionTree().fit(X, y)
    pruned = C45DecisionTree(prune=True, confidence=0.1).fit(X, y)
    assert full.n_nodes_ > 50 and pruned.n_nodes_ <= 5
    assert pruned.feature_names_[pruned.tree_.feature] == "x0"              # lo único que queda es el corte que sirve
    assert pruned.feature_importances_[0] == pytest.approx(1.0)
    assert pruned.score(X, y) <= full.score(X, y)                           # en train rinde menos...
    assert pruned.score(X_test, y_test) > full.score(X_test, y_test) + 0.15  # ...y generaliza mucho mejor
    np.testing.assert_allclose(pruned.predict_proba(X_test).sum(axis=1), 1.0)


def test_a_smaller_confidence_prunes_more():
    X, y, _ = noisy_data(300, 0.2, seed=0)
    sizes = [C45DecisionTree(prune=True, confidence=cf).fit(X, y).n_nodes_ for cf in (0.01, 0.1, 0.25, 0.5)]
    assert sizes == sorted(sizes) and sizes[0] < sizes[-1] <= C45DecisionTree().fit(X, y).n_nodes_


def test_pruning_works_with_missing_values_and_is_deterministic():
    X, y, _ = noisy_data(300, 0.2, seed=1)
    X = X.mask(np.random.default_rng(5).random(X.shape) < 0.1)             # 10 % de faltantes
    a = C45DecisionTree(prune=True, confidence=0.1).fit(X, y)
    b = C45DecisionTree(prune=True, confidence=0.1).fit(X, y)
    assert a.export_text() == b.export_text()
    assert a.n_nodes_ <= C45DecisionTree().fit(X, y).n_nodes_ and a.n_pruned_ > 0
    np.testing.assert_allclose(a.predict_proba(X).sum(axis=1), 1.0)
    X_clean, _, y_clean = noisy_data(1000, 0.0, seed=7)
    assert a.score(X_clean, y_clean) > 0.85


# --------------------------------------------------------------------- con el resto del sistema
def test_pruning_runs_through_the_evaluation_pipeline_and_the_parameter_sweep():
    iris = load_iris_dataset()
    splits = make_splits(iris.y, n_splits=3, n_repeats=1, seed=0)
    assert make_model("dt", prune=True, confidence=0.1).prune is True
    results = run_experiment(iris, {"DT": ("dt", {}), "DT-poda": ("dt", {"prune": True})}, splits,
                             register_decision=False)
    means = results.groupby("model")["test_accuracy"].mean()
    assert means["DT-poda"] > 0.9
    sweep = parameter_sweep(iris, "dt", "confidence", [0.05, 0.25, 0.5], splits, base_params={"prune": True})
    assert list(sweep["value"]) == [0.05, 0.25, 0.5] and (sweep["test_macro_f1_mean"] > 0.85).all()


In [ ]:
%%writefile tests/test_tree.py
"""Tests del árbol C4.5: estructura esperada, propiedades, faltantes, parámetros y validaciones.

Iris se usa solo como DATOS de prueba (sklearn.datasets); el árbol que se prueba es el nuestro.
"""
import numpy as np
import pandas as pd
import pytest
from sklearn.datasets import load_iris

from tp_ia.c45 import C45DecisionTree
from tp_ia.data.fixtures import liu_loans, play_tennis


@pytest.fixture(scope="module")
def iris():
    data = load_iris(as_frame=True)
    return data.data, data.target_names[data.target.to_numpy()]


def leaf_label(tree, node):
    return tree.classes_[int(np.argmax(node.class_counts))]


def child_by_value(node, value):
    return dict(zip(node.values, node.children))[value]


# --------------------------------------------------------------------- estructura esperada
def test_play_tennis_matches_textbook_tree():
    X, y = play_tennis()
    tree = C45DecisionTree().fit(X, y)
    root = tree.tree_
    assert tree.feature_names_[root.feature] == "Outlook"

    overcast = child_by_value(root, "Overcast")
    assert overcast.is_leaf and leaf_label(tree, overcast) == "Yes"

    sunny = child_by_value(root, "Sunny")
    assert tree.feature_names_[sunny.feature] == "Humidity"
    assert leaf_label(tree, child_by_value(sunny, "High")) == "No"
    assert leaf_label(tree, child_by_value(sunny, "Normal")) == "Yes"

    rain = child_by_value(root, "Rain")
    assert tree.feature_names_[rain.feature] == "Wind"
    assert leaf_label(tree, child_by_value(rain, "Strong")) == "No"
    assert leaf_label(tree, child_by_value(rain, "Weak")) == "Yes"

    assert tree.score(X, y) == 1.0
    assert tree.n_leaves_ == 5 and tree.depth_ == 2


def test_liu_loans_tree_matches_book():
    X, y = liu_loans()
    tree = C45DecisionTree().fit(X, y)
    root = tree.tree_
    assert tree.feature_names_[root.feature] == "Own_house"
    assert leaf_label(tree, child_by_value(root, "true")) == "Yes"
    no_house = child_by_value(root, "false")
    assert tree.feature_names_[no_house.feature] == "Has_job"
    assert leaf_label(tree, child_by_value(no_house, "true")) == "Yes"
    assert leaf_label(tree, child_by_value(no_house, "false")) == "No"
    assert tree.score(X, y) == 1.0


def test_iris_root_isolates_setosa(iris):
    X, y = iris
    tree = C45DecisionTree().fit(X, y)
    root = tree.tree_
    assert root.kind == "numeric"
    assert tree.feature_names_[root.feature] == "petal length (cm)"   # empate con petal width: gana el de menor índice
    assert root.threshold == pytest.approx(1.9)                      # modo "c45": valor observado <= punto medio
    assert root.children[0].class_counts.tolist() == [50, 0, 0]      # rama "<=": solo setosa
    assert root.children[1].class_counts.tolist() == [0, 50, 50]


def test_iris_training_accuracy_is_high(iris):
    X, y = iris
    assert C45DecisionTree().fit(X, y).score(X, y) >= 0.97


# --------------------------------------------------------------------- propiedades
def test_predict_proba_rows_sum_to_one(iris):
    X, y = iris
    proba = C45DecisionTree().fit(X, y).predict_proba(X)
    assert proba.shape == (150, 3)
    np.testing.assert_allclose(proba.sum(axis=1), 1.0)
    assert (proba >= 0).all() and (proba <= 1).all()


def test_fit_is_deterministic_and_invariant_to_row_order(iris):
    X, y = iris
    a = C45DecisionTree().fit(X, y)
    b = C45DecisionTree().fit(X, y)
    perm = np.random.default_rng(0).permutation(len(X))
    c = C45DecisionTree().fit(X.iloc[perm], y[perm])
    assert a.export_text() == b.export_text() == c.export_text()


def test_sample_weight_equals_duplicated_rows():
    X, y = play_tennis()
    weights = np.ones(len(X))
    weights[[0, 5]] = 2
    weighted = C45DecisionTree().fit(X, y, sample_weight=weights)
    duplicated = C45DecisionTree().fit(pd.concat([X, X.iloc[[0, 5]]], ignore_index=True),
                                       pd.concat([y, y.iloc[[0, 5]]], ignore_index=True))
    assert weighted.export_text() == duplicated.export_text()
    np.testing.assert_allclose(weighted.predict_proba(X), duplicated.predict_proba(X))


def test_sample_weight_equals_duplicated_rows_numeric(iris):
    X, y = iris
    weights = np.ones(len(X))
    weights[:40] = 2
    weighted = C45DecisionTree().fit(X, y, sample_weight=weights)
    dup_idx = list(range(len(X))) + list(range(40))
    duplicated = C45DecisionTree().fit(X.iloc[dup_idx], y[dup_idx])
    assert weighted.export_text() == duplicated.export_text()


def test_feature_importances_are_a_distribution(iris):
    X, y = iris
    tree = C45DecisionTree().fit(X, y)
    assert len(tree.feature_importances_) == X.shape[1]
    assert tree.feature_importances_.sum() == pytest.approx(1.0)
    assert (tree.feature_importances_ >= 0).all()


# --------------------------------------------------------------------- parámetros
def test_max_depth_and_min_samples_leaf_limit_the_tree(iris):
    X, y = iris
    stump = C45DecisionTree(max_depth=1).fit(X, y)
    assert stump.depth_ == 1 and stump.n_leaves_ == 2
    assert C45DecisionTree(max_depth=0).fit(X, y).n_nodes_ == 1
    assert C45DecisionTree(min_samples_leaf=80).fit(X, y).n_nodes_ == 1   # ningún corte deja 80+80 casos
    assert C45DecisionTree(min_gain=5.0).fit(X, y).n_nodes_ == 1          # ninguna ganancia llega a 5 bits


def test_more_restrictive_params_never_grow_the_tree(iris):
    X, y = iris
    sizes = [C45DecisionTree(min_samples_leaf=m).fit(X, y).n_nodes_ for m in (1, 2, 5, 10, 30)]
    assert sizes == sorted(sizes, reverse=True)


def test_criterion_gain_and_threshold_modes(iris):
    X, y = iris
    pure_gain = C45DecisionTree(criterion="gain").fit(*play_tennis())
    assert pure_gain.feature_names_[pure_gain.tree_.feature] == "Outlook"

    midpoint = C45DecisionTree(threshold_mode="midpoint").fit(X, y)
    assert midpoint.tree_.threshold == pytest.approx(2.45)               # (1.9 + 3.0) / 2
    corrected = C45DecisionTree(numeric_gain_correction=True).fit(X, y)
    assert corrected.score(X, y) >= 0.95                                  # la corrección no rompe el árbol


def test_get_and_set_params():
    tree = C45DecisionTree(max_depth=3)
    assert tree.get_params()["max_depth"] == 3
    assert tree.set_params(min_samples_leaf=5) is tree and tree.min_samples_leaf == 5
    with pytest.raises(ValueError):
        tree.set_params(no_existe=1)


# --------------------------------------------------------------------- faltantes y valores nuevos
def test_missing_values_in_training_conserve_weight_and_still_classify():
    X, y = play_tennis()
    X = X.copy()
    X.loc[[0, 5, 8], "Humidity"] = np.nan
    X.loc[[2], "Outlook"] = None
    tree = C45DecisionTree().fit(X, y)
    leaves = [n for n in tree._iter_nodes() if n.is_leaf]
    assert sum(n.n_weight for n in leaves) == pytest.approx(14.0)         # los pesos fraccionarios suman 14
    proba = tree.predict_proba(X)
    np.testing.assert_allclose(proba.sum(axis=1), 1.0)
    # Con 4 valores faltantes el árbol ya no es perfecto, pero debe superar al clasificador trivial
    # que siempre responde la clase mayoritaria (9 Yes de 14 = 0.643).
    assert tree.score(X, y) > 9 / 14


def test_missing_values_in_numeric_attributes(iris):
    X, y = iris
    X = X.copy()
    X.iloc[::7, 2] = np.nan          # faltantes en petal length (el atributo de la raíz)
    X.iloc[3::11, 0] = np.nan
    tree = C45DecisionTree().fit(X, y)
    leaves = [n for n in tree._iter_nodes() if n.is_leaf]
    assert sum(n.n_weight for n in leaves) == pytest.approx(150.0)
    proba = tree.predict_proba(X)
    np.testing.assert_allclose(proba.sum(axis=1), 1.0)
    assert tree.score(X, y) > 0.9
    all_missing = pd.DataFrame([{c: np.nan for c in X.columns}])
    assert tree.predict_proba(all_missing)[0].sum() == pytest.approx(1.0)


def test_missing_values_at_prediction_mix_the_branches():
    X, y = play_tennis()
    tree = C45DecisionTree().fit(X, y)
    row = pd.DataFrame([{"Outlook": None, "Temperature": "Hot", "Humidity": "High", "Wind": "Weak"}])
    proba = tree.predict_proba(row)[0]
    assert proba.sum() == pytest.approx(1.0)
    assert 0 < proba[list(tree.classes_).index("Yes")] < 1                # mezcla de ramas, no una certeza


def test_unseen_category_is_treated_as_missing():
    X, y = play_tennis()
    tree = C45DecisionTree().fit(X, y)
    row = pd.DataFrame([{"Outlook": "Foggy", "Temperature": "Hot", "Humidity": "High", "Wind": "Weak"}])
    proba = tree.predict_proba(row)[0]
    assert proba.sum() == pytest.approx(1.0) and (proba >= 0).all()


# --------------------------------------------------------------------- datos mixtos
def test_numeric_threshold_is_found_on_mixed_data():
    rng = np.random.default_rng(0)
    n = 300
    X = pd.DataFrame({"x": rng.uniform(0, 1, n), "color": rng.choice(["red", "blue", "green"], n)})
    y = np.where(X["x"] > 0.5, "A", "B")
    tree = C45DecisionTree().fit(X, y)
    assert tree.score(X, y) == 1.0
    assert tree.feature_names_[tree.tree_.feature] == "x"
    assert 0.4 < tree.tree_.threshold < 0.6


def test_nominal_split_is_multiway_on_mixed_data():
    rng = np.random.default_rng(1)
    n = 300
    X = pd.DataFrame({"x": rng.uniform(0, 1, n), "color": rng.choice(["red", "blue", "green"], n)})
    y = X["color"].map({"red": "R", "blue": "B", "green": "G"}).to_numpy()
    tree = C45DecisionTree().fit(X, y)
    assert tree.score(X, y) == 1.0
    assert tree.tree_.kind == "nominal" and len(tree.tree_.children) == 3


def test_integer_coded_category_can_be_declared_nominal():
    # El código 1/2/3 no tiene orden: como numérico habría que cortar "x <= 1", "x <= 2"...
    X = pd.DataFrame({"code": [1, 2, 3] * 20})
    y = np.where(X["code"] == 2, "mid", "edge")
    as_nominal = C45DecisionTree().fit(X, y, feature_types={"code": "nominal"})
    assert as_nominal.tree_.kind == "nominal" and as_nominal.score(X, y) == 1.0
    as_numeric = C45DecisionTree().fit(X, y)
    assert as_numeric.tree_.kind == "numeric" and as_numeric.depth_ == 2   # necesita dos cortes


# --------------------------------------------------------------------- texto y explicación
def test_export_text_and_explain_on_play_tennis():
    X, y = play_tennis()
    tree = C45DecisionTree().fit(X, y)
    text = tree.export_text()
    assert "Outlook = Overcast: Yes (4)" in text
    assert "|   Humidity = High: No (3)" in text
    info = tree.explain({"Outlook": "Sunny", "Temperature": "Hot", "Humidity": "High", "Wind": "Weak"})
    assert info["label"] == "No"
    assert info["rules"] == ["Outlook = Sunny", "Humidity = High"]
    assert info["proba"]["No"] == pytest.approx(1.0)


# --------------------------------------------------------------------- validaciones
def test_input_validation():
    X, y = play_tennis()
    with pytest.raises(ValueError):
        C45DecisionTree().fit(pd.DataFrame({"a": []}), [])                       # datos vacíos
    with pytest.raises(ValueError):
        C45DecisionTree().fit(X, y.iloc[:-1])                                    # largos distintos
    with pytest.raises(ValueError):
        C45DecisionTree().fit(X, y.where(y.index != 0, None))                    # y con faltantes
    with pytest.raises(ValueError):
        C45DecisionTree().fit(X, y, sample_weight=-np.ones(len(X)))              # pesos negativos
    with pytest.raises(ValueError):
        C45DecisionTree().fit(X, y, feature_types={"Outlook": "ordinal"})        # tipo inválido
    with pytest.raises(ValueError):
        C45DecisionTree(criterion="otro").fit(X, y)                              # parámetro inválido
    with pytest.raises(RuntimeError):
        C45DecisionTree().predict(X)                                             # sin entrenar
    tree = C45DecisionTree().fit(X, y)
    with pytest.raises(ValueError):
        tree.predict(X.drop(columns="Wind"))                                     # falta una columna


def test_float_coded_categories_are_shown_as_integers():
    # Una columna de códigos con NaN llega como float (3.0, 7.0): el texto debe decir "thal = 3".
    X = pd.DataFrame({"thal": [3.0, 7.0, 3.0, 7.0, np.nan, 3.0, 7.0, 3.0]})
    y = np.where(X["thal"] == 7.0, "sick", "ok")
    tree = C45DecisionTree().fit(X, y, feature_types={"thal": "nominal"})
    text = tree.export_text()
    assert "thal = 3" in text and "3.0" not in text
    assert tree.predict(pd.DataFrame({"thal": [3.0, 7.0]})).tolist() == ["ok", "sick"]


def test_single_class_gives_a_single_leaf():
    X, _ = play_tennis()
    tree = C45DecisionTree().fit(X, ["Yes"] * len(X))
    assert tree.n_nodes_ == 1 and set(tree.predict(X)) == {"Yes"}


In [ ]:
%%writefile tests/test_web_service.py
"""Tests de la capa de servicio de la web: todo lo que la interfaz pide, sin abrir ningún navegador."""
import json
import math

import numpy as np
import pandas as pd
import pytest

from tp_ia.web import LabService, to_jsonable


@pytest.fixture()
def lab():
    return LabService()


@pytest.fixture()
def iris(lab):
    lab.select("iris")
    return lab


@pytest.fixture()
def tennis(lab):
    lab.select("play_tennis")
    return lab


def roundtrip(value):
    """Lo que le llega a la interfaz: pasa por JSON y vuelve."""
    return json.loads(json.dumps(value, allow_nan=False))


# --------------------------------------------------------------------- utilidades
def test_to_jsonable_converts_numpy_pandas_and_non_finite_numbers():
    data = {1: np.int64(3), "x": np.float32(2.5), "n": float("nan"), "i": math.inf, "a": np.array([1, 2]),
            "b": np.bool_(True), "t": (1, np.nan), "none": pd.NA, "df": pd.DataFrame({"u": [1]})}
    out = to_jsonable(data)
    assert out == {"1": 3, "x": 2.5, "n": None, "i": None, "a": [1, 2], "b": True, "t": [1, None], "none": None,
                   "df": [{"u": 1}]}
    json.dumps(out, allow_nan=False)


def test_unknown_methods_unknown_datasets_and_missing_selection_are_rejected(lab):
    with pytest.raises(ValueError, match="desconocido"):
        lab.call("os_system", {})
    with pytest.raises(ValueError, match="Dataset desconocido"):
        lab.select("no_existe")
    with pytest.raises(RuntimeError, match="elegí un dataset"):
        lab.call("train")


# --------------------------------------------------------------------- datasets
def test_the_catalog_lists_every_included_dataset(lab):
    keys = [d["key"] for d in lab.call("datasets")["datasets"]]
    assert keys == ["heart", "play_tennis", "iris", "wine", "breast_cancer"]      # enfermedad cardíaca, primero
    assert all(d["title"] and d["description"] for d in lab.call("datasets")["datasets"])


def test_iris_info_has_what_the_data_tab_shows(lab):
    info = roundtrip(lab.call("select", {"key": "iris"}))
    assert (info["n_rows"], info["n_features"], info["target"]) == (150, 4, "species")
    assert [c["count"] for c in info["classes"]] == [50, 50, 50]
    assert info["split"] == {"n_train": 105, "n_test": 45, "test_size": 0.3}
    assert {"D-01", "D-02", "D-03", "D-04", "D-05", "D-06"} <= {d["id"] for d in info["decisions"]}
    assert info["quality"]["duplicates"] == 1 and len(info["preview"]) == 8
    width = next(f for f in info["features"] if f["name"] == "sepal_width")
    assert width["kind"] == "numeric" and width["outliers"] == 4 and width["default"] == pytest.approx(3.0)
    assert len(width["hist"]["edges"]) == 13
    totals = {c: sum(counts) for c, counts in width["hist"]["by_class"].items()}
    assert totals == {"setosa": 50, "versicolor": 50, "virginica": 50}


def test_play_tennis_uses_every_case_for_training_and_describes_nominal_features(tennis):
    info = roundtrip(tennis.info())
    assert info["split"] == {"n_train": 14, "n_test": 0, "test_size": 0.0}
    outlook = next(f for f in info["features"] if f["name"] == "Outlook")
    assert outlook["kind"] == "nominal" and outlook["values"] == ["Overcast", "Rain", "Sunny"]
    assert outlook["counts"] == {"Overcast": 4, "Rain": 5, "Sunny": 5} and outlook["default"] in outlook["values"]
    result = roundtrip(tennis.train())
    assert result["metrics"]["test"] is None and result["metrics"]["train"]["accuracy"] == 1.0
    assert result["tree"]["root"]["feature"] == "Outlook" and result["tree"]["n_nodes"] == 8


@pytest.mark.parametrize("key, rows, classes", [("wine", 178, 3), ("breast_cancer", 569, 2)])
def test_scikit_learn_datasets_load_through_the_same_loader(lab, key, rows, classes):
    info = roundtrip(lab.select(key))
    assert info["n_rows"] == rows and len(info["classes"]) == classes
    assert all(f["kind"] == "numeric" for f in info["features"])


def test_a_csv_uploaded_from_the_interface_becomes_the_current_dataset(lab):
    text = "edad;peso;clase\n30;70,5;a\n41;?;b\n52;80,25;a\n33;66,0;b\n45;71,1;a\n38;69,9;b\n"
    info = roundtrip(lab.call("load_csv", {"text": text, "target": "clase", "name": "Mis datos"}))
    assert (info["n_rows"], info["title"]) == (6, "Mis datos") and info["quality"]["missing"]["peso"]["count"] == 1
    assert [c["count"] for c in info["classes"]] == [3, 3]
    with pytest.raises(ValueError, match="objetivo"):
        lab.call("load_csv", {"text": text, "target": "no_existe"})


# --------------------------------------------------------------------- entrenar
def test_training_returns_the_tree_metrics_and_importances(iris):
    result = roundtrip(iris.call("train"))
    assert result["tree"]["root"]["id"] == 0 and result["dot"].startswith("digraph Arbol {")
    assert result["metrics"]["test"]["n"] == 45 and result["metrics"]["train"]["n"] == 105
    assert result["metrics"]["test"]["accuracy"] > 0.85 and result["metrics"]["train"]["accuracy"] >= 0.95
    assert sum(sum(row) for row in result["metrics"]["test"]["confusion"]) == 45
    values = [i["value"] for i in result["importances"]]
    assert values == sorted(values, reverse=True) and sum(values) == pytest.approx(1.0)
    assert result["params"]["criterion"] == "gain_ratio" and result["fit_time"] > 0
    assert "petal" in result["text"]


def test_parameters_are_cleaned_and_pruning_shrinks_the_tree(iris):
    full = iris.train()
    pruned = iris.train({"prune": True, "confidence": 0.05, "foo": 1, "max_depth": 3.0})
    assert pruned["params"]["prune"] is True and pruned["params"]["max_depth"] == 3 and "foo" not in pruned["params"]
    assert pruned["tree"]["n_nodes"] <= full["tree"]["n_nodes"] and pruned["tree"]["depth"] <= 3
    assert iris.train({"max_depth": None})["params"]["max_depth"] is None


def test_the_split_can_change_and_is_reproducible(iris):
    assert iris.call("set_split", {"test_size": 0.5, "seed": 3}) == {"n_train": 75, "n_test": 75, "test_size": 0.5}
    a = iris.train()["metrics"]["test"]["accuracy"]
    iris.call("set_split", {"test_size": 0.5, "seed": 3})
    assert iris.train()["metrics"]["test"]["accuracy"] == a
    for bad in (-0.1, 0.95):
        with pytest.raises(ValueError):
            iris.call("set_split", {"test_size": bad})


def test_the_forest_uses_the_catedra_preset_unless_overridden(iris):
    result = roundtrip(iris.call("train_forest"))
    assert (result["params"]["n_estimators"], result["params"]["max_depth"], result["params"]["random_state"]) == (100, 10, 42)
    assert result["metrics"]["test"]["accuracy"] > 0.85 and result["metrics"]["train"]["accuracy"] > 0.95
    assert iris.train_forest({"n_estimators": 7})["params"]["n_estimators"] == 7


# --------------------------------------------------------------------- predecir un caso
def test_predicting_a_case_returns_both_models_and_the_path(iris):
    row = {f["name"]: f["default"] for f in iris.info()["features"]}
    out = roundtrip(iris.call("predict", {"row": row}))
    assert out["classes"] == ["setosa", "versicolor", "virginica"] and out["missing"] == []
    for model in ("dt", "rf"):
        assert sum(out[model]["proba"]) == pytest.approx(1.0) and out[model]["label"] in out["classes"]
    trace = {int(k): v for k, v in out["dt"]["trace"].items()}
    assert trace[0] == 1.0 and out["dt"]["rules"]
    root_leaves = [i for i, w in trace.items() if w == 1.0]
    assert len(root_leaves) >= 2                                              # el camino completo pesa 1 en cada nodo


def test_a_flower_of_each_species_is_classified_and_missing_values_split_the_path(iris):
    setosa = {"sepal_length": 5.0, "sepal_width": 3.4, "petal_length": 1.4, "petal_width": 0.2}
    virginica = {"sepal_length": 6.8, "sepal_width": 3.0, "petal_length": 5.6, "petal_width": 2.1}
    assert iris.call("predict", {"row": setosa})["dt"]["label"] == "setosa"
    assert iris.call("predict", {"row": virginica})["rf"]["label"] == "virginica"
    broken = iris.call("predict", {"row": {**setosa, "petal_length": None, "petal_width": ""}})
    assert broken["missing"] == ["petal_length", "petal_width"]
    assert sum(broken["dt"]["proba"]) == pytest.approx(1.0)
    assert any(0 < w < 1 for w in broken["dt"]["trace"].values())             # el caso se reparte entre ramas


def test_predicting_with_nominal_attributes_and_an_unseen_value(tennis):
    row = {"Outlook": "Sunny", "Temperature": "Hot", "Humidity": "High", "Wind": "Weak"}
    assert tennis.call("predict", {"row": row})["dt"]["label"] == "No"
    unseen = tennis.call("predict", {"row": {**row, "Outlook": "Foggy"}})
    assert sum(unseen["dt"]["proba"]) == pytest.approx(1.0)


# --------------------------------------------------------------------- comparar y experimentar
def test_compare_runs_both_models_on_the_same_partitions(iris):
    out = roundtrip(iris.call("compare", {"rf_params": {"n_estimators": 20}, "n_splits": 3, "n_repeats": 1}))
    assert out["n_partitions"] == 3 and set(out["models"]) == {"DT", "RF"}
    for model in ("DT", "RF"):
        data = out["models"][model]
        assert len(data["per_split"]) == 3 and data["stats"]["accuracy"]["mean"] > 0.85
        assert data["stats"]["gap"]["mean"] == pytest.approx(
            data["stats"]["train_accuracy"]["mean"] - data["stats"]["accuracy"]["mean"])
        assert np.sum(data["confusion"]) == 150                                  # cada caso se prueba una vez
    assert out["paired"]["accuracy"]["n"] == 3 and out["decision"]["id"] == "D-07"
    assert out["classes"] == ["setosa", "versicolor", "virginica"]


def test_the_number_of_folds_is_capped_by_the_smallest_class(tennis):
    out = tennis.call("compare", {"n_splits": 10, "n_repeats": 1, "rf_params": {"n_estimators": 5}})
    assert out["n_splits"] == 5                                                   # la clase chica de Play Tennis tiene 5 casos


def test_the_parameter_sweep_returns_one_row_per_value_and_marks_the_best(iris):
    out = roundtrip(iris.call("sweep", {"kind": "dt", "param": "max_depth", "values": [1, 2, None],
                                        "n_splits": 3, "n_repeats": 1}))
    assert [r["value"] for r in out["rows"]] == [1, 2, None] and out["metric"] == "macro_f1"
    assert out["rows"][0]["test_mean"] < 0.6 < out["rows"][1]["test_mean"]       # un corte solo separa a setosa
    assert out["best"] in (2, None) and all(r["gap"] >= -1e-9 for r in out["rows"])


def test_the_sweep_validates_the_parameter_and_prunes_when_sweeping_the_confidence(iris):
    with pytest.raises(ValueError, match="no permitido"):
        iris.call("sweep", {"kind": "dt", "param": "criterion", "values": ["gain"]})
    with pytest.raises(ValueError, match="no permitido"):
        iris.call("sweep", {"kind": "xx", "param": "max_depth", "values": [2]})
    out = iris.call("sweep", {"kind": "dt", "param": "confidence", "values": [0.05, 0.5], "n_splits": 3, "n_repeats": 1})
    assert out["base"]["prune"] is True
    forest = iris.call("sweep", {"kind": "rf", "param": "n_estimators", "values": [3, 20], "n_splits": 3, "n_repeats": 1})
    assert [r["value"] for r in forest["rows"]] == [3, 20] and "n_estimators" not in forest["base"]


def test_robustness_to_missing_values(iris):
    out = roundtrip(iris.call("robustness", {"fractions": [0, 0.4], "n_splits": 3, "n_repeats": 1,
                                             "rf_params": {"n_estimators": 20}}))
    assert [r["fraction"] for r in out["rows"]] == [0.0, 0.4] and out["metric"] == "macro_f1"
    clean, damaged = out["rows"]
    assert 0 < damaged["DT"] < clean["DT"] <= 1 and 0 < damaged["RF"] < clean["RF"] <= 1
    assert clean["DT_std"] is not None and out["seeds"] == [0]


def test_decisions_are_available_as_records_and_markdown(iris):
    iris.compare(rf_params={"n_estimators": 5}, n_splits=3, n_repeats=1)
    out = iris.call("decisions")
    ids = [d["id"] for d in out["records"]]
    assert ids == sorted(ids) and "D-07" in ids and "(código D-06)" in out["markdown"]


In [ ]:
%%writefile tests/test_web_widget.py
"""Tests del transporte de la web: protocolo de mensajes, widget de anywidget y servidor de desarrollo."""
import json
import shutil
import subprocess
import sys
import threading
import types
import urllib.error
import urllib.request
from http.server import ThreadingHTTPServer
from pathlib import Path

import pytest

from tp_ia.web import LabService
from tp_ia.web.devserver import make_handler

STATIC = Path(__file__).resolve().parents[1] / "src" / "tp_ia" / "web" / "static"


# --------------------------------------------------------------------- protocolo {id, method, params}
def test_handle_answers_with_the_same_id_and_the_result():
    lab = LabService()
    reply = lab.handle({"id": 7, "method": "datasets", "params": {}})
    assert reply["id"] == 7 and reply["ok"] is True
    assert [d["key"] for d in reply["result"]["datasets"]][:3] == ["heart", "play_tennis", "iris"]
    json.dumps(reply, allow_nan=False)


def test_handle_turns_every_failure_into_an_error_reply_instead_of_raising():
    lab = LabService()
    for message, expected in [({"id": 1, "method": "no_existe"}, "ValueError"),
                              ({"id": 2, "method": "train"}, "RuntimeError"),                       # sin dataset elegido
                              ({"id": 3, "method": "select", "params": {"key": "nada"}}, "ValueError"),
                              ({"id": 4, "method": "select", "params": {"clave": "iris"}}, "TypeError"),
                              ({"id": 5}, "ValueError")]:
        reply = lab.handle(message)
        assert reply["id"] == message["id"] and reply["ok"] is False and reply["error"].startswith(expected)
    bad = lab.handle("no soy un diccionario")
    assert bad["id"] is None and bad["ok"] is False and "objeto" in bad["error"]


def test_a_failed_request_does_not_break_the_session():
    lab = LabService()
    assert lab.handle({"id": 1, "method": "select", "params": {"key": "iris"}})["ok"]
    assert not lab.handle({"id": 2, "method": "sweep", "params": {"kind": "dt", "param": "criterion", "values": ["x"]}})["ok"]
    assert lab.handle({"id": 3, "method": "train"})["ok"]


# --------------------------------------------------------------------- widget de anywidget
def test_the_widget_answers_messages_through_the_service():
    pytest.importorskip("anywidget")
    from tp_ia.web.widget import TreeLab

    sent = []
    widget = TreeLab()
    widget.send = lambda content, buffers=None: sent.append(content)
    widget._on_message(widget, {"id": 11, "method": "select", "params": {"key": "iris"}}, [])
    widget._on_message(widget, {"id": 12, "method": "train", "params": {"params": {"prune": True}}}, [])
    widget._on_message(widget, {"id": 13, "method": "inexistente"}, [])
    assert [m["id"] for m in sent] == [11, 12, 13] and [m["ok"] for m in sent] == [True, True, False]
    assert sent[0]["result"]["n_rows"] == 150 and sent[1]["result"]["tree"]["n_pruned"] >= 0
    assert widget.service.dataset is not None


def test_the_widget_carries_the_javascript_and_the_styles():
    pytest.importorskip("anywidget")
    from tp_ia.web.widget import TreeLab

    widget = TreeLab()
    assert "export default" in str(widget._esm) and "render" in str(widget._esm)
    assert ".tl " in str(widget._css) and "tl-tree" in str(widget._css)


def fake_colab(monkeypatch, shell):
    """Simula google.colab y la shell de IPython, para que el resultado no dependa de dónde corra pytest."""
    pytest.importorskip("anywidget")
    pytest.importorskip("IPython")
    calls = []
    colab = types.ModuleType("google.colab")
    colab.output = types.SimpleNamespace(enable_custom_widget_manager=lambda: calls.append(True))
    monkeypatch.setitem(sys.modules, "google.colab", colab)
    monkeypatch.setattr("IPython.get_ipython", lambda: shell)
    return calls


def test_enable_colab_does_nothing_when_google_colab_is_not_installed(monkeypatch):
    pytest.importorskip("anywidget")
    from tp_ia.web.widget import enable_colab

    monkeypatch.setitem(sys.modules, "google.colab", None)                       # None en sys.modules hace fallar el import
    assert enable_colab() is False


def test_enable_colab_does_nothing_in_a_colab_subprocess_without_an_ipython_shell(monkeypatch):
    # El pytest de la última celda del notebook corre en un subproceso de Colab: google.colab se importa, pero no hay kernel.
    calls = fake_colab(monkeypatch, shell=None)
    from tp_ia.web.widget import enable_colab

    assert enable_colab() is False and calls == []


def test_enable_colab_turns_on_the_widget_manager_inside_a_colab_kernel(monkeypatch):
    calls = fake_colab(monkeypatch, shell=object())
    from tp_ia.web.widget import enable_colab

    assert enable_colab() is True and calls == [True]


# --------------------------------------------------------------------- servidor de desarrollo
@pytest.fixture()
def server():
    httpd = ThreadingHTTPServer(("127.0.0.1", 0), make_handler(LabService()))
    thread = threading.Thread(target=httpd.serve_forever, daemon=True)
    thread.start()
    yield f"http://127.0.0.1:{httpd.server_address[1]}"
    httpd.shutdown()
    httpd.server_close()


def post(url, payload):
    request = urllib.request.Request(url, data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.loads(response.read())


def test_the_dev_server_serves_the_page_the_script_and_the_styles(server):
    for path, kind, marker in [("/", "text/html", "app.js"), ("/app.js", "text/javascript", "export default"),
                               ("/app.css", "text/css", ".tl ")]:
        with urllib.request.urlopen(server + path, timeout=10) as response:
            assert response.status == 200 and kind in response.headers["Content-Type"]
            assert marker in response.read().decode("utf-8")
            assert response.headers["Cache-Control"] == "no-store"
    with pytest.raises(urllib.error.HTTPError) as missing:
        urllib.request.urlopen(server + "/secreto.py", timeout=10)
    assert missing.value.code == 404


def test_the_dev_server_speaks_the_same_protocol_as_the_widget(server):
    assert post(server + "/rpc", {"id": 1, "method": "select", "params": {"key": "iris"}})["result"]["n_rows"] == 150
    reply = post(server + "/rpc", {"id": 2, "method": "train", "params": {"params": {"max_depth": 2}}})
    assert reply["ok"] and reply["id"] == 2 and reply["result"]["tree"]["depth"] <= 2
    assert post(server + "/rpc", {"id": 3, "method": "no_existe"})["ok"] is False
    with pytest.raises(urllib.error.HTTPError) as wrong:
        post(server + "/otra_ruta", {})
    assert wrong.value.code == 404


def test_the_dev_server_survives_a_body_that_is_not_json(server):
    request = urllib.request.Request(server + "/rpc", data=b"esto no es json")
    with urllib.request.urlopen(request, timeout=10) as response:
        assert json.loads(response.read())["ok"] is False


# --------------------------------------------------------------------- archivos estáticos
@pytest.mark.skipif(shutil.which("node") is None, reason="Node no está instalado")
def test_the_javascript_has_valid_module_syntax(tmp_path):
    # La extensión .mjs fuerza el análisis como módulo ES en cualquier versión de Node
    # (--experimental-default-type se eliminó en Node 24).
    module = tmp_path / "app.mjs"
    shutil.copyfile(STATIC / "app.js", module)
    result = subprocess.run(["node", "--check", str(module)],
                            capture_output=True, text=True, timeout=60)
    assert result.returncode == 0, result.stderr


def test_the_stylesheet_is_balanced_and_scoped_to_the_widget():
    css = (STATIC / "app.css").read_text(encoding="utf-8")
    assert css.count("{") == css.count("}")
    body = css.split(";", 1)[1]                                                      # sin el @import de la tipografía
    for line in body.splitlines():
        line = line.strip()
        if line and line[0] in ".@" and "{" in line and not line.startswith(("@media", "@keyframes")):
            assert line.startswith(".tl"), f"selector fuera de .tl: {line[:60]}"


In [ ]:
# Corre las pruebas automáticas del proyecto y muestra solo el resumen (o el detalle, si algo falla).
run_quiet([sys.executable, "-m", "pip", "install", "-q", "pytest"])
result = subprocess.run([sys.executable, "-m", "pytest", "--color=no", "-p", "no:cacheprovider"],
                        cwd=ROOT, capture_output=True, text=True)
summary = result.stdout.strip().splitlines()
if result.returncode == 0:
    print("✔", summary[-1])
else:
    print(result.stdout[-4000:] or result.stderr[-4000:])